# Clasificación de clústeres biosintéticos (BGC) por su perfil de dominios Pfam

Este notebook parte de **una sola tabla** (`tabla_bgc-5.csv`, una fila por región BGC) y, sin ningún otro archivo de entrada:

1. **explora** la colección de BGCs y sus dominios Pfam;
2. **entrena** un clasificador (SVM lineal con eliminación recursiva de dominios, RFE) que predice la categoría de producto (`product_primary`) de cada BGC a partir de sus dominios;
3. **evalúa** el modelo con BGCs que nunca vio durante el entrenamiento;
4. **explora los resultados**: agrupa los BGCs, identifica los dominios responsables de cada grupo y compara la arquitectura gen a gen de un BGC representativo de cada grupo, con un análisis específico de sideróforos y de similitud de secuencia;
5. **compara los clústeres entre sí**: mapa de calor de clústeres contra dominios, dominios y genes que se repiten, sintenia (orden de los genes) y todos los clústeres alineados en orden.

## Estructura

| Parte | Pregunta que responde | Secciones | Carpeta de resultados |
|---|---|---|---|
| **0 · Preparación** | Instalación, configuración y funciones | 0.1 – 0.4 | — |
| **1 · Análisis y exploración de los datos** | ¿Qué contiene la tabla? ¿Las clases se distinguen por sus dominios? | 1.1 – 1.8 | `resultados/1_datos/` |
| **2 · Entrenamiento del modelo** | ¿Cuántos dominios necesita el modelo? ¿Está sobreajustado? | 2.1 – 2.7 | `resultados/2_entrenamiento/` |
| **3 · Análisis de los resultados del modelo** | ¿Qué tan bien clasifica BGCs nuevos y en qué se equivoca? | 3.1 – 3.5 | `resultados/3_evaluacion/` |
| **4 · Exploración de los resultados: grupos y arquitectura** | ¿Qué dominios separan a los grupos de BGCs y cómo cambia su arquitectura? | 4.1 – 4.6 | `resultados/4_grupos_arquitectura/` |
| **5 · Comparación de clústeres** | ¿Qué dominios y genes se repiten y qué tan conservado está el orden de los genes? | 5.1 – 5.5 | `resultados/5_comparacion_clusteres/` |
| **Anexo** | Modelo exportado y lista de archivos generados | A.1 – A.3 | `resultados/anexo_modelo/` |

Cada archivo empieza con el número de la sección que lo genera (por ejemplo `2.2_curva_seleccion_dominios.png`), así que se ubica directamente en el notebook. Cada figura tiene a su lado una tabla CSV con los valores que muestra.

## Datos de entrada

| Columna | Uso |
|---|---|
| `product_primary` | Clase a predecir: categoría de producto del BGC asignada por antiSMASH |
| `PF…:…` (una por dominio Pfam) | Puntuación del dominio en el BGC; 0 = dominio ausente. Son las características del modelo |
| `assembly_accession`, `region_number`, `location` | Identifican cada región y permiten unir las copias GenBank (`GCA_`) y RefSeq (`GCF_`) del mismo genoma |
| `protein_sequences` | Proteínas del BGC separadas por `\|` (secciones 4.3 – 4.6 y Parte 5) |
| `protein_seq_pfam_map` | Las mismas proteínas con cada dominio sustituido por `[Dominio(valor)]`: de aquí salen la arquitectura gen a gen (Parte 4) y el orden de los genes (Parte 5) |

## Cómo ejecutarlo

- **Google Colab:** sube este notebook y `tabla_bgc-5.csv` a la misma carpeta y usa *Entorno de ejecución → Ejecutar todo*. La celda 0.1 instala lo que falte.
- **Local:** `pip install -r requirements.txt` y abre el notebook con Jupyter.
- **Duración:** unos 8–11 minutos con 2 núcleos. Lo más lento es la validación cruzada *leave-one-out* (Parte 2), los alineamientos de la sección 4.6 y el agrupamiento de proteínas de la sección 5.3. Para una corrida rápida cambia `"cv_strategy"` a `"kfold"` en la celda 0.2.

## Reproducibilidad

Todo lo aleatorio depende de `CONFIG["random_state"]`, así que dos corridas con la misma tabla dan exactamente las mismas cifras y figuras.

## Limitaciones (conviene leerlas antes de interpretar)

1. La partición entrenamiento/prueba se hace por BGC, no por genoma: BGCs casi idénticos de cepas cercanas pueden quedar a ambos lados. La sección 3.1 mide cuántos BGCs de prueba tienen un perfil de dominios idéntico en el entrenamiento y la exactitud en los que no lo tienen.
2. antiSMASH asigna `product_primary` con reglas basadas en dominios; el modelo aprende a reproducir esa asignación a partir de los dominios Pfam.
3. La tabla no guarda la hebra de cada gen: en las figuras de arquitectura los genes van en el orden de la tabla y todas las flechas apuntan a la derecha. Las categorías funcionales se asignan por familia Pfam con una lista editable (sección 4.2), no por anotación manual. Además, la columna `protein_seq_pfam_map` no contiene todos los dominios que registran las columnas `PF…` (la sección 4.3 mide cuántos faltan): las arquitecturas gen a gen y los nombres de los genes solo incluyen los dominios del mapa. La tabla tampoco trae las coordenadas de cada gen: la sintenia y los alineamientos de la Parte 5 asumen que los genes de cada BGC están listados en su orden en el genoma, y sus resultados dependen del umbral de similitud con el que se decide que dos genes son el mismo gen (sección 5.3).
4. t-SNE es una herramienta descriptiva: las distancias entre grupos y su tamaño en el mapa no se interpretan.
5. Algunas filas del CSV describen tramos del mismo genoma que se solapan con clases distintas (por ejemplo, un BGC `nrps` dentro de las coordenadas de un BGC `nrp-metallophore`). Sus perfiles de dominios son casi iguales, así que el modelo no puede separarlos. La sección 1.3 los cuenta y la 3.4 marca los errores que caen en estos casos.

## Referencias

- Blin K, Shaw S, Augustijn HE, *et al.* (2023). antiSMASH 7.0: new and improved predictions for detection, regulation, chemical structures and visualisation. *Nucleic Acids Research* 51: W46–W50. doi:10.1093/nar/gkad344
- Mistry J, *et al.* (2021). Pfam: The protein families database in 2021. *Nucleic Acids Research* 49: D412–D419.
- Guyon I, Weston J, Barnhill S, Vapnik V (2002). Gene selection for cancer classification using support vector machines. *Machine Learning* 46: 389–422.
- Pedregosa F, *et al.* (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research* 12: 2825–2830.
- van der Maaten L, Hinton G (2008). Visualizing data using t-SNE. *Journal of Machine Learning Research* 9: 2579–2605.
- Cock PJA, *et al.* (2009). Biopython: freely available Python tools for computational molecular biology and bioinformatics. *Bioinformatics* 25: 1422–1423.
- Kralova S, Spacek P, Gafriller J, *et al.* (2026). Kineochelins—A new group of siderophores from an Antarctic bacterium. *Microbial Biotechnology* 19: e70386. doi:10.1111/1751-7915.70386 (referencia para la figura de arquitectura de la sección 4.3)
- Navarro-Muñoz JC, Selem-Mojica N, Mullowney MW, *et al.* (2020). A computational framework to explore large-scale biosynthetic diversity. *Nature Chemical Biology* 16: 60–68. doi:10.1038/s41589-019-0400-9 (BiG-SCAPE; fórmula del índice de adyacencia que se usa en la sección 5.4)

---
# Parte 0 · Preparación

Instalación, configuración y funciones auxiliares. **La única celda que normalmente hay que editar es la 0.2.**

## 0.1 Instalación

Instala solo los paquetes que falten (en Google Colab suele faltar únicamente Biopython) y no imprime el registro de `pip`.

In [ ]:
import importlib.util, subprocess, sys

REQUIRED = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib", "seaborn": "seaborn", "joblib": "joblib", "Bio": "biopython"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Paquetes instalados:", ", ".join(missing) if missing else "ninguno (ya estaban todos)")

## 0.2 Configuración

Todos los parámetros del análisis, agrupados por la parte del notebook que los usa. Los valores por defecto reproducen el protocolo del estudio (SVM lineal, RFE, validación *leave-one-out*, regla de un error estándar).

In [ ]:
import numpy as np

CONFIG = {
    # ---- Entrada, salida y reproducibilidad -----------------
    "input_csv":      "tabla_bgc-5.csv",     # único archivo de entrada
    "output_dir":     "resultados",  # una subcarpeta por parte del notebook
    "save_figures":   True,
    "figure_formats": ("png", "pdf"),
    "figure_dpi":     300,                   # resolución de los archivos guardados
    "inline_max_width_px": 1400,             # ancho máximo (px) de una figura dentro del notebook; no afecta a los archivos
    "show_individual_panels": False,         # True = mostrar aquí también los paneles individuales (siempre se guardan)
    "random_state":   42,                    # semilla única para todo lo aleatorio
    "n_jobs":         -1,                    # núcleos de CPU (-1 = todos)

    # ---- Columnas del CSV --------------------------------------------------------
    "target_column":   "product_primary",          # clase a predecir
    "feature_prefix":  "PF",                       # columnas numéricas que empiezan así = dominios Pfam
    "assembly_column": "assembly_accession",
    "region_column":   "region_number",
    "location_column": "location",
    "sequence_column": "protein_sequences",        # proteínas del BGC separadas por "|"
    "pfam_map_column": "protein_seq_pfam_map",     # las mismas proteínas con [Dominio(valor)]

    # ---- Parte 1 · Datos ------------------------------------------------------
    "collapse_assembly_redundancy": True,    # unir las copias GenBank (GCA_) y RefSeq (GCF_) del mismo genoma
    "min_class_size":  10,                   # las clases con menos BGCs se excluyen
    "drop_zero_variance": True,              # quitar dominios que nunca varían
    "tsne_perplexity_main": 30,              # perplejidad de los mapas t-SNE principales
    "tsne_perplexities": (5, 15, 30),        # perplejidades de la figura de sensibilidad (1.5c)
    "tsne_pca_components": 50,               # compresión interna previa al t-SNE (None = ninguna)
    "tsne_ellipses":   True,                 # dibujar elipses en los mapas t-SNE
    "ellipse_level":   0.95,                 # nivel de confianza de cada elipse
    "corr_circle_fraction": 0.25,            # fracción estratificada para el círculo de correlación
    "corr_circle_top_features": 15,          # dominios (flechas) dibujados en el círculo

    # ---- Parte 2 · Entrenamiento ------------------------------------------
    "test_size":       0.30,                 # prueba apartada: no se usa hasta la Parte 3
    "validation_size": 0.20,                 # validación interna (sale del entrenamiento)
    "svm_c":           1.0,                  # C inicial del SVM lineal
    "rfe_step":        0.1,                  # fracción de dominios eliminados por ronda de RFE
    "cv_strategy":     "loocv",              # "loocv" (protocolo del estudio) o "kfold" (más rápido)
    "kfold_splits":    10,
    "kfold_repeats":   3,
    "feature_grid":    (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
                        22, 25, 28, 30, 35, 40, 45, 50, 60, 75, 100, 125, 150, 200),
    "tune_c":          True,                 # elegir C por validación cruzada
    "c_grid":          (0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 100.0),
    "learning_curve_sizes": np.linspace(0.15, 1.0, 8),
    "n_permutations":  100,                  # permutaciones de etiquetas (prueba de permutación)
    "max_generalization_gap": 0.05,          # diferencia tolerable entre exactitud de entrenamiento y de prueba

    # ---- Parte 4 · Grupos y arquitectura -----------------------------
    "n_groups":        3,                    # grupos que se forman sobre el t-SNE
    "group_on":        "all",                # "all" = t-SNE con todos los dominios · "signature" = con los dominios del modelo
    "marker_domains_per_group": 5,           # dominios marcadores mostrados por grupo
    "tree_max_depth":  3,                    # profundidad del árbol de reglas (legible)
    "tree_min_prevalence": 0.05,             # el árbol solo usa dominios presentes en al menos 5 % de los BGCs
    "arch_representatives": 1,               # BGCs representativos dibujados por grupo
    "siderophore_classes": ("nrp-metallophore", "ni-siderophore"),   # clases de sideróforos (4.5)
    "siderophore_compare_classes": ("nrps",),                        # clases que se muestran como comparación en 4.5
    "sid_min_bgcs":    3,                    # mínimo de BGCs de una clase en un grupo para dibujar su representante
    "seq_per_class":   6,                    # BGCs por clase en la matriz de alineamientos (el costo crece con n²)
    "seq_max_proteins_per_bgc": 10,          # se alinean las N proteínas más largas de cada BGC
    "seq_min_length":  30,                   # se descartan péptidos más cortos (aa)
    "mantel_permutations": 999,              # permutaciones de la prueba de Mantel

    # ---- Parte 5 · Comparación de clústeres ---------------------------------------
    "heatmap_domains_per_class": 8,          # dominios característicos por clase en el mapa de calor (5.1)
    "recurrent_top_domains": 30,             # dominios más repetidos entre clústeres que se muestran (5.2)
    "gene_family_min_similarity": 0.5,       # similitud mínima con el representante para pertenecer a una familia de genes (5.3)
    "repeat_top_items": 15,                  # genes / dominios repetidos que se muestran (5.3)
    "synteny_type_threshold": 0.5,           # índice de sintenia medio mínimo dentro de un tipo de sintenia (5.4)
    "align_representatives": 4,              # tipos de sintenia cuyo representante se dibuja a escala (5.5)
    "align_label_min_fraction": 0.25,        # se nombra el gen de las columnas que comparte al menos esta fracción de los BGCs (5.5)
    "align_min_bgcs_shown": 3,               # los loci con menos BGCs se guardan pero no se muestran en el notebook (5.5)
}

## 0.3 Librerías y funciones de modelado

Funciones que se usan en todo el notebook: carpetas de resultados, el modelo (escalado + SVM lineal), la estrategia de validación cruzada, el intervalo de Wilson y el cálculo de métricas.

In [ ]:
import re, time, warnings
from pathlib import Path

import pandas as pd
from scipy.stats import chi2, fisher_exact, spearmanr
from sklearn.cluster import AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE
from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report, cohen_kappa_score,
                             confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score,
                             roc_auc_score, silhouette_score)
from sklearn.model_selection import (LeaveOneOut, RepeatedStratifiedKFold, StratifiedKFold, cross_val_score,
                                     learning_curve, permutation_test_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")      # avisos de sklearn por clases ausentes en particiones pequeñas
RS = CONFIG["random_state"]
np.random.seed(RS)

# ---- Carpetas de resultados: una por parte del notebook -------------------
OUT = Path(CONFIG["output_dir"])
PART_DIRS = {"1": OUT / "1_datos", "2": OUT / "2_entrenamiento", "3": OUT / "3_evaluacion",
             "4": OUT / "4_grupos_arquitectura", "5": OUT / "5_comparacion_clusteres",
             "A": OUT / "anexo_modelo"}
for folder in PART_DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)


def out_path(section, name, ext):
    """"2.2", "curva" → resultados/2_entrenamiento/2.2_curva.<ext>"""
    return PART_DIRS[section.split(".")[0]] / f"{section}_{name}.{ext}"


def save_table(df, section, name):
    """Guarda la tabla como CSV junto a las figuras de su sección y la devuelve."""
    df.to_csv(out_path(section, name, "csv"), index=False)
    return df


def slug(text):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(text))


# ---- Modelo y validación -----------------------------------------------------------
def make_estimator(C=None, probability=False):
    """Escalado + SVM lineal. El escalador va dentro del pipeline para que, en validación cruzada,
    se ajuste solo con la parte de entrenamiento de cada partición (sin fuga de información)."""
    return Pipeline([("scaler", StandardScaler()),
                     ("svm", SVC(kernel="linear", C=CONFIG["svm_c"] if C is None else C,
                                 probability=probability, random_state=RS))])


def make_cv():
    """LOOCV (protocolo del estudio) o K-fold estratificado repetido (más rápido)."""
    if CONFIG["cv_strategy"] == "loocv":
        return LeaveOneOut()
    return RepeatedStratifiedKFold(n_splits=CONFIG["kfold_splits"], n_repeats=CONFIG["kfold_repeats"], random_state=RS)


def cv_summary(scores):
    """Media y error estándar. En LOOCV cada partición vale 0 o 1, así que la desviación estándar entre
    particiones no describe al modelo; el error estándar de la media (sd/√n) sí."""
    scores = np.asarray(scores, float)
    se = scores.std(ddof=1) / np.sqrt(len(scores)) if len(scores) > 1 else 0.0
    return float(scores.mean()), float(se)


def wilson_interval(successes, n, z=1.96):
    """Intervalo de Wilson para una proporción (la exactitud); se comporta mejor que el normal con n pequeño."""
    p, d = successes / n, 1 + z ** 2 / n
    centre = (p + z ** 2 / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
    return max(0.0, centre - half), min(1.0, centre + half)


def compute_metrics(y_true, y_pred, classes, partition):
    """Métricas globales (una fila) y por clase (una fila por clase)."""
    kw = {"zero_division": 0, "labels": list(classes)}
    overall = pd.DataFrame([{
        "partition": partition, "n": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", **kw),
        "recall_macro": recall_score(y_true, y_pred, average="macro", **kw),
        "f1_macro": f1_score(y_true, y_pred, average="macro", **kw),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred)}]).round(4)
    report = classification_report(y_true, y_pred, labels=list(classes), output_dict=True, zero_division=0)
    per_class = pd.DataFrame([{"partition": partition, "class": c, "n": int(report[c]["support"]),
                               "precision": report[c]["precision"], "recall": report[c]["recall"],
                               "f1": report[c]["f1-score"]} for c in classes])
    per_class["errors"] = (per_class["n"] * (1 - per_class["recall"])).round().astype(int)
    per_class["error_rate"] = 1 - per_class["recall"]
    return overall, per_class.round(4)

## 0.4 Estilo y funciones de figuras

Todas las figuras comparten el mismo estilo para que se lean igual:

- **Colores fijos por clase.** Las 8 primeras clases (de mayor a menor tamaño) usan una paleta categórica verificada para daltonismo. Si hay más de 8 clases, a partir de la novena se reutiliza un color pero con un marcador distinto y hueco, para que ninguna clase dependa solo del color.
- **Etiquetas directas.** En los mapas t-SNE cada clase lleva su nombre escrito sobre su grupo principal, además de la leyenda.
- **Elipses por grupo compacto.** Una clase puede aparecer en varios sitios del mapa. Por eso cada elipse rodea un grupo compacto de BGCs de la misma clase (95 %), en lugar de una sola elipse por clase que cubriría zonas vacías.
- **Mapas de calor** en una escala de azul claro a azul oscuro; para valores con signo (más o menos que el resto), azul → gris → rojo.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, to_rgba
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse, FancyBboxPatch, Polygon

%matplotlib inline

# ---- Paleta y tinta (paleta categórica validada para daltonismo) --
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
CONTEXT = "#d9d8d2"                                         # puntos de contexto
SEQ_CMAP = LinearSegmentedColormap.from_list("seq_blue", [SURFACE, "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5",
                                                          "#256abf", "#184f95", "#0d366b"])
DIV_CMAP = LinearSegmentedColormap.from_list("div_blue_red", ["#2a78d6", "#f0efec", "#e34948"])
HALO = [pe.withStroke(linewidth=3, foreground=SURFACE)]    # contorno claro para que el texto se lea sobre los puntos

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "figure.dpi": 110,
    "font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.labelsize": 11, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-", "axes.axisbelow": True,
    "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.frameon": False, "legend.fontsize": 10,
    "lines.linewidth": 2, "lines.solid_capstyle": "round"})


def titles(ax, title, subtitle=None):
    """Título (qué muestra la figura) y subtítulo (cómo leerla), alineados a la izquierda."""
    n_lines = 0 if not subtitle else subtitle.count("\n") + 1
    ax.set_title(title, loc="left", pad=10 + 15 * n_lines)
    if subtitle:
        ax.text(0, 1.012, subtitle, transform=ax.transAxes, ha="left", va="bottom", fontsize=10, color=INK2)


def show_and_save(fig, section, name, individual=False):
    """Guarda la figura (png/pdf) en la carpeta de su parte y la muestra. Los paneles individuales
    se guardan siempre y se muestran solo si CONFIG["show_individual_panels"] es True."""
    if CONFIG["save_figures"]:
        for fmt in CONFIG["figure_formats"]:
            fig.savefig(out_path(section, name, fmt), dpi=CONFIG["figure_dpi"], bbox_inches="tight")
    if individual and not CONFIG["show_individual_panels"]:
        plt.close(fig)
    else:
        if fig.get_figwidth() * fig.dpi > CONFIG["inline_max_width_px"]:        # las figuras muy anchas se muestran a menor resolución
            fig.set_dpi(CONFIG["inline_max_width_px"] / fig.get_figwidth())
        plt.show()


def make_style(order, start=0):
    """Color y marcador fijos por categoría. Más allá de 8 categorías se reutiliza un color con un
    marcador distinto y hueco (codificación compuesta color × forma)."""
    reuse = [(6, "D"), (0, "s"), (2, "^"), (5, "v"), (1, "P")]
    style = {}
    for i, c in enumerate(order):
        if start + i < len(SERIES):
            style[c] = {"color": SERIES[start + i], "marker": "o", "hollow": False}
        else:
            k, marker = reuse[(start + i - len(SERIES)) % len(reuse)]
            style[c] = {"color": SERIES[k], "marker": marker, "hollow": True}
    return style


def repel(xy, box, n_iter=400):
    """Separa etiquetas que se enciman. `box` = (ancho, alto) común o uno por etiqueta."""
    pos = np.asarray(xy, float).copy()
    wh = np.broadcast_to(np.asarray(box, float), pos.shape)
    for _ in range(n_iter):
        moved = False
        for i in range(len(pos)):
            for j in range(i + 1, len(pos)):
                bx, by = (wh[i] + wh[j]) / 2
                d = (pos[i] - pos[j]) / (bx, by)
                dist = np.hypot(*d)
                if dist < 1:
                    u = d / dist if dist > 1e-9 else np.array([0.0, 1.0])
                    step = u * (1 - dist) / 2 * np.array([bx, by])
                    pos[i] += step
                    pos[j] -= step
                    moved = True
        if not moved:
            break
    return pos


def compact_groups(pts, span, eps_frac=0.05, min_pts=3):
    """Grupos compactos de puntos de una misma categoría en el mapa (DBSCAN), del mayor al menor."""
    if len(pts) < min_pts:
        return []
    lab = DBSCAN(eps=eps_frac * span, min_samples=min_pts).fit_predict(pts)
    return [pts[lab == k] for k in sorted(set(lab) - {-1}, key=lambda k: -(lab == k).sum())]


def ellipse_patch(pts, span, color, level=None, fill=True):
    """Elipse de confianza (normal bivariada) de un grupo de puntos, con un tamaño mínimo para que
    los grupos de puntos idénticos sigan siendo visibles."""
    level = CONFIG["ellipse_level"] if level is None else level
    cov = np.cov(pts, rowvar=False) + np.eye(2) * (0.012 * span) ** 2
    vals, vecs = np.linalg.eigh(cov)
    vals, vecs = vals[::-1], vecs[:, ::-1]
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    width, height = 2 * np.sqrt(vals * chi2.ppf(level, df=2))
    return Ellipse(pts.mean(axis=0), width, height, angle=angle, lw=1.2, zorder=1,
                   facecolor=to_rgba(color, 0.08) if fill else "none", edgecolor=to_rgba(color, 0.95))


def cat_scatter(ax, E, labels, order, style, flip=(1, 1), ellipses=None, direct_labels=True, s=26,
                label_size=9.5, alpha=0.9):
    """Mapa 2D con un color/marcador por categoría, elipses por grupo compacto y etiquetas directas."""
    ellipses = CONFIG["tsne_ellipses"] if ellipses is None else ellipses
    E = np.asarray(E)[:, :2] * np.array(flip)
    span = float(np.ptp(E, axis=0).max())
    anchors = {}
    for c in order:
        m = labels == c
        if not m.any():
            continue
        st = style[c]
        ax.scatter(E[m, 0], E[m, 1], s=s * (1.6 if st["hollow"] else 1), marker=st["marker"], alpha=alpha,
                   facecolors="none" if st["hollow"] else st["color"],
                   edgecolors=st["color"] if st["hollow"] else SURFACE,
                   linewidths=1.3 if st["hollow"] else 0.6, zorder=3 if st["hollow"] else 2)
        groups = compact_groups(E[m], span)
        if ellipses:
            for g in groups:
                ax.add_patch(ellipse_patch(g, span, st["color"]))
        anchors[c] = (groups[0] if groups else E[m]).mean(axis=0)
    if direct_labels and anchors:
        keys = list(anchors)
        xy = np.array([anchors[k] for k in keys])
        rx, ry = np.ptp(E, axis=0)
        pos = repel(xy + [0, 0.045 * ry], (0.15 * rx, 0.05 * ry))
        for k, (x0, y0), (x1, y1) in zip(keys, xy, pos):
            if abs(x1 - x0) > 0.02 * rx or abs(y1 - y0 - 0.045 * ry) > 0.02 * ry:      # línea guía solo si la etiqueta se movió
                ax.plot([x0, x1], [y0, y1 - 0.01 * ry], color=INK2, lw=0.6, zorder=4)
            ax.text(x1, y1, k, ha="center", va="bottom", fontsize=label_size, fontweight="bold", color=INK,
                    path_effects=HALO, zorder=5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)


def cat_handles(order, style):
    return [Line2D([], [], ls="", marker=style[c]["marker"], ms=8,
                   mfc="none" if style[c]["hollow"] else style[c]["color"],
                   mec=style[c]["color"] if style[c]["hollow"] else SURFACE, mew=1.3 if style[c]["hollow"] else 0.8)
            for c in order]


def cat_legend(ax, order, style, counts=None, title=None, anchor=(1.01, 0.5)):
    labels = [f"{c} ({counts[c]})" if counts is not None else c for c in order]
    ax.legend(cat_handles(order, style), labels, loc="center left", bbox_to_anchor=anchor, title=title,
              title_fontsize=10, labelcolor=INK2, handletextpad=0.4)


def fig_legend(fig, order, style, counts=None, ncol=5, y=-0.02):
    labels = [f"{c} ({counts[c]})" if counts is not None else c for c in order]
    fig.legend(cat_handles(order, style), labels, loc="upper center", bbox_to_anchor=(0.5, y), ncol=ncol,
               labelcolor=INK2, handletextpad=0.4, columnspacing=1.4)


def highlight_panel(ax, E, labels, c, style, s_fg=16):
    """Una categoría resaltada (con sus elipses) sobre el resto en gris."""
    m = labels == c
    st = style[c]
    span = float(np.ptp(E[:, :2], axis=0).max())
    ax.scatter(E[~m, 0], E[~m, 1], s=7, c=CONTEXT, linewidths=0, zorder=1)
    ax.scatter(E[m, 0], E[m, 1], s=s_fg * (1.5 if st["hollow"] else 1), marker=st["marker"],
               facecolors="none" if st["hollow"] else st["color"],
               edgecolors=st["color"] if st["hollow"] else SURFACE, linewidths=1.2 if st["hollow"] else 0.5, zorder=3)
    for g in compact_groups(E[m, :2], span):
        ax.add_patch(ellipse_patch(g, span, st["color"], fill=False))
    ax.set_title(f"{c}  (n = {m.sum()})", loc="left", fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)


def hbox(ax, values, labels, color=None):
    """Diagrama de caja horizontal con los puntos encima (sin depender de la versión de matplotlib)."""
    color = SERIES[0] if color is None else color
    rng = np.random.default_rng(RS)
    for i, v in enumerate(values):
        v = np.asarray(v, float)
        q1, med, q3 = np.percentile(v, [25, 50, 75])
        iqr = q3 - q1
        lo, hi = v[v >= q1 - 1.5 * iqr].min(), v[v <= q3 + 1.5 * iqr].max()
        ax.plot([lo, q1], [i, i], color=color, lw=1)
        ax.plot([q3, hi], [i, i], color=color, lw=1)
        ax.add_patch(FancyBboxPatch((q1, i - 0.28), max(q3 - q1, 1e-9), 0.56, boxstyle="square,pad=0",
                                    facecolor=to_rgba(color, 0.12), edgecolor=color, lw=1.2, zorder=2))
        ax.plot([med, med], [i - 0.28, i + 0.28], color=color, lw=2.2, zorder=3)
        ax.scatter(v, i + rng.uniform(-0.2, 0.2, len(v)), s=9, color=color, alpha=0.35, linewidths=0, zorder=4)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_ylim(len(labels) - 0.5, -0.5)
    ax.grid(axis="y", visible=False)


def heatmap(ax, M, row_labels, col_labels, cmap=None, vmin=None, vmax=None, annot=None, fmt=".2f",
            cbar_label=None, center=None, annot_size=9, cbar=True):
    """Mapa de calor con separación clara entre celdas. Con cbar=False no dibuja la barra de color
    (para añadirla aparte cuando la figura tiene más paneles)."""
    sns.heatmap(np.asarray(M, float), ax=ax, cmap=SEQ_CMAP if cmap is None else cmap, vmin=vmin, vmax=vmax,
                center=center, annot=annot if annot is not None else True, fmt=fmt if annot is None else "",
                annot_kws={"fontsize": annot_size}, linewidths=2, linecolor=SURFACE,
                xticklabels=col_labels, yticklabels=row_labels, cbar=cbar,
                cbar_kws={"label": cbar_label, "shrink": 0.75} if cbar_label else {"shrink": 0.75})
    ax.tick_params(length=0)
    ax.grid(False)


def rotate_xlabels(ax, angle=40):
    """Etiquetas del eje x inclinadas y alineadas con su columna."""
    plt.setp(ax.get_xticklabels(), rotation=angle, ha="right", rotation_mode="anchor")


def tsne_axes(ax, sx=1, sy=1):
    ax.set_xlabel(("−" if sx < 0 else "") + "t-SNE 1")
    ax.set_ylabel(("−" if sy < 0 else "") + "t-SNE 2")


def run_tsne(X_in, perplexity, n_components=2):
    """t-SNE con semilla fija. Antes comprime los datos a `tsne_pca_components` dimensiones, un paso
    interno estándar que quita ruido y acelera el cálculo."""
    X_red = X_in
    if CONFIG["tsne_pca_components"]:
        k = min(CONFIG["tsne_pca_components"], X_in.shape[1], X_in.shape[0] - 1)
        X_red = PCA(n_components=k, random_state=RS).fit_transform(X_in)
    perplexity = min(perplexity, (X_in.shape[0] - 1) / 3)       # t-SNE exige perplejidad < n/3
    return TSNE(n_components=n_components, perplexity=perplexity, init="pca", learning_rate="auto",
                random_state=RS).fit_transform(X_red)

---
# Parte 1 · Análisis y exploración de los datos

**Pregunta:** ¿qué contiene la tabla y se distinguen las clases de BGC por sus dominios Pfam?

| Sección | Qué hace | Salidas principales |
|---|---|---|
| 1.1 | Carga el CSV | — |
| 1.2 | Une las copias GenBank/RefSeq del mismo genoma | — |
| 1.3 | Cuenta los BGCs por clase, excluye las clases raras y detecta BGCs que se solapan | `1.3_bgcs_por_clase`, `1.3_bgcs_solapados` |
| 1.4 | Construye la matriz de dominios y describe cuántos dominios tiene cada BGC | `1.4_dominios_por_bgc` |
| 1.5 | Mapa t-SNE por clase (general, clase por clase y sensibilidad a la perplejidad) | `1.5a/b/c` |
| 1.6 | El mismo t-SNE en sus cuatro orientaciones | `1.6_tsne_cuatro_vistas` |
| 1.7 | t-SNE en 3D | `1.7_tsne_3d` |
| 1.8 | Círculo de correlación antes del modelo (25 % de los datos) | `1.8_circulo_correlacion_antes` |

## 1.1 Carga del CSV

Se identifican las columnas de dominios Pfam (numéricas, nombre `PF…:…`) y se muestran las primeras filas de los metadatos.

In [ ]:
raw = pd.read_csv(CONFIG["input_csv"], encoding="utf-8-sig", low_memory=False)    # utf-8-sig quita el BOM inicial
TARGET = CONFIG["target_column"]
assert TARGET in raw.columns, f"Falta la columna {TARGET}"

pfam_columns = [c for c in raw.columns
                if c.startswith(CONFIG["feature_prefix"]) and pd.api.types.is_numeric_dtype(raw[c])]
print(f"{len(raw):,} regiones BGC · {len(pfam_columns):,} columnas de dominios Pfam · "
      f"{raw.shape[1] - len(pfam_columns)} columnas de metadatos")
if "organism" in raw.columns:
    genera = raw["organism"].astype(str).str.split().str[0].value_counts()
    print(f"Organismos: {raw['organism'].nunique()} nombres distintos · género(s): "
          + ", ".join(f"{g} ({n} filas)" for g, n in genera.items()))

preview = [c for c in ("organism", CONFIG["assembly_column"], TARGET, CONFIG["region_column"], "length_kb", "num_cds_total")
           if c in raw.columns]
raw[preview].head()

## 1.2 Copias GenBank / RefSeq del mismo genoma

NCBI distribuye muchos genomas dos veces: `GCA_…` (GenBank) y `GCF_…` (RefSeq). Ambas copias producen los mismos BGCs con perfiles idénticos. Si se dejaran las dos, una copia podría quedar en entrenamiento y su gemela en prueba, y la exactitud saldría inflada. Se conserva una sola fila por genoma, clase, región y ubicación.

In [ ]:
ACC, REG, LOC = CONFIG["assembly_column"], CONFIG["region_column"], CONFIG["location_column"]
data = raw.copy()
data["genome"] = data[ACC].astype(str).str.replace(r"^GC[AF]_", "", regex=True)    # GCA_x.1 y GCF_x.1 → x.1

if CONFIG["collapse_assembly_redundancy"]:
    n_twins = int((data.groupby("genome")[ACC].nunique() > 1).sum())
    keys = ["genome", TARGET] + [c for c in (REG, LOC) if c in data.columns]
    data = data.drop_duplicates(subset=keys).reset_index(drop=True)
    print(f"{n_twins} genomas aparecen dos veces (GenBank y RefSeq) → "
          f"se quitan {len(raw) - len(data)} filas duplicadas · quedan {len(data)} "
          f"BGCs de {data['genome'].nunique()} genomas")

## 1.3 BGCs por clase

Las clases con menos de `min_class_size` BGCs se excluyen: no alcanzan para repartirse entre entrenamiento, validación y prueba, y sus métricas no serían interpretables. El orden de las clases que queda aquí (de mayor a menor) y sus colores se mantienen en todo el notebook.

In [ ]:
MIN_N = CONFIG["min_class_size"]
counts_all = data[TARGET].value_counts()
class_order = counts_all[counts_all >= MIN_N].index.tolist()      # orden fijo para todas las figuras
CLASS_STYLE = make_style(class_order)

inventory = save_table(pd.DataFrame({
    "clase": counts_all.index, "n_BGC": counts_all.values,
    "porcentaje": (100 * counts_all.values / counts_all.sum()).round(1),
    "estado": np.where(counts_all.values >= MIN_N, "retenida", f"excluida (< {MIN_N})")}),
    "1.3", "bgcs_por_clase")
data = data[data[TARGET].isin(class_order)].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8.5, 0.36 * len(counts_all) + 1.8))
kept = counts_all.index.isin(class_order)
ax.barh(np.arange(len(counts_all)), counts_all.values, height=0.62, color=np.where(kept, SERIES[0], CONTEXT))
for i, v in enumerate(counts_all.values):
    ax.text(v + counts_all.max() * 0.01, i, str(v), va="center", fontsize=9.5, color=INK2)
ax.axvline(MIN_N, color=MUTED, lw=1)
ax.text(MIN_N + counts_all.max() * 0.01, len(counts_all) - 0.6, f"mínimo = {MIN_N}", color=MUTED, fontsize=9)
ax.set_yticks(np.arange(len(counts_all)))
ax.set_yticklabels(counts_all.index)
ax.set_ylim(len(counts_all) - 0.4, -0.6)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Número de BGCs (tras unir GenBank/RefSeq)")
titles(ax, "BGCs por clase de producto",
       f"{len(data)} BGCs en {len(class_order)} clases retenidas (azul) · en gris, clases con menos de {MIN_N} BGCs, excluidas")
show_and_save(fig, "1.3", "bgcs_por_clase")
inventory

### BGCs del mismo genoma que se solapan

Algunas filas del CSV describen tramos del genoma que se solapan entre sí con clases distintas (por ejemplo, un BGC dentro de las coordenadas de otro BGC del mismo genoma). No se eliminan, pero conviene saberlo: si dos filas cubren casi el mismo tramo, sus perfiles de dominios son casi iguales aunque tengan clases distintas, y ningún clasificador puede separarlas. La tabla `1.3_bgcs_solapados` lista cada par y la sección 3.4 indica qué errores del modelo caen en estos casos.

Se considera que dos BGCs se solapan si pertenecen al mismo genoma y sus coordenadas (`location`) se cruzan.

In [ ]:
overlap_rows, overlap_classes = [], {}                          # overlap_classes[fila] = clases de los BGCs que se le solapan
if LOC in data.columns:
    coords = data[LOC].astype(str).str.extract(r"(\d+)\D+(\d+)").astype(float).to_numpy()
    for _, idx in data.groupby("genome").indices.items():
        for a in range(len(idx)):
            for b in range(a + 1, len(idx)):
                i, j = idx[a], idx[b]
                ov = min(coords[i, 1], coords[j, 1]) - max(coords[i, 0], coords[j, 0])
                if ov > 0:
                    shorter = min(coords[i, 1] - coords[i, 0], coords[j, 1] - coords[j, 0])
                    pair = sorted([(data.at[i, TARGET], i), (data.at[j, TARGET], j)])
                    overlap_rows.append({ACC: data.at[i, ACC], "clase_a": pair[0][0], "fila_a": pair[0][1],
                                         "clase_b": pair[1][0], "fila_b": pair[1][1], "solape_kb": round(ov / 1000, 1),
                                         "pct_del_mas_corto": round(100 * ov / shorter, 1)})
                    overlap_classes.setdefault(i, set()).add(data.at[j, TARGET])
                    overlap_classes.setdefault(j, set()).add(data.at[i, TARGET])
overlaps = save_table(pd.DataFrame(overlap_rows), "1.3", "bgcs_solapados")

if overlaps.empty:
    print("Ningún par de BGCs del mismo genoma se solapa.")
else:
    overlaps["par"] = overlaps["clase_a"] + " + " + overlaps["clase_b"]
    overlap_summary = (overlaps.groupby("par").agg(pares=("solape_kb", "size"),
                                                         pct_del_mas_corto_mediana=("pct_del_mas_corto", "median"))
                       .sort_values("pares", ascending=False).reset_index())
    print(f"{len(overlaps)} pares de BGCs del mismo genoma se solapan ({len(overlap_classes)} BGCs implicados)")
    display(overlap_summary)

## 1.4 Matriz de dominios Pfam

`X` contiene la puntuación de cada dominio en cada BGC (0 = ausente) y `y` la clase. Los dominios que nunca varían se eliminan porque no aportan información. La figura muestra cuántos dominios Pfam distintos tiene cada BGC, por clase.

**Cómo leerla:** cada punto es un BGC; la caja abarca el 50 % central y la línea gruesa marca la mediana.

In [ ]:
X_frame = data[pfam_columns].astype(float).fillna(0.0)
n_before = X_frame.shape[1]
if CONFIG["drop_zero_variance"]:
    X_frame = X_frame.loc[:, X_frame.std(axis=0) > 0]           # dominios constantes: no discriminan
feature_names = X_frame.columns.tolist()
domain_names = [f.split(":", 1)[-1] for f in feature_names]       # "PF00005.34:ABC_tran" → "ABC_tran"
X, y = X_frame.to_numpy(), data[TARGET].to_numpy()
P = X > 0                                                         # presencia / ausencia de cada dominio
n_domains = P.sum(axis=1)
n_profiles = len({row.tobytes() for row in P})

print(f"Matriz X: {X.shape[0]} BGCs × {X.shape[1]} dominios ({n_before - X.shape[1]} sin variación, eliminados)")
print(f"{100 * P.mean():.1f} % de las celdas tienen un dominio · valores no nulos entre "
      f"{X[P].min():.2f} y {X[P].max():.2f}")
print(f"Perfiles de presencia/ausencia distintos: {n_profiles} de {len(y)} BGCs")

dom_table = (pd.DataFrame({"cls": y, "n": n_domains}).groupby("cls")["n"].agg(["size", "min", "median", "max"])
             .reindex(class_order).reset_index())
dom_table.columns = ["clase", "n_BGC", "mínimo", "mediana", "máximo"]
save_table(dom_table, "1.4", "dominios_por_bgc")

fig, ax = plt.subplots(figsize=(8.5, 0.5 * len(class_order) + 1.8))
hbox(ax, [n_domains[y == c] for c in class_order], class_order)
ax.set_xlabel("Dominios Pfam distintos en el BGC")
titles(ax, "Dominios Pfam por BGC, por clase",
       "Cada punto es un BGC · caja = 50 % central · línea gruesa = mediana")
show_and_save(fig, "1.4", "dominios_por_bgc")
dom_table

## 1.5 Mapa t-SNE de los BGCs

t-SNE coloca cerca a los BGCs con perfiles de dominios parecidos. Sirve para ver si las clases forman grupos propios **antes** de entrenar ningún modelo. Se generan tres figuras:

- **1.5a · Mapa general:** todas las clases a la vez. Cada elipse rodea un grupo compacto de BGCs de la misma clase (95 %); una clase puede formar varios grupos.
- **1.5b · Clase por clase:** el mismo mapa una vez por clase, resaltando esa clase (el resto en gris). Es la forma más clara de ver dónde cae cada clase y si se solapa con otra.
- **1.5c · Sensibilidad a la perplejidad:** el mismo análisis con tres valores del parámetro principal de t-SNE. Si los grupos se mantienen, no son un artefacto del parámetro.

**Precaución:** en t-SNE las distancias *entre* grupos y su tamaño no tienen significado; solo importa qué BGCs quedan juntos.

In [ ]:
perp_main = CONFIG["tsne_perplexity_main"]
X_scaled = StandardScaler().fit_transform(X)                     # media 0 y varianza 1 por dominio
tsne_runs = {p: run_tsne(X_scaled, p) for p in sorted(set(CONFIG["tsne_perplexities"]) | {perp_main})}
tsne_all = tsne_runs[perp_main]
counts = pd.Series(y).value_counts()
save_table(pd.DataFrame({"clase": y, "tSNE_1": tsne_all[:, 0].round(3), "tSNE_2": tsne_all[:, 1].round(3)}),
           "1.5", "coordenadas_tsne")

# ---- 1.5a Mapa general -----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 7.4))
cat_scatter(ax, tsne_all, y, class_order, CLASS_STYLE)
tsne_axes(ax)
cat_legend(ax, class_order, CLASS_STYLE, counts, title="Clase (n BGCs)")
titles(ax, "Mapa t-SNE de los BGCs, coloreado por clase",
       f"{len(y)} BGCs · {X.shape[1]} dominios Pfam · perplejidad {perp_main}\n"
       "Cada elipse rodea un grupo compacto de BGCs de la misma clase (95 %)")
show_and_save(fig, "1.5", "a_tsne_por_clase")

# ---- 1.5b Clase por clase (el resto en gris) --------------------------------
ncol = 3
nrow = int(np.ceil(len(class_order) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.3 * ncol, 3.8 * nrow), squeeze=False)
for ax, c in zip(axes.ravel(), class_order):
    highlight_panel(ax, tsne_all, y, c, CLASS_STYLE)
for ax in axes.ravel()[len(class_order):]:
    ax.axis("off")
fig.suptitle("Dónde cae cada clase en el mapa t-SNE (en gris, el resto de los BGCs)",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
show_and_save(fig, "1.5", "b_tsne_clase_por_clase")
for c in class_order:                                            # paneles individuales
    fig, ax = plt.subplots(figsize=(6.5, 5.4))
    highlight_panel(ax, tsne_all, y, c, CLASS_STYLE, s_fg=26)
    show_and_save(fig, "1.5", f"b_tsne_clase_{slug(c)}", individual=True)

# ---- 1.5c Sensibilidad a la perplejidad ------------------------------------------------
perps = sorted(CONFIG["tsne_perplexities"])
fig, axes = plt.subplots(1, len(perps), figsize=(5.3 * len(perps), 5.0))
for ax, p in zip(np.atleast_1d(axes), perps):
    cat_scatter(ax, tsne_runs[p], y, class_order, CLASS_STYLE, direct_labels=False, s=14)
    tsne_axes(ax)
    ax.set_title(f"Perplejidad {p}", loc="left", fontsize=11)
fig.suptitle("El mismo mapa t-SNE con tres perplejidades", x=0.01, ha="left",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig_legend(fig, class_order, CLASS_STYLE, counts)
show_and_save(fig, "1.5", "c_tsne_perplejidades")

## 1.6 El mismo t-SNE en sus cuatro orientaciones

Los ejes de t-SNE no tienen signo ni unidades: invertir un eje produce un mapa equivalente. Aquí se muestran las cuatro orientaciones (±t-SNE 1 × ±t-SNE 2) para elegir la más legible o la que coincida con otra figura. Los datos son exactamente los mismos en las cuatro. Cada vista también se guarda como figura individual.

In [ ]:
views = [(+1, +1, "t-SNE 1 vs t-SNE 2", "1"), (+1, -1, "t-SNE 1 vs −t-SNE 2", "2"),
         (-1, +1, "−t-SNE 1 vs t-SNE 2", "3"), (-1, -1, "−t-SNE 1 vs −t-SNE 2", "4")]

fig, axes = plt.subplots(2, 2, figsize=(12.5, 10.6))
for ax, (sx, sy, name, k) in zip(axes.ravel(), views):
    cat_scatter(ax, tsne_all, y, class_order, CLASS_STYLE, flip=(sx, sy), s=16, label_size=8.5)
    tsne_axes(ax, sx, sy)
    ax.set_title(f"Vista {k}: {name}", loc="left", fontsize=11)
fig.suptitle("Mapa t-SNE en sus cuatro orientaciones (mismos datos)", x=0.01,
             ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
fig_legend(fig, class_order, CLASS_STYLE, counts)
show_and_save(fig, "1.6", "tsne_cuatro_vistas")

for sx, sy, name, k in views:                                    # una figura por vista
    fig, ax = plt.subplots(figsize=(10, 7.4))
    cat_scatter(ax, tsne_all, y, class_order, CLASS_STYLE, flip=(sx, sy))
    tsne_axes(ax, sx, sy)
    cat_legend(ax, class_order, CLASS_STYLE, counts, title="Clase (n BGCs)")
    titles(ax, f"Mapa t-SNE, vista {k}: {name}")
    show_and_save(fig, "1.6", f"tsne_vista{k}", individual=True)

## 1.7 t-SNE en 3D

t-SNE con tres dimensiones de salida, visto desde tres ángulos. La tercera dimensión no es una "fuente de varianza": el algoritmo reparte la estructura de vecindad entre los tres ejes. Sirve para comprobar si dos clases que se enciman en 2D se separan en 3D.

In [ ]:
tsne_3d = run_tsne(X_scaled, perp_main, n_components=3)
angles = [(25, 45), (25, 135), (60, -60)]

fig = plt.figure(figsize=(18, 6.2))
for i, (elev, azim) in enumerate(angles, start=1):
    ax = fig.add_subplot(1, 3, i, projection="3d")
    for c in class_order:
        m, st = y == c, CLASS_STYLE[c]
        ax.scatter(*tsne_3d[m].T, s=16 * (1.5 if st["hollow"] else 1), marker=st["marker"], depthshade=False,
                   facecolors="none" if st["hollow"] else st["color"],
                   edgecolors=st["color"] if st["hollow"] else SURFACE, linewidths=1.1 if st["hollow"] else 0.4)
    ax.view_init(elev=elev, azim=azim)
    ax.set_box_aspect(None, zoom=0.88)
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    ax.set_xlabel("t-SNE 1", labelpad=-8); ax.set_ylabel("t-SNE 2", labelpad=-8); ax.set_zlabel("t-SNE 3", labelpad=-8)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color(to_rgba(SURFACE, 1.0))
    ax.set_title(f"Ángulo {i}: elevación {elev}°, azimut {azim}°", loc="left", fontsize=11)
fig.suptitle("Mapa t-SNE en 3D, coloreado por clase", x=0.01, ha="left", fontsize=13,
             fontweight="bold")
fig_legend(fig, class_order, CLASS_STYLE, counts, y=0.04)
show_and_save(fig, "1.7", "tsne_3d")

## 1.8 Círculo de correlación — antes del modelo (25 % de los datos)

Indica **qué dominios organizan el mapa**. Para cada dominio Pfam se calcula su correlación con el eje 1 y con el eje 2 de un t-SNE, y se dibuja como una flecha:

- se usa una **submuestra estratificada del 25 %** de los BGCs (la misma se reutiliza en la sección 3.5, después del modelo, para comparar);
- "antes del modelo" = con **todos** los dominios; se dibujan los `corr_circle_top_features` con mayor correlación;
- una flecha larga, cerca del círculo exterior, indica que el dominio cambia mucho a lo largo de esa dirección del mapa: los BGCs hacia los que apunta la flecha tienen ese dominio. El círculo punteado marca |r| = 0.5.

**Precaución:** t-SNE no es lineal, así que estas correlaciones son descriptivas: dicen qué dominios varían a lo largo de cada eje, no son las cargas de una proyección.

In [ ]:
_, corr_idx = train_test_split(np.arange(len(y)), test_size=CONFIG["corr_circle_fraction"], stratify=y, random_state=RS)
corr_idx = np.sort(corr_idx)                                    # misma submuestra antes y después del modelo


def correlation_circle(X_sub, y_sub, names, section, name, title):
    """t-SNE de la submuestra + correlación de cada dominio con los dos ejes del mapa."""
    Xs = StandardScaler().fit_transform(X_sub)
    emb = run_tsne(Xs, perp_main)
    E = (emb - emb.mean(axis=0)) / emb.std(axis=0)
    corr = Xs.T @ E / len(y_sub)                                # r de Pearson (ambas variables estandarizadas)
    norm = np.linalg.norm(corr, axis=1)
    top = np.argsort(norm)[::-1][:CONFIG["corr_circle_top_features"]]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7), gridspec_kw={"width_ratios": [1.1, 1]})
    cat_scatter(ax1, emb, y_sub, class_order, CLASS_STYLE, s=24, label_size=8.5)
    tsne_axes(ax1)
    ax1.set_title(f"BGCs de la submuestra (n = {len(y_sub)})", loc="left", fontsize=11)

    # Dominios con la misma correlación (aparecen juntos en los mismos BGCs) comparten una flecha
    bundles = []
    for i in top:
        for b in bundles:
            if np.linalg.norm(corr[i] - corr[b[0]]) < 0.03:
                b.append(i)
                break
        else:
            bundles.append([i])
    labels = ["\n".join(names[i].split(":", 1)[-1] for i in b[:3]) + (f"\n(+{len(b) - 3})" if len(b) > 3 else "") for b in bundles]
    sizes = [(0.03 * max(len(t) for t in lab.split("\n")) + 0.06, 0.085 * (lab.count("\n") + 1)) for lab in labels]

    theta = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(theta), np.sin(theta), color=AXIS, lw=1)
    ax2.plot(0.5 * np.cos(theta), 0.5 * np.sin(theta), color=AXIS, lw=0.8, ls=":")
    ax2.axhline(0, color=GRID, lw=0.8)
    ax2.axvline(0, color=GRID, lw=0.8)
    heads = np.array([corr[b[0]] for b in bundles])
    label_pos = repel(heads * 1.18, sizes)
    for (hx, hy), (lx, ly), lab in zip(heads, label_pos, labels):
        ax2.annotate("", xy=(hx, hy), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1.1))
        ax2.plot([hx, lx], [hy, ly], color=CONTEXT, lw=0.6)
        ax2.text(lx, ly, lab, fontsize=8.5, ha="center", va="center", color=INK, path_effects=HALO, linespacing=1.1)
    ax2.set_xlim(-1.5, 1.5)
    ax2.set_ylim(-1.5, 1.5)
    ax2.set_aspect("equal")
    ax2.grid(False)
    ax2.set_xlabel("correlación con t-SNE 1")
    ax2.set_ylabel("correlación con t-SNE 2")
    ax2.set_title(f"Los {len(top)} dominios más correlacionados con el mapa\n"
                  "(los dominios que aparecen juntos en los mismos BGCs comparten flecha)",
                  loc="left", fontsize=11)
    fig.suptitle(title, x=0.01, ha="left", fontsize=13, fontweight="bold")
    fig.tight_layout()
    show_and_save(fig, section, name)
    return pd.DataFrame({"dominio": [names[i] for i in top], "r_tSNE_1": corr[top, 0].round(3),
                         "r_tSNE_2": corr[top, 1].round(3), "longitud": norm[top].round(3)})


corr_before = save_table(
    correlation_circle(X[corr_idx], y[corr_idx], feature_names, "1.8", "circulo_correlacion_antes",
                       f"Círculo de correlación antes del modelo: todos los dominios ({X.shape[1]})"),
    "1.8", "circulo_correlacion_antes")
corr_before

---
# Parte 2 · Entrenamiento del modelo

**Pregunta:** ¿cuántos dominios necesita el clasificador y está sobreajustado?

| Sección | Qué hace | Salidas principales |
|---|---|---|
| 2.1 | Aparta el 30 % de los BGCs como prueba (no se toca hasta la Parte 3) | `2.1_balance_particiones` |
| 2.2 | Ordena los dominios con RFE y elige cuántos usar | `2.2_curva_seleccion_dominios` |
| 2.3 | Elige la regularización `C` del SVM | `2.3_curva_validacion_C` |
| 2.4 | Curva de aprendizaje | `2.4_curva_aprendizaje` |
| 2.5 | Prueba de permutación | `2.5_prueba_permutacion` |
| 2.6 | Ajusta el modelo final | `2.6_matriz_confusion_entrenamiento` |
| 2.7 | Ranking de los dominios del modelo (*loadings*) | `2.7_ranking_dominios` |

**Regla de oro:** todo lo de esta parte usa solo el 70 % de entrenamiento. La prueba apartada se abre una única vez, en la Parte 3.

## 2.1 Partición estratificada 70 / 30

El 30 % de los BGCs se aparta como **prueba** y no se usa para ninguna decisión. La partición es estratificada: cada clase conserva su proporción en entrenamiento y en prueba (la tabla lo verifica).

In [ ]:
idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=CONFIG["test_size"], stratify=y, random_state=RS)
X_train, y_train = X[idx_train], y[idx_train]
SEALED_TEST = {"idx": idx_test}           # se abre solo en la Parte 3

balance = pd.DataFrame({"clase": class_order,
                        "total": [int((y == c).sum()) for c in class_order],
                        "entrenamiento": [int((y_train == c).sum()) for c in class_order],
                        "prueba": [int((y[idx_test] == c).sum()) for c in class_order]})
for col in ("total", "entrenamiento", "prueba"):
    balance[f"% {col}"] = (100 * balance[col] / balance[col].sum()).round(1)
pct = balance[[c for c in balance.columns if c.startswith("%")]]
max_dev = float((pct.max(axis=1) - pct.min(axis=1)).max())
save_table(balance, "2.1", "balance_particiones")
print(f"Entrenamiento: {len(idx_train)} BGCs · prueba apartada: {len(idx_test)} BGCs · "
      f"mayor diferencia de proporción entre particiones: {max_dev:.1f} puntos porcentuales")
balance

## 2.2 ¿Cuántos dominios necesita el modelo? (RFE)

1. El entrenamiento se divide otra vez (80 / 20) en **entrenamiento interno** y **validación**.
2. **RFE** (eliminación recursiva de características) con un SVM lineal ordena los dominios de más a menos útil, usando solo el entrenamiento interno.
3. Para cada número de dominios *k* se mide la exactitud por validación cruzada (en el entrenamiento interno) y en la validación independiente.
4. **Regla de un error estándar:** se elige el *k* más pequeño cuya exactitud de validación queda a menos de 1 error estándar del máximo. Así el modelo usa pocos dominios sin perder exactitud.

**Cómo leer la figura:** la línea azul es optimista, porque RFE ya vio esos BGCs; la decisión se toma con la línea naranja (validación independiente). La banda de color = ± 1 error estándar.

In [ ]:
X_inner, X_valid, y_inner, y_valid = train_test_split(X_train, y_train, test_size=CONFIG["validation_size"],
                                                      stratify=y_train, random_state=RS)

# ---- Ranking RFE (solo entrenamiento interno) ---------------------------------------
rfe = RFE(SVC(kernel="linear", C=CONFIG["svm_c"], random_state=RS), n_features_to_select=1, step=CONFIG["rfe_step"])
rfe.fit(StandardScaler().fit_transform(X_inner), y_inner)         # hasta 1 dominio → ranking completo
ranked_idx = np.argsort(rfe.ranking_, kind="stable")             # de más a menos útil
save_table(pd.DataFrame({"rank_rfe": np.arange(1, len(ranked_idx) + 1),
                         "dominio": [feature_names[i] for i in ranked_idx]}), "2.2", "ranking_rfe")

# ---- Exactitud frente al número de dominios ------------------------------------------
rows = []
for k in [k for k in CONFIG["feature_grid"] if k <= X_inner.shape[1]]:
    cols = ranked_idx[:k]
    cv_mean, cv_se = cv_summary(cross_val_score(make_estimator(), X_inner[:, cols], y_inner, cv=make_cv(),
                                                scoring="accuracy", n_jobs=CONFIG["n_jobs"]))
    acc = accuracy_score(y_valid, make_estimator().fit(X_inner[:, cols], y_inner).predict(X_valid[:, cols]))
    lo, hi = wilson_interval(round(acc * len(y_valid)), len(y_valid), z=1.0)
    rows.append({"k": k, "cv_accuracy": cv_mean, "cv_se": cv_se, "validation_accuracy": acc,
                 "validation_low_1se": lo, "validation_high_1se": hi})
curve = pd.DataFrame(rows)
save_table(curve.round(4), "2.2", "curva_seleccion_dominios")

# ---- Regla de un error estándar -----------------------------------------------------------
best = curve.loc[curve["validation_accuracy"].idxmax()]
se_best = np.sqrt(best["validation_accuracy"] * (1 - best["validation_accuracy"]) / len(y_valid))
threshold = best["validation_accuracy"] - se_best
k_optimal = int(curve.loc[curve["validation_accuracy"] >= threshold, "k"].min())
sel_idx = ranked_idx[:k_optimal]
selected_features = [feature_names[i] for i in sel_idx]

fig, ax = plt.subplots(figsize=(9.5, 5.6))
ax.fill_between(curve["k"], curve["cv_accuracy"] - curve["cv_se"], curve["cv_accuracy"] + curve["cv_se"],
                color=SERIES[0], alpha=0.12, lw=0)
ax.plot(curve["k"], curve["cv_accuracy"], color=SERIES[0], marker="o", ms=4,
        label="Validación cruzada en el entrenamiento interno (optimista)")
ax.fill_between(curve["k"], curve["validation_low_1se"], curve["validation_high_1se"], color=SERIES[1], alpha=0.12, lw=0)
ax.plot(curve["k"], curve["validation_accuracy"], color=SERIES[1], marker="o", ms=4,
        label="Validación independiente (decide k)")
ax.axhline(threshold, color=MUTED, lw=1)
ax.text(curve["k"].min(), threshold + 0.005, "máximo de validación − 1 error estándar", color=MUTED, fontsize=9,
        ha="left", va="bottom")
ax.axvline(k_optimal, color=INK2, lw=1)
ax.annotate(f"k = {k_optimal} dominios", xy=(k_optimal, threshold), xytext=(8, -32), textcoords="offset points",
            fontsize=10.5, fontweight="bold", color=INK, path_effects=HALO)
ax.set_xscale("log")
ticks = [t for t in (1, 2, 5, 10, 20, 50, 100, 200) if t <= curve["k"].max()]
ax.set_xticks(ticks)
ax.set_xticklabels([str(t) for t in ticks])
ax.set_xlabel("Número de dominios Pfam usados (escala logarítmica)")
ax.set_ylabel("Exactitud")
ax.legend(loc="lower right")
titles(ax, "¿Cuántos dominios necesita el modelo?",
       f"Se elige el menor número de dominios a menos de 1 error estándar del máximo de validación → k = {k_optimal}")
show_and_save(fig, "2.2", "curva_seleccion_dominios")
print(f"Máximo de validación: {best['validation_accuracy']:.3f} con k = {int(best['k'])} · umbral = {threshold:.3f} "
      f"→ se usan k = {k_optimal} dominios")

## 2.3 Control de sobreajuste: regularización `C`

`C` controla cuánto se ajusta el SVM a los datos de entrenamiento: un `C` grande da un margen estrecho y riesgo de sobreajuste. Con los *k* dominios elegidos se compara la exactitud en entrenamiento y por validación cruzada para varios `C`, y se elige el `C` **más pequeño** (más regularizado) cuya exactitud de validación cruzada queda a menos de 1 error estándar del máximo.

**Cómo leer la figura:** si la línea de entrenamiento sube y la de validación cruzada no, el modelo memoriza en lugar de generalizar.

In [ ]:
Xtr_sel = X_train[:, sel_idx]
rows = []
for C in CONFIG["c_grid"]:
    m, se = cv_summary(cross_val_score(make_estimator(C), Xtr_sel, y_train, cv=make_cv(), scoring="accuracy",
                                       n_jobs=CONFIG["n_jobs"]))
    train_acc = make_estimator(C).fit(Xtr_sel, y_train).score(Xtr_sel, y_train)     # resustitución
    rows.append({"C": C, "train_accuracy": train_acc, "cv_accuracy": m, "cv_se": se})
c_curve = pd.DataFrame(rows)
save_table(c_curve.round(4), "2.3", "curva_validacion_C")

best_c = c_curve.loc[c_curve["cv_accuracy"].idxmax()]
eligible = c_curve.loc[c_curve["cv_accuracy"] >= best_c["cv_accuracy"] - best_c["cv_se"], "C"]
C_final = float(eligible.min()) if CONFIG["tune_c"] else CONFIG["svm_c"]

fig, ax = plt.subplots(figsize=(9, 5.2))
ax.plot(c_curve["C"], c_curve["train_accuracy"], color=SERIES[0], marker="o", ms=4, label="Entrenamiento (resustitución)")
ax.fill_between(c_curve["C"], c_curve["cv_accuracy"] - c_curve["cv_se"], c_curve["cv_accuracy"] + c_curve["cv_se"],
                color=SERIES[1], alpha=0.12, lw=0)
ax.plot(c_curve["C"], c_curve["cv_accuracy"], color=SERIES[1], marker="o", ms=4, label="Validación cruzada")
ax.axvline(C_final, color=INK2, lw=1)
ax.annotate(f"C elegido = {C_final:g}", xy=(C_final, ax.get_ylim()[0]), xytext=(6, 12), textcoords="offset points",
            fontsize=10.5, fontweight="bold", color=INK, path_effects=HALO)
ax.set_xscale("log")
ax.set_xlabel("C (regularización inversa, escala logarítmica)")
ax.set_ylabel("Exactitud")
ax.legend(loc="lower right")
titles(ax, f"Regularización del SVM con k = {k_optimal} dominios",
       "Se elige el C más pequeño a menos de 1 error estándar del máximo de validación cruzada")
show_and_save(fig, "2.3", "curva_validacion_C")
print(f"C con mayor exactitud de validación cruzada: {best_c['C']:g} ({best_c['cv_accuracy']:.3f}) · C elegido: {C_final:g}")

## 2.4 Control de sobreajuste: curva de aprendizaje

Exactitud en entrenamiento y por validación cruzada (5 particiones) a medida que se usan más BGCs para entrenar.

**Cómo leerla:** si ambas curvas terminan juntas y altas, el modelo generaliza; si la de entrenamiento queda muy por encima de la de validación, memoriza.

In [ ]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
sizes, tr_scores, va_scores = learning_curve(make_estimator(C_final), Xtr_sel, y_train, cv=cv5, scoring="accuracy",
                                             train_sizes=CONFIG["learning_curve_sizes"], n_jobs=CONFIG["n_jobs"],
                                             shuffle=True, random_state=RS)
lc = pd.DataFrame({"n_train": sizes, "train_mean": tr_scores.mean(axis=1), "train_sd": tr_scores.std(axis=1),
                   "cv_mean": va_scores.mean(axis=1), "cv_sd": va_scores.std(axis=1)})
save_table(lc.round(4), "2.4", "curva_aprendizaje")

fig, ax = plt.subplots(figsize=(9, 5.2))
for col, sd, color, label in (("train_mean", "train_sd", SERIES[0], "Entrenamiento"),
                              ("cv_mean", "cv_sd", SERIES[1], "Validación cruzada (5 particiones)")):
    ax.fill_between(lc["n_train"], lc[col] - lc[sd], lc[col] + lc[sd], color=color, alpha=0.12, lw=0)
    ax.plot(lc["n_train"], lc[col], color=color, marker="o", ms=4, label=label)
ax.set_xlabel("BGCs usados para entrenar")
ax.set_ylabel("Exactitud")
ax.legend(loc="lower right")
gap_lc = lc["train_mean"].iloc[-1] - lc["cv_mean"].iloc[-1]
titles(ax, "Curva de aprendizaje",
       f"Diferencia final entrenamiento − validación = {gap_lc:+.3f} · banda = ± 1 desviación estándar entre particiones")
show_and_save(fig, "2.4", "curva_aprendizaje")

## 2.5 Control de sobreajuste: prueba de permutación

Se barajan las clases al azar y se reentrena el modelo `n_permutations` veces. Si el modelo real no supera con claridad a los modelos entrenados con clases barajadas, su exactitud no sería real.

**Cómo leerla:** las barras grises son la exactitud con clases al azar; la línea azul es el modelo real. El valor *p* es la proporción de permutaciones que igualan o superan al modelo real (el mínimo posible es 1 / (permutaciones + 1)).

In [ ]:
score_real, perm_scores, p_perm = permutation_test_score(
    make_estimator(C_final), Xtr_sel, y_train, cv=cv5, scoring="accuracy",
    n_permutations=CONFIG["n_permutations"], n_jobs=CONFIG["n_jobs"], random_state=RS)
save_table(pd.DataFrame({"exactitud_con_clases_permutadas": perm_scores.round(4)}),
           "2.5", "prueba_permutacion")

fig, ax = plt.subplots(figsize=(9, 5.0))
ax.hist(perm_scores, bins=np.arange(0, 1.0001, 0.01), color=AXIS, edgecolor=SURFACE, linewidth=0.6,
        label="Clases permutadas al azar")             # barras de 0.01 de ancho
ax.axvline(score_real, color=SERIES[0], lw=2.5, label=f"Modelo real = {score_real:.3f}")
ax.axvline(1 / len(class_order), color=MUTED, lw=1, label=f"Azar (1 / número de clases) = {1 / len(class_order):.3f}")
ax.set_xlim(0, 1.02)
ax.set_xlabel("Exactitud por validación cruzada")
ax.set_ylabel("Permutaciones")
ax.legend(loc="upper center")
titles(ax, "Prueba de permutación",
       f"{CONFIG['n_permutations']} permutaciones · p = {p_perm:.4f}")
show_and_save(fig, "2.5", "prueba_permutacion")

## 2.6 Modelo final

Se reajusta el modelo con todo el entrenamiento (70 %), los *k* dominios elegidos y el `C` elegido. `probability=True` permite que el modelo exportado dé la probabilidad de cada clase.

La matriz de confusión de entrenamiento es casi perfecta por construcción (el modelo ya vio esos BGCs); la exactitud honesta es la de validación cruzada que se imprime abajo y, sobre todo, la de la prueba apartada (Parte 3).

**Cómo leer la matriz:** filas = clase real, columnas = clase predicha. El número es la cantidad de BGCs y el color el porcentaje de la clase real; la diagonal son los aciertos.

In [ ]:
def confusion_figure(y_true, y_pred, section, name, title):
    """Matriz de confusión: número = BGCs; color = % de la clase real (fila)."""
    cm = confusion_matrix(y_true, y_pred, labels=class_order)
    share = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(8.6, 7.2))
    heatmap(ax, share, class_order, class_order, vmin=0, vmax=1, annot=np.where(cm > 0, cm.astype(str), ""),
            cbar_label="Fracción de la clase real", annot_size=10)
    ax.set_xlabel("Clase predicha")
    ax.set_ylabel("Clase real")
    rotate_xlabels(ax)
    ok = int(np.trace(cm))
    titles(ax, title, f"{ok} de {cm.sum()} BGCs en la diagonal (aciertos) · celdas vacías = 0")
    show_and_save(fig, section, name)
    return pd.DataFrame(cm, index=class_order, columns=class_order).rename_axis("clase_real").reset_index()


final_model = make_estimator(C_final, probability=True).fit(Xtr_sel, y_train)
y_train_pred = final_model.predict(Xtr_sel)
metrics_train, per_class_train = compute_metrics(y_train, y_train_pred, class_order, "entrenamiento (resustitución)")
cv_final, cv_final_se = cv_summary(cross_val_score(make_estimator(C_final), Xtr_sel, y_train, cv=make_cv(),
                                                   scoring="accuracy", n_jobs=CONFIG["n_jobs"]))
metrics_train["cv_accuracy"], metrics_train["cv_se"] = round(cv_final, 4), round(cv_final_se, 4)

save_table(confusion_figure(y_train, y_train_pred, "2.6", "matriz_confusion_entrenamiento",
                            "Matriz de confusión — entrenamiento"),
           "2.6", "matriz_confusion_entrenamiento")
print(f"Modelo final: {len(y_train)} BGCs · {k_optimal} dominios · C = {C_final:g} · "
      f"{int(final_model.named_steps['svm'].n_support_.sum())} vectores de soporte")
print(f"Exactitud en entrenamiento (resustitución): {metrics_train.at[0, 'accuracy']:.3f} · "
      f"por validación cruzada: {cv_final:.3f} ± {cv_final_se:.3f} (error estándar)")

## 2.7 Ranking de los dominios del modelo (*loadings*)

Para cada uno de los *k* dominios del modelo:

| Columna | Qué mide |
|---|---|
| `rank_svm` / `peso_svm` | Peso del dominio en el SVM final: norma de sus coeficientes sumando todos los clasificadores uno-contra-uno. Es el *loading* del modelo: cuánto influye el dominio en las fronteras de decisión |
| `rank_rfe` | Posición en el ranking de RFE (1 = el último en ser eliminado) |
| `prevalencia_entrenamiento` | Fracción de BGCs de entrenamiento que tienen el dominio |
| `clase_dominante` | Clase con la puntuación media más alta del dominio: ayuda a la lectura biológica |

**Cómo leer la figura:** barras más largas = dominios que más pesan en la decisión del modelo; el texto al final de cada barra indica en qué clase abunda ese dominio.

In [ ]:
W = final_model.named_steps["svm"].coef_                     # (pares uno-contra-uno × k), espacio estandarizado
class_means = pd.DataFrame(Xtr_sel, columns=selected_features).groupby(y_train).mean()
loadings = pd.DataFrame({
    "dominio": [f.split(":", 1)[-1] for f in selected_features],
    "columna_pfam": selected_features,
    "rank_rfe": np.arange(1, k_optimal + 1),
    "peso_svm": np.sqrt((W ** 2).sum(axis=0)).round(4),
    "prevalencia_entrenamiento": (Xtr_sel > 0).mean(axis=0).round(3),
    "clase_dominante": class_means.idxmax(axis=0).to_numpy()})
loadings = loadings.sort_values("peso_svm", ascending=False).reset_index(drop=True)
loadings.insert(0, "rank_svm", np.arange(1, len(loadings) + 1))
save_table(loadings, "2.7", "ranking_dominios")

top = loadings.head(20)
fig, ax = plt.subplots(figsize=(9.5, 7.6))
ax.barh(np.arange(len(top)), top["peso_svm"], height=0.62, color=SERIES[0])
xmax = top["peso_svm"].max()
for i, (v, c) in enumerate(zip(top["peso_svm"], top["clase_dominante"])):
    ax.text(v + xmax * 0.015, i, c, va="center", fontsize=9, color=INK2)
ax.set_yticks(np.arange(len(top)))
ax.set_yticklabels(top["dominio"])
ax.set_ylim(len(top) - 0.4, -0.6)
ax.set_xlim(0, xmax * 1.32)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Peso del dominio en el SVM (norma de sus coeficientes)")
titles(ax, f"Los 20 dominios con más peso en el modelo (de {k_optimal})",
       "Texto al final de la barra = clase en la que el dominio es más abundante")
show_and_save(fig, "2.7", "ranking_dominios")
loadings.head(20)

---
# Parte 3 · Análisis de los resultados del modelo

**Pregunta:** ¿qué tan bien clasifica el modelo BGCs que nunca vio, y en qué se equivoca?

| Sección | Qué hace | Salidas principales |
|---|---|---|
| 3.1 | Abre la prueba apartada y calcula las métricas | `3.1_metricas` |
| 3.2 | Matriz de confusión de la prueba | `3.2_matriz_confusion_prueba` |
| 3.3 | Error por clase | `3.3_error_por_clase` |
| 3.4 | Mapa t-SNE con los dominios del modelo y los errores marcados | `3.4a/b` |
| 3.5 | Círculo de correlación después del modelo (mismo 25 % que en 1.8) | `3.5_circulo_correlacion_despues` |

## 3.1 Métricas en la prueba apartada

Primera y única apertura de la prueba. El modelo, los dominios y `C` ya están fijos: nada de lo que se calcule aquí los modifica.

- **Exactitud** con su intervalo de confianza del 95 % (Wilson); **exactitud balanceada** y **F1 macro**, que pesan igual a todas las clases aunque sean pequeñas; **kappa de Cohen** y **MCC**, que corrigen por el acierto esperado al azar.
- **Brecha de generalización** = exactitud de entrenamiento − exactitud de prueba. Junto con la prueba de permutación (2.5) da el veredicto sobre sobreajuste.
- **BGCs con un "gemelo" en el entrenamiento:** BGCs de prueba cuyo perfil de dominios (presencia/ausencia) es idéntico al de algún BGC de entrenamiento. Se reporta la exactitud en esos BGCs y en los demás, porque los primeros son más fáciles de acertar.

In [ ]:
idx_test = SEALED_TEST["idx"]
X_test, y_test = X[idx_test], y[idx_test]
y_test_pred = final_model.predict(X_test[:, sel_idx])

metrics_test, per_class_test = compute_metrics(y_test, y_test_pred, class_order, "prueba apartada")
n_correct = int((y_test == y_test_pred).sum())
ci_low, ci_high = wilson_interval(n_correct, len(y_test))
metrics_test["ci95_low"], metrics_test["ci95_high"] = round(ci_low, 4), round(ci_high, 4)
gap = float(metrics_train.at[0, "accuracy"] - metrics_test.at[0, "accuracy"])

# ¿Cuántos BGCs de prueba tienen un perfil idéntico en el entrenamiento?
train_profiles = {P[i].tobytes() for i in idx_train}
has_twin = np.array([P[i].tobytes() in train_profiles for i in idx_test])
acc_twin = accuracy_score(y_test[has_twin], y_test_pred[has_twin]) if has_twin.any() else np.nan
acc_new = accuracy_score(y_test[~has_twin], y_test_pred[~has_twin]) if (~has_twin).any() else np.nan

metrics = pd.concat([metrics_train, metrics_test], ignore_index=True)
save_table(metrics, "3.1", "metricas")
save_table(pd.concat([per_class_train, per_class_test], ignore_index=True), "3.1", "metricas_por_clase")

no_overfit = gap <= CONFIG["max_generalization_gap"] and p_perm < 0.05
print(f"Exactitud en prueba: {metrics_test.at[0, 'accuracy']:.3f} ({n_correct}/{len(y_test)} BGCs) · "
      f"IC 95 %: {ci_low:.3f}–{ci_high:.3f}")
print(f"Exactitud balanceada: {metrics_test.at[0, 'balanced_accuracy']:.3f} · F1 macro: {metrics_test.at[0, 'f1_macro']:.3f} · "
      f"kappa: {metrics_test.at[0, 'cohen_kappa']:.3f} · MCC: {metrics_test.at[0, 'mcc']:.3f}")
print(f"Brecha entrenamiento − prueba: {gap:+.3f} (tolerable ≤ {CONFIG['max_generalization_gap']}) · "
      f"permutación p = {p_perm:.4f} → "
      + ("sin evidencia de sobreajuste" if no_overfit else "revisar: posible sobreajuste"))
print(f"BGCs de prueba con un perfil idéntico en el entrenamiento: {has_twin.sum()} de {len(y_test)} "
      f"({100 * has_twin.mean():.0f} %) → exactitud {acc_twin:.3f} · resto (n = {(~has_twin).sum()}) → exactitud {acc_new:.3f}")
metrics[["partition", "n", "accuracy", "balanced_accuracy", "f1_macro", "cohen_kappa", "mcc", "ci95_low", "ci95_high"]]

## 3.2 Matriz de confusión de la prueba

**Cómo leerla:** filas = clase real, columnas = clase predicha. El número es la cantidad de BGCs y el color el porcentaje de la clase real. Todo lo que queda fuera de la diagonal es un error, y su posición dice con qué clase se confundió.

In [ ]:
cm_test = save_table(confusion_figure(y_test, y_test_pred, "3.2", "matriz_confusion_prueba",
                                      "Matriz de confusión — prueba apartada"),
                     "3.2", "matriz_confusion_prueba")

## 3.3 Error por clase

Porcentaje de BGCs de cada clase que el modelo clasificó mal en la prueba. Las clases pequeñas tienen pocos BGCs de prueba: un solo error mueve mucho su porcentaje, por eso se indica también cuántos errores son.

In [ ]:
err = per_class_test.sort_values(["error_rate", "n"], ascending=[False, False]).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8.8, 0.5 * len(err) + 1.8))
ax.barh(np.arange(len(err)), 100 * err["error_rate"], height=0.62, color=SERIES[0])
xmax = max(100 * err["error_rate"].max(), 5)
for i, (e, n, r) in enumerate(zip(err["errors"], err["n"], err["error_rate"])):
    ax.text(100 * r + xmax * 0.02, i, f"{e} de {n}", va="center", fontsize=9.5, color=INK2)
ax.set_yticks(np.arange(len(err)))
ax.set_yticklabels(err["class"])
ax.set_ylim(len(err) - 0.4, -0.6)
ax.set_xlim(0, xmax * 1.25)
ax.grid(axis="y", visible=False)
ax.set_xlabel("BGCs de prueba mal clasificados (%)")
titles(ax, "Error por clase en la prueba apartada",
       "Texto = errores de BGCs de prueba de esa clase")
show_and_save(fig, "3.3", "error_por_clase")

## 3.4 ¿Dónde se equivoca el modelo? t-SNE con los dominios del modelo

t-SNE no puede proyectar BGCs nuevos sobre un mapa ya calculado, así que se calcula un mapa con **todos** los BGCs usando solo los *k* dominios del modelo y después se marcan los de prueba. Es descriptivo: el modelo ya está fijo.

- **3.4a:** el mapa coloreado por clase. Si las clases quedan tan separadas como en 1.5a, los *k* dominios conservan la información que distingue a las clases.
- **3.4b:** solo los BGCs de prueba, con su clase real; los BGCs de entrenamiento quedan en gris como contexto. Los **errores** llevan un círculo negro y la etiqueta `real → predicha`. Si los errores caen donde dos clases se tocan, el modelo se equivoca donde los datos son ambiguos.
- **Tabla de errores:** para cada error, la probabilidad que el modelo dio a su predicción, si su perfil de dominios es idéntico al de algún BGC de entrenamiento y con qué clases se solapa en su genoma (sección 1.3).

In [ ]:
X_sig_scaled = StandardScaler().fit_transform(X[:, sel_idx])
tsne_sig = run_tsne(X_sig_scaled, perp_main)
save_table(pd.DataFrame({"clase": y, "particion": np.where(np.isin(np.arange(len(y)), idx_test), "prueba", "entrenamiento"),
                         "tSNE_1": tsne_sig[:, 0].round(3), "tSNE_2": tsne_sig[:, 1].round(3)}),
           "3.4", "coordenadas_tsne_firma")

# ---- 3.4a Mapa con los k dominios del modelo --------------------------------------------
fig, ax = plt.subplots(figsize=(10, 7.4))
cat_scatter(ax, tsne_sig, y, class_order, CLASS_STYLE)
tsne_axes(ax)
cat_legend(ax, class_order, CLASS_STYLE, counts, title="Clase (n BGCs)")
titles(ax, f"Mapa t-SNE con los {k_optimal} dominios del modelo",
       f"Todos los BGCs (n = {len(y)}) · compárese con 1.5a, que usa los {X.shape[1]} dominios")
show_and_save(fig, "3.4", "a_tsne_dominios_modelo")

# ---- 3.4b Errores de la prueba -----------------------------------------------------------------------
T_test = tsne_sig[idx_test]
wrong = y_test != y_test_pred
fig, ax = plt.subplots(figsize=(10, 7.4))
ax.scatter(*tsne_sig[idx_train].T, s=9, c=CONTEXT, linewidths=0, zorder=1)
cat_scatter(ax, T_test, y_test, class_order, CLASS_STYLE, ellipses=False, direct_labels=False, s=30)
ax.scatter(*T_test[wrong].T, s=230, facecolors="none", edgecolors=INK, linewidths=1.6, zorder=6)
if wrong.any():
    rx, ry = np.ptp(tsne_sig, axis=0)
    label_pos = repel(T_test[wrong] + [0, 0.06 * ry], (0.3 * rx, 0.07 * ry))
    for (x0, y0), (x1, y1), t, p in zip(T_test[wrong], label_pos, y_test[wrong], y_test_pred[wrong]):
        ax.plot([x0, x1], [y0, y1], color=INK2, lw=0.6, zorder=6)
        ax.text(x1, y1, f"{t} → {p}", fontsize=8.5, ha="center", va="bottom", color=INK, path_effects=HALO, zorder=7)
tsne_axes(ax)
handles = cat_handles(class_order, CLASS_STYLE) + [
    Line2D([], [], ls="", marker="o", ms=11, mfc="none", mec=INK, mew=1.6),
    Line2D([], [], ls="", marker="o", ms=6, mfc=CONTEXT, mec=CONTEXT)]
ax.legend(handles, class_order + [f"error ({wrong.sum()})", "BGC de entrenamiento"], loc="center left",
          bbox_to_anchor=(1.01, 0.5), title="BGCs de prueba (clase real)", title_fontsize=10, labelcolor=INK2)
titles(ax, f"Errores del modelo en la prueba apartada: {wrong.sum()} de {len(y_test)} BGCs",
       "Círculo negro = BGC mal clasificado · etiqueta = clase real → clase predicha")
show_and_save(fig, "3.4", "b_errores_prueba")

errors_table = pd.DataFrame({
    "fila_en_datos": idx_test[wrong], ACC: data.loc[idx_test[wrong], ACC].to_numpy(),
    REG: data.loc[idx_test[wrong], REG].to_numpy() if REG in data.columns else np.nan,
    "clase_real": y_test[wrong], "clase_predicha": y_test_pred[wrong],
    "probabilidad_predicha": (final_model.predict_proba(X_test[wrong][:, sel_idx]).max(axis=1).round(3)
                                                      if wrong.any() else []),
    "perfil_identico_en_entrenamiento": has_twin[wrong],
    "se_solapa_con": [", ".join(sorted(overlap_classes.get(i, ()))) or "—" for i in idx_test[wrong]]})
save_table(errors_table, "3.4", "errores_prueba")
n_err_overlap = int(sum(bool(overlap_classes.get(i)) for i in idx_test[wrong]))
print(f"Errores en la prueba: {wrong.sum()} · de ellos, en BGCs que se solapan con otro BGC del mismo genoma (1.3): {n_err_overlap}")
errors_table

## 3.5 Círculo de correlación — después del modelo (mismo 25 %)

La misma submuestra estratificada del 25 % que en 1.8, ahora usando solo los *k* dominios del modelo. Al compararlo con 1.8 se ve si los dominios que organizan el mapa forman parte de la firma del modelo.

In [ ]:
corr_after = save_table(
    correlation_circle(X[corr_idx][:, sel_idx], y[corr_idx], selected_features, "3.5",
                       "circulo_correlacion_despues",
                       f"Círculo de correlación después del modelo: los {k_optimal} dominios del modelo"),
    "3.5", "circulo_correlacion_despues")
shared = set(corr_before["dominio"]) & set(selected_features)
print(f"Dominios del círculo 'antes' (1.8) que forman parte del modelo: {len(shared)} de {len(corr_before)}")
corr_after

---
# Parte 4 · Exploración de los resultados: grupos y arquitectura

**Pregunta:** al agrupar los BGCs aparecen *k* grupos (3 por defecto). ¿Qué dominios Pfam son responsables de esos grupos y cómo cambia la arquitectura de los BGCs de un grupo a otro?

**Referencia metodológica.** Kralova *et al.* (2026) describen un cluster de sideróforos en dos niveles: la arquitectura del cluster (genes coloreados por categoría funcional, su Fig. 4A) y los dominios y módulos de los genes biosintéticos (su Fig. 4B). Esta parte aplica la misma lectura a los grupos de este conjunto de datos.

| Sección | Qué hace | Salidas principales |
|---|---|---|
| 4.1 | Forma los grupos y muestra qué clases contiene cada uno | `4.1a/b/c` |
| 4.2 | Busca los dominios responsables de cada grupo (enriquecimiento y reglas de un árbol de decisión) | `4.2a/b/c` |
| 4.3 | Dibuja la arquitectura gen a gen del BGC representativo de cada grupo | `4.3_arquitectura_grupos` |
| 4.4 | Compara dominios de ensamblaje, tamaño y arquitecturas proteicas entre grupos | `4.4a/b/c` |
| 4.5 | Análisis específico de los BGCs de sideróforos | `4.5a/b/c` |
| 4.6 | Compara los BGCs por similitud de secuencia proteica (alineamientos) | `4.6a/b/c` |

## 4.1 Formación de los grupos

Los grupos se forman con **agrupamiento jerárquico de Ward** sobre las coordenadas del mapa t-SNE de la sección 1.5a (`CONFIG["group_on"] = "signature"` usa el mapa con los dominios del modelo, 3.4a). Se nombran G1, G2, … de mayor a menor tamaño.

- **4.1a · Mapa de los grupos:** cada punto es un BGC con el color de su clase; la línea discontinua rodea cada grupo.
- **4.1b · Composición:** cuántos BGCs de cada clase caen en cada grupo.
- **4.1c · ¿Cuántos grupos hay?** La silueta mide qué tan compactos y separados quedan los grupos para k = 2…8. Si sigue subiendo después del k usado, el mapa tiene más grupos naturales y los *k* grupos son un resumen grueso.

In [ ]:
from scipy.spatial import ConvexHull

if CONFIG["group_on"] == "signature":
    group_space, space_label = tsne_sig, f"mapa t-SNE con los {k_optimal} dominios del modelo (3.4a)"
else:
    group_space, space_label = tsne_all, "mapa t-SNE con todos los dominios (1.5a)"

# ---- Silueta para k = 2…8 (más alta = grupos más compactos y separados) ----
sil = pd.DataFrame([{"k": k, "silhouette": silhouette_score(
    group_space, AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(group_space))} for k in range(2, 9)])
save_table(sil.round(4), "4.1", "silueta")

# ---- Ward con k = n_groups; G1 = el grupo más grande --------------------------------
k_groups = CONFIG["n_groups"]
ward = AgglomerativeClustering(n_clusters=k_groups, linkage="ward").fit_predict(group_space)
by_size = pd.Series(ward).value_counts().index.tolist()
groups = np.array([f"G{by_size.index(g) + 1}" for g in ward])
group_names = [f"G{i + 1}" for i in range(k_groups)]
group_sizes = pd.Series(groups).value_counts().reindex(group_names)

composition = (pd.crosstab(pd.Series(groups, name="grupo"), pd.Series(y, name="clase"))
               .reindex(index=group_names, columns=class_order, fill_value=0))
save_table(composition.reset_index(), "4.1", "composicion_grupos")
save_table(pd.DataFrame({"fila_en_datos": np.arange(len(y)), ACC: data[ACC].to_numpy(),
                         REG: data[REG].to_numpy() if REG in data.columns else np.nan,
                         "clase": y, "grupo": groups}), "4.1", "grupo_de_cada_bgc")


def group_outlines(ax, E, labels, names, sizes=None, fontsize=12):
    """Contorno discontinuo (envolvente convexa con margen) y nombre de cada grupo."""
    span = float(np.ptp(E, axis=0).max())
    tops = []
    for g in names:
        pts = E[labels == g]
        if len(pts) < 3:
            continue
        hull = pts[ConvexHull(pts).vertices]
        d = hull - pts.mean(axis=0)
        hull = hull + d / np.linalg.norm(d, axis=1, keepdims=True) * 0.03 * span      # margen alrededor de los puntos
        ax.add_patch(Polygon(hull, closed=True, fill=False, ls=(0, (5, 3)), lw=1.4, edgecolor=INK2, zorder=4))
        top = hull[hull[:, 1].argmax()]
        tops.append(top[1])
        text = g if sizes is None else f"{g} · {sizes[g]} BGCs"
        ax.text(top[0], top[1] + 0.012 * span, text, ha="center", va="bottom", fontsize=fontsize, fontweight="bold",
                color=INK, path_effects=HALO, zorder=6)
    lo, hi = ax.get_ylim()
    ax.set_ylim(lo, max(hi, max(tops) + 0.09 * span))


# ---- 4.1a Mapa de los grupos -------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 7.4))
cat_scatter(ax, group_space, y, class_order, CLASS_STYLE, ellipses=False, direct_labels=False, s=20, alpha=0.8)
group_outlines(ax, group_space, groups, group_names, group_sizes)
tsne_axes(ax)
cat_legend(ax, class_order, CLASS_STYLE, counts, title="Clase (n BGCs)")
titles(ax, f"{k_groups} grupos de BGCs sobre el mapa t-SNE",
       f"Ward sobre el {space_label}\nLínea discontinua = contorno de cada grupo · color = clase del BGC")
show_and_save(fig, "4.1", "a_grupos_tsne")

# ---- 4.1b Composición de los grupos ------------------------------------------------------------
share = composition / composition.sum(axis=0).replace(0, 1)
fig, ax = plt.subplots(figsize=(0.95 * len(class_order) + 3, 0.7 * k_groups + 2.4))
heatmap(ax, share, [f"{g}  (n = {group_sizes[g]})" for g in group_names], class_order, vmin=0, vmax=1,
        annot=np.where(composition > 0, composition.astype(str), ""), cbar_label="Fracción de la clase",
        annot_size=10)
rotate_xlabels(ax)
ax.tick_params(axis="y", rotation=0)
titles(ax, "Qué clases forman cada grupo",
       "Número = BGCs · color = fracción de los BGCs de la clase que cae en el grupo · vacío = 0")
show_and_save(fig, "4.1", "b_composicion_grupos")

# ---- 4.1c Silueta -------------------------------------------------------------------------------------
best_k = int(sil.loc[sil["silhouette"].idxmax(), "k"])
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.plot(sil["k"], sil["silhouette"], color=SERIES[0], marker="o", ms=5)
ax.axvline(k_groups, color=INK2, lw=1)
ax.annotate(f"k usado = {k_groups}", xy=(k_groups, sil.set_index("k").at[k_groups, "silhouette"]), xytext=(8, 10),
            textcoords="offset points", fontsize=10.5, fontweight="bold", color=INK, path_effects=HALO)
ax.set_xlabel("Número de grupos (k)")
ax.set_ylabel("Silueta")
titles(ax, "Calidad del agrupamiento según el número de grupos",
       "Silueta: más alta = grupos más compactos y mejor separados"
       + (f"\nSigue subiendo después de k = {k_groups}: el mapa tiene más grupos naturales; los {k_groups} "
          "grupos son un resumen grueso" if best_k > k_groups else ""))
show_and_save(fig, "4.1", "c_silueta")

for g in group_names:
    comp = composition.loc[g]
    print(f"{g} ({group_sizes[g]} BGCs): " + ", ".join(f"{c} {n}" for c, n in comp[comp > 0].sort_values(ascending=False).items()))
print(f"Silueta con k = {k_groups}: {sil.set_index('k').at[k_groups, 'silhouette']:.3f} · k con la silueta más alta: {best_k}"
      + (f" → el mapa tiene más grupos naturales que {k_groups}; los grupos son un resumen grueso del mapa"
         if best_k > k_groups else ""))

## 4.2 Dominios responsables de los grupos

Para cada dominio Pfam y cada grupo (el grupo contra el resto de los BGCs):

| Columna | Qué mide |
|---|---|
| `prevalencia_grupo` / `prevalencia_resto` | Fracción de BGCs con el dominio dentro y fuera del grupo |
| `log2_OR` | log2 de la razón de momios (corrección de Haldane). Positivo = más frecuente en el grupo (↑); negativo = menos frecuente (↓), es decir, el dominio *falta* en el grupo |
| `p_fisher` / `q_fdr` | Prueba exacta de Fisher y su corrección por comparaciones múltiples (Benjamini–Hochberg) |
| `AUC` / `discriminacion` | Capacidad del dominio, por sí solo, de separar el grupo del resto: discriminación = \|2·AUC − 1\|, de 0 (no separa) a 1 (separa perfectamente) |

Los **dominios marcadores** de cada grupo son los de mayor discriminación con `q_fdr < 0.05`. Además, un **árbol de decisión** poco profundo, entrenado solo con la presencia o ausencia de dominios comunes, da la combinación mínima de dominios que reproduce los grupos.

Cada dominio recibe también una **categoría funcional** según su familia Pfam (listas editables al inicio de la celda; las usa la sección 4.3):

- **Biosíntesis central** depende de la clase del BGC: son los dominios Pfam equivalentes a los perfiles de la regla de detección de antiSMASH 7 para esa clase (`CORE_RULES`; por ejemplo betalactone = `AMP-binding` + `HMGL-like`, ni-siderophore = `IucA_IucC`, nrps = `Condensation` + `AMP-binding`, a los que se suman T y TE de la línea NRPS). Así, `HMGL-like` es biosíntesis central en un BGC de betalactone pero no en uno de ectoine. Además, cualquier proteína con dos o más dominios de ensamblaje NRPS/PKS distintos (p. ej. C-A-T) es biosíntesis central.
- **Precursores, modificación, transporte y regulación** se asignan por familia Pfam (`DOMAIN_CATEGORIES`); el resto es "otro dominio".

La celda comprueba que cada entrada de las listas corresponde a dominios reales del CSV. En la tabla de enriquecimiento, la columna `regla_antismash_de` indica qué clases usan el dominio en su regla.

- **4.2a:** porcentaje de BGCs de cada grupo que tienen cada dominio marcador.
- **4.2b:** el árbol de decisión dibujado como reglas sí/no.
- **4.2c:** dónde están en el mapa los BGCs que tienen los dos mejores marcadores de cada grupo.

In [ ]:
from functools import lru_cache

# ---- Categorías funcionales (editable) -----------------------------------------------------------
CAT_CORE, CAT_PRECURSOR = "biosíntesis central", "precursores"
CAT_TAILORING, CAT_TRANSPORT = "modificación", "transporte y captación de hierro"
CAT_REGULATION, CAT_OTHER, CAT_NONE = "regulación y señalización", "otro dominio", "sin dominio Pfam"

# (1) Dominios de las líneas de ensamblaje NRPS/PKS. Un gen con dos o más de ellos (p. ej. C-A-T) es "biosíntesis central" en cualquier BGC.
ASSEMBLY_LINE = ["Condensation", "AMP-binding", "PP-binding", "Thioesterase", "ketoacyl-synt", "Ketoacyl-synt_C",
                 "KAsynt_C_assoc", "Acyl_transf_1", "PS-DH"]
# (2) Familias Pfam equivalentes a los perfiles de las reglas de detección de antiSMASH 7 de cada clase:
#     "biosíntesis central" solo en los BGCs de esa clase (p. ej. HMGL-like lo es en betalactone, no en ectoine).
CORE_RULES = {"betalactone": ["AMP-binding", "HMGL-like"], "redox-cofactor": ["Radical_SAM", "SPASM", "PqqD"],
              "naggn": ["Asn_synthase", "GATase_7", "Peptidase_M42"], "ectoine": ["Ectoine_synth"],
              "terpene": ["SQS_PSY", "Lycopene_cycl"], "arylpolyene": ["ketoacyl-synt", "Ketoacyl-synt_C"],
              "ni-siderophore": ["IucA_IucC"],
              "nrps": ["Condensation", "AMP-binding", "PP-binding", "Thioesterase"],          # + T y TE de la línea NRPS
              "nrp-metallophore": ["Condensation", "AMP-binding", "PP-binding", "Thioesterase"]}
# (3) Resto de categorías por familia Pfam: nombre exacto o inicio del nombre (sin distinguir mayúsculas).
DOMAIN_CATEGORIES = {
    CAT_PRECURSOR: ["Chorismate_bind", "Isochorismatase", "Lys_Orn_oxgnase", "polyprenyl_synt"],
    CAT_TAILORING: ["Methyltransf_", "adh_short", "KR", "Epimerase", "Glycos_transf_", "Glyco_tran", "UDPGT",
                    "Acetyltransf_", "Aldedh", "Aldo_ket_red", "FAD_binding_", "Trp_halogenase", "Abhydrolase_",
                    "NAD_binding_", "Oxidored_", "Aminotran_", "TauD", "PhyH"],
    CAT_TRANSPORT: ["ABC_tran", "ABC_membrane", "ABC2_membrane", "BPD_transp_", "MFS_", "Peripla_BP_", "SBP_bac_",
                    "FecCD", "TonB_", "Plug", "OMP_b-brl", "DctP", "DctM", "DctQ", "ACR_tran", "OEP", "EamA",
                    "Sugar_tr", "Na_H_Exchanger", "FhuF"],
    CAT_REGULATION: ["HTH_", "Response_reg", "HisKA", "HATPase_c", "LysR_substrate", "TetR_", "GntR", "MarR", "MerR",
                     "AraC_", "Sigma70_", "Sigma54_", "Trans_reg_C", "AsnC_trans_reg", "GerE", "GGDEF", "EAL",
                     "PAS", "HAMP"],
}
CATEGORY_COLORS = {CAT_CORE: "#e34948", CAT_PRECURSOR: "#eda100", CAT_TAILORING: "#4a3aa7", CAT_TRANSPORT: "#2a78d6",
                   CAT_REGULATION: "#1baf7a", CAT_OTHER: "#a9a8a0", CAT_NONE: "#dcdbd4"}
RULE_CLASSES = {}                                                # dominio → clases cuya regla lo usa
for _cls, _doms in CORE_RULES.items():
    for _d in _doms:
        RULE_CLASSES.setdefault(_d, []).append(_cls)


@lru_cache(maxsize=None)
def domain_category(name, cls=None):
    """Categoría de un dominio. Con `cls`, "biosíntesis central" sigue la regla de esa clase; sin `cls`,
    lo es cualquier dominio de alguna regla."""
    if (name in CORE_RULES.get(cls, ())) if cls is not None else (name in ASSEMBLY_LINE or name in RULE_CLASSES):
        return CAT_CORE
    n = name.lower()
    for cat, entries in DOMAIN_CATEGORIES.items():
        if any(n.startswith(e.lower()) for e in entries):
            return cat
    return CAT_OTHER


all_pfam = [c.split(":", 1)[-1] for c in pfam_columns]          # todos los dominios del CSV
entries = ASSEMBLY_LINE + [d for v in CORE_RULES.values() for d in v] + [e for v in DOMAIN_CATEGORIES.values() for e in v]
unmatched = sorted({e for e in entries if not any(n.lower().startswith(e.lower()) for n in all_pfam)})
cat_count = pd.Series([domain_category(n) for n in domain_names]).value_counts()
print("Dominios por categoría: " + " · ".join(f"{c} {cat_count.get(c, 0)}" for c in CATEGORY_COLORS if c != CAT_NONE))
print("Entradas de las listas sin dominios en el CSV:", ", ".join(unmatched) if unmatched else "ninguna")


def bh_fdr(p):
    """Corrección de Benjamini–Hochberg (tasa de falsos descubrimientos)."""
    p = np.asarray(p, float)
    order = np.argsort(p)
    q = np.empty(len(p))
    q[order] = np.minimum.accumulate((p[order] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    return np.minimum(q, 1)


# ---- Enriquecimiento: cada grupo contra el resto --------------------------------
rows = []
for g in group_names:
    m = groups == g
    for j, name in enumerate(domain_names):
        a, c = int(P[m, j].sum()), int(P[~m, j].sum())            # BGCs con el dominio dentro / fuera
        if a + c == 0:
            continue
        b, d = int(m.sum()) - a, int((~m).sum()) - c
        rows.append({"grupo": g, "dominio": name, "categoria": domain_category(name),
                     "regla_antismash_de": ", ".join(RULE_CLASSES.get(name, [])),
                     "prevalencia_grupo": a / m.sum(), "prevalencia_resto": c / (~m).sum(),
                     "log2_OR": np.log2(((a + .5) * (d + .5)) / ((b + .5) * (c + .5))),     # corrección de Haldane
                     "p_fisher": fisher_exact([[a, b], [c, d]])[1], "AUC": roc_auc_score(m, X[:, j])})
enrich = pd.DataFrame(rows)
enrich["q_fdr"] = enrich.groupby("grupo")["p_fisher"].transform(bh_fdr)
enrich["direccion"] = np.where(enrich["log2_OR"] > 0, "↑", "↓")
enrich["discriminacion"] = (2 * enrich["AUC"] - 1).abs()
save_table(enrich.round(4), "4.2", "enriquecimiento_dominios")

markers = (enrich[enrich["q_fdr"] < 0.05]
           .sort_values(["grupo", "discriminacion"], ascending=[True, False])
           .groupby("grupo").head(CONFIG["marker_domains_per_group"]).reset_index(drop=True))
save_table(markers.round(4), "4.2", "dominios_marcadores")
markers_by_group = {g: set(markers.loc[markers["grupo"] == g, "dominio"]) for g in group_names}

# ---- Árbol de decisión sobre la presencia de dominios comunes --------------------
common = np.where(P.mean(axis=0) >= CONFIG["tree_min_prevalence"])[0]       # evita reglas con dominios raros
tree_names = [domain_names[j] for j in common]
tree = DecisionTreeClassifier(max_depth=CONFIG["tree_max_depth"], min_samples_leaf=5, random_state=RS)
tree_cv = cross_val_score(tree, P[:, common], groups, scoring="balanced_accuracy",
                          cv=StratifiedKFold(5, shuffle=True, random_state=RS))
tree.fit(P[:, common], groups)

for g in group_names:
    top3 = markers[markers["grupo"] == g].head(3)
    print(f"{g}: " + "; ".join(f"{r['dominio']} {r['direccion']} ({100 * r['prevalencia_grupo']:.0f} % en el grupo vs "
                               f"{100 * r['prevalencia_resto']:.0f} % en el resto)" for _, r in top3.iterrows()))
print(f"Árbol de decisión (profundidad ≤ {CONFIG['tree_max_depth']}, {len(common)} dominios comunes): exactitud balanceada por validación cruzada = "
      f"{tree_cv.mean():.3f} ± {tree_cv.std():.3f}")
markers[["grupo", "dominio", "direccion", "prevalencia_grupo", "prevalencia_resto",
         "discriminacion", "q_fdr", "categoria"]].round(3)

**Figuras 4.2a – 4.2c.** La celda anterior calcula el enriquecimiento, los dominios marcadores y el árbol de decisión; esta dibuja las tres figuras descritas arriba.

In [ ]:
# ---- 4.2a Prevalencia de los dominios marcadores en cada grupo ----------------
rows_lab, block_of, M = [], [], []
for g in group_names:
    for _, r in markers[markers["grupo"] == g].iterrows():
        j = domain_names.index(r["dominio"])
        rows_lab.append(f"{r['dominio']} {r['direccion']}")
        block_of.append(g)
        M.append([100 * P[groups == h, j].mean() for h in group_names])
M = np.array(M)
fig, ax = plt.subplots(figsize=(1.25 * k_groups + 4.6, 0.36 * len(rows_lab) + 2.2))
heatmap(ax, M, rows_lab, [f"{g}\n(n = {group_sizes[g]})" for g in group_names], vmin=0, vmax=100,
        annot=np.vectorize(lambda v: f"{v:.0f}")(M), cbar_label="% de BGCs del grupo con el dominio")
ax.tick_params(axis="y", rotation=0)
ax.tick_params(axis="x", rotation=0)
for i in range(1, len(block_of)):                                # separa los bloques de cada grupo
    if block_of[i] != block_of[i - 1]:
        ax.axhline(i, color=INK, lw=1.4)
for g in group_names:                                            # etiqueta de bloque a la izquierda
    idx = [i for i, b in enumerate(block_of) if b == g]
    if idx:
        ax.text(-0.02, np.mean(idx) + 0.5, f"marcadores de {g}", transform=ax.get_yaxis_transform(), ha="right", va="center",
                fontsize=10, fontweight="bold", color=INK, rotation=90)
ax.set_yticklabels(rows_lab)
ax.tick_params(axis="y", pad=30)
titles(ax, "Dominios marcadores de cada grupo",
       "Número = % de BGCs del grupo con el dominio · ↑ más frecuente en el grupo · ↓ menos frecuente")
show_and_save(fig, "4.2", "a_dominios_marcadores")


# ---- 4.2b Árbol de decisión dibujado como reglas sí / no ------------------------------------
def draw_tree(ax, tree, names, X_tree, labels):
    t = tree.tree_
    leaf_of = tree.apply(X_tree)
    pos, leaves = {}, []

    def walk(node, depth):                                       # hojas de izquierda a derecha
        if t.children_left[node] == -1:
            pos[node] = (len(leaves), -depth)
            leaves.append(node)
            return
        walk(t.children_left[node], depth + 1)
        walk(t.children_right[node], depth + 1)
        pos[node] = ((pos[t.children_left[node]][0] + pos[t.children_right[node]][0]) / 2, -depth)

    walk(0, 0)
    for node, (x, yv) in pos.items():
        left, right = t.children_left[node], t.children_right[node]
        if left == -1:                                           # hoja: grupo mayoritario y pureza
            members = pd.Series(labels[leaf_of == node]).value_counts()
            ax.text(x, yv, f"{members.index[0]}\n{members.iloc[0]} de {members.sum()} BGCs", ha="center", va="center",
                    fontsize=10.5, fontweight="bold", color=INK, linespacing=1.4,
                    bbox=dict(boxstyle="round,pad=0.55", fc=SURFACE, ec=INK, lw=1.3), zorder=3)
            continue
        for child, word in ((left, "no"), (right, "sí")):        # izquierda = ausente, derecha = presente
            cx, cy = pos[child]
            ax.plot([x, cx], [yv, cy], color=AXIS, lw=1.3, zorder=1)
            ax.text((x + cx) / 2, (yv + cy) / 2, word, ha="center", va="center", fontsize=9.5, color=INK2,
                    bbox=dict(boxstyle="round,pad=0.2", fc=SURFACE, ec="none"), zorder=2)
        ax.text(x, yv, f"¿Tiene {names[t.feature[node]]}?", ha="center", va="center", fontsize=10, color=INK,
                bbox=dict(boxstyle="round,pad=0.5", fc="#f0efec", ec=AXIS, lw=1), zorder=3)
    ax.set_xlim(-0.7, len(leaves) - 0.3)
    ax.set_ylim(-tree.get_depth() - 0.5, 0.5)
    ax.axis("off")


n_leaves = tree.get_n_leaves()
fig, ax = plt.subplots(figsize=(max(9, 2.4 * n_leaves), 1.7 * (tree.get_depth() + 1) + 0.8))
draw_tree(ax, tree, tree_names, P[:, common], groups)
titles(ax, "Reglas de presencia de dominios que reproducen los grupos",
       f"Árbol de decisión de profundidad ≤ {CONFIG['tree_max_depth']} · exactitud balanceada por validación cruzada = {tree_cv.mean():.2f}\n"
       "Cada hoja indica el grupo mayoritario y cuántos de sus BGCs pertenecen a él")
show_and_save(fig, "4.2", "b_arbol_decision")


# ---- 4.2c Dónde están los BGCs con cada marcador --------------------------------------
def presence_panel(ax, name, g):
    j = domain_names.index(name)
    ax.scatter(*group_space[~P[:, j]].T, s=7, c=CONTEXT, linewidths=0, zorder=1)
    ax.scatter(*group_space[P[:, j]].T, s=14, c=SERIES[0], linewidths=0, zorder=2)
    group_outlines(ax, group_space, groups, group_names, fontsize=10)
    inside, rest = 100 * P[groups == g, j].mean(), 100 * P[groups != g, j].mean()
    ax.set_title(f"{name}\n{g}: {inside:.0f} %  ·  resto: {rest:.0f} %", loc="left", fontsize=10.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)


top_markers, shown = [], set()                                  # 2 por grupo: primero los ↑, sin repetir dominios
for g in group_names:
    cand = markers[markers["grupo"] == g].sort_values(["direccion", "discriminacion"], ascending=[True, False],
                                                          key=lambda c: c.map({"↑": 0, "↓": 1}) if c.name == "direccion" else c)
    picks = [n for n in cand["dominio"] if n not in shown][:2]
    shown.update(picks)
    top_markers += [(g, n) for n in picks]
ncol = 3
nrow = int(np.ceil(len(top_markers) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.0 * ncol, 4.6 * nrow), squeeze=False)
for ax, (g, name) in zip(axes.ravel(), top_markers):
    presence_panel(ax, name, g)
for ax in axes.ravel()[len(top_markers):]:
    ax.axis("off")
fig.legend([Line2D([], [], ls="", marker="o", ms=7, mfc=SERIES[0], mec=SERIES[0]), Line2D([], [], ls="", marker="o", ms=6, mfc=CONTEXT, mec=CONTEXT)],
           ["BGC con el dominio", "BGC sin el dominio"], loc="upper center",
           bbox_to_anchor=(0.5, -0.0), ncol=2, labelcolor=INK2)
fig.suptitle("Dónde están los BGCs con los dos mejores marcadores de cada grupo",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
show_and_save(fig, "4.2", "c_marcadores_en_mapa")
for g, name in top_markers:                                      # paneles individuales
    fig, ax = plt.subplots(figsize=(6.5, 5.6))
    presence_panel(ax, name, g)
    show_and_save(fig, "4.2", f"c_marcador_{g}_{slug(name)}", individual=True)

## 4.3 Arquitectura del BGC representativo de cada grupo

La columna `protein_seq_pfam_map` contiene las proteínas de cada BGC (separadas por `|`) con cada dominio sustituido por `[Dominio(valor)]`; al compararla con `protein_sequences` se recupera qué dominios tiene cada gen y en qué orden. Con eso se dibuja, para el **representante** de cada grupo (su *medoide*: el BGC más parecido en promedio al resto del grupo, por distancia de Jaccard entre perfiles de dominios), una figura al estilo de la Fig. 4 de Kralova *et al.* (2026), sin estructuras químicas:

- **A · Arquitectura del cluster.** Cada gen es una flecha con longitud proporcional a su proteína (aa × 3 pb) y color según la categoría funcional de sus dominios (4.2; si tiene varios, manda la de mayor prioridad). Los tres grupos comparten la escala. Se numeran los genes del panel B y los genes con un dominio marcador del grupo (4.2); estos últimos llevan **borde negro** y número en negrita: así se ve en qué parte del cluster están los dominios responsables del grupo.
- **B · Dominios de los genes biosintéticos centrales**, en orden N→C. Abreviaturas de las líneas de ensamblaje: C = condensación (`Condensation`), A = adenilación (`AMP-binding`), T = tiolación (`PP-binding`), TE = tioesterasa, KS = cetosintasa (`ketoacyl-synt`), AT = aciltransferasa (`Acyl_transf_1`), DH (`PS-DH`), KR, MT = metiltransferasa; los demás dominios conservan su nombre Pfam. Los **módulos** (M1, M2…) se detectan automáticamente: empiezan en C o KS (o en A si no hay C antes) y solo cuentan si tienen A + T (NRPS) o KS + T (PKS). Los dominios con **borde negro** son marcadores del grupo.
- **Sobre `C/E`:** Pfam no distingue el dominio de epimerización (E) del de condensación: ambos pertenecen a la familia `Condensation`. Cuando un dominio C sigue a un T y cierra el gen o va seguido de otro C (la posición típica de un E), se rotula `C/E`.

La tabla `4.3_genes_representantes` lista todos los genes de cada representante (número, longitud, categoría y dominios) para leer la figura gen por gen.

**El mapa no contiene todos los dominios de la tabla.** La celda mide qué parte de las presencias de dominio de la tabla aparece también en `protein_seq_pfam_map` y guarda los dominios que faltan (`4.3_dominios_ausentes_del_mapa`). Todo lo que depende del orden de los genes (esta sección, los módulos de 4.4 y 4.5, y la Parte 5) solo ve los dominios que están en el mapa; la presencia o ausencia de un dominio en un BGC se toma siempre de las columnas de la tabla.

In [ ]:
from scipy.spatial.distance import pdist, squareform

TAG_RE = re.compile(r"\[([^\[\]()]+)\(([0-9.]+)\)\]")              # etiqueta [Dominio(0.94)]
AA_OK = set("ACDEFGHIKLMNPQRSTVWYBZX")
# Abreviaturas de los dominios de ensamblaje (nombre Pfam exacto)
DOMAIN_ABBREV = {"Condensation": "C", "AMP-binding": "A", "PP-binding": "T", "Thioesterase": "TE", "ketoacyl-synt": "KS",
                 "Acyl_transf_1": "AT", "PS-DH": "DH", "KR": "KR"}
# Partes C-terminales: se unen a la caja anterior del mismo dominio; si el mapa no trae la parte N-terminal, forman su propia caja
C_TERMINAL = {"AMP-binding_C": "A", "Ketoacyl-synt_C": "KS", "KAsynt_C_assoc": "KS"}
SHORT_LABELS = {"C", "A", "T", "TE", "KS", "AT", "DH", "KR", "MT", "C/E"}
MODULE_START, MODULE_BODY = {"C", "KS"}, {"A", "T", "C/E", "AT", "DH", "KR", "MT"}


def parse_architecture(map_cell, seq_cell):
    """Genes y dominios (con posición en aa) a partir de "...SEQ[Dominio(0.94)]SEQ...|..."."""
    if pd.isna(map_cell):
        return []
    maps = str(map_cell).split("|")
    seqs = str(seq_cell).split("|") if not pd.isna(seq_cell) else []
    seqs = seqs + [None] * (len(maps) - len(seqs))
    genes = []
    for mp, seq in zip(maps, seqs):
        parts = TAG_RE.split(mp)                                 # [texto, nombre, valor, texto, …]
        pos, pending, doms = 0, [], []
        for k in range(0, len(parts), 3):
            chunk = parts[k]
            if chunk:
                if pending:                                      # el dominio termina donde reaparece la secuencia
                    end = seq.find(chunk, pos) if seq else -1
                    end = end if end > pos else pos + 100
                    doms += [(n, v, pos, end) for n, v in pending]
                    pending, pos = [], end
                pos += len(chunk)
            if k + 1 < len(parts):
                pending.append((parts[k + 1], float(parts[k + 2])))
        length = len(seq) if seq else pos
        doms += [(n, v, pos, max(length, pos + 1)) for n, v in pending]
        genes.append({"length_aa": max(length, 1), "domains": doms})
    return genes


def abbrev(name):
    if name in DOMAIN_ABBREV:
        return DOMAIN_ABBREV[name]
    if name.startswith("Methyltransf"):
        return "MT"
    return name if len(name) <= 13 else name[:12] + "…"


def gene_category(gene, cls):
    """Categoría del gen = la de mayor prioridad entre sus dominios (orden de CATEGORY_COLORS). Una proteína con dos o más
    dominios de ensamblaje distintos (p. ej. C-A-T, KS-AT) es siempre biosíntesis central."""
    names = {d[0] for d in gene["domains"]}
    if len({DOMAIN_ABBREV.get(n) or C_TERMINAL.get(n) for n in names} & {"C", "A", "T", "TE", "KS", "AT", "DH"}) >= 2:
        return CAT_CORE
    cats = {domain_category(n, cls) for n in names}
    return next((c for c in CATEGORY_COLORS if c in cats), CAT_NONE)


def gene_boxes(gene, marker_set=frozenset()):
    """Dominios del gen en orden N→C: (abreviatura, nombre Pfam, ¿marcador?)."""
    out = []
    for n, *_ in sorted(gene["domains"], key=lambda d: d[2]):
        if n in C_TERMINAL:                                      # parte C-terminal de A o KS
            if not (out and out[-1][0] == C_TERMINAL[n]):
                out.append([C_TERMINAL[n], n, n in marker_set])
            continue
        ab = abbrev(n)
        if not (out and out[-1][1] == n and ab not in ("C", "A", "T")):
            out.append([ab, n, n in marker_set])                 # une repeticiones solapadas (salvo C, A, T)
    for i in range(1, len(out)):                                 # C en posición de E → "C/E"
        nxt = out[i + 1][0] if i + 1 < len(out) else None
        if out[i][0] == "C" and out[i - 1][0] == "T" and nxt in (None, "C"):
            out[i][0] = "C/E"
    return [tuple(b) for b in out]


def assign_modules(core_boxes):
    """Módulos NRPS/PKS por gen: (inicio, fin, "M1"), numerados a lo largo del BGC."""
    counter, result = 0, []
    for boxes in core_boxes:
        mods, cur = [], None
        for i, (ab, *_rest) in enumerate(boxes):
            if ab in MODULE_START or (ab == "A" and (cur is None or "A" in cur["types"])):
                cur = {"start": i, "end": i, "types": {ab}}
                mods.append(cur)
            elif ab in MODULE_BODY and cur is not None:
                cur["end"], cur["types"] = i, cur["types"] | {ab}
            else:
                cur = None                                       # TE y dominios sueltos quedan fuera de los módulos
        labelled = []
        for m in mods:
            if {"A", "T"} <= m["types"] or {"KS", "T"} <= m["types"]:     # NRPS: A + T · PKS: KS + T
                counter += 1
                labelled.append((m["start"], m["end"], f"M{counter}"))
        result.append(labelled)
    return result


def core_genes(genes, cls, marker_set=frozenset()):
    """Genes de biosíntesis central con sus cajas de dominio: [(número de gen, cajas)]."""
    out = [(k, gene_boxes(g, marker_set)) for k, g in enumerate(genes, start=1) if gene_category(g, cls) == CAT_CORE]
    return [(k, b) for k, b in out if b]


def n_modules(genes, cls):
    return sum(len(m) for m in assign_modules([b for _, b in core_genes(genes, cls)]))


# ---- Dibujo: panel A (cluster) y panel B (dominios de los genes centrales) --------
GAP_KB, BOX_FILL, ROW_H = 0.12, "#f6dcdc", 2.35


def draw_panel_a(ax, genes, kb_max, cls, numbered, marker_set):
    total = sum(g["length_aa"] * 3e-3 for g in genes) + GAP_KB * (len(genes) - 1)
    ax.plot([0, total], [0, 0], color=MUTED, lw=1, zorder=1)        # esqueleto del cluster
    x, last_label, H = 0.0, -1e9, 0.42
    for k, gene in enumerate(genes, start=1):
        L = gene["length_aa"] * 3e-3                                 # kb = aa × 3 pb
        head = min(0.45, 0.45 * L)
        marker = any(d[0] in marker_set for d in gene["domains"])
        ax.add_patch(Polygon([(x, -H / 2), (x + L - head, -H / 2), (x + L - head, -H * 0.85), (x + L, 0),
                              (x + L - head, H * 0.85), (x + L - head, H / 2), (x, H / 2)], closed=True,
                             facecolor=CATEGORY_COLORS[gene_category(gene, cls)], edgecolor=INK if marker else SURFACE,
                             lw=1.8 if marker else 0.8, zorder=3 if marker else 2))
        if k in numbered:
            lift = 0.28 if x + L / 2 - last_label < 0.6 else 0.0     # escalona números muy juntos
            ax.text(x + L / 2, H * 0.85 + 0.08 + lift, str(k), ha="center", va="bottom", fontsize=9, color=INK,
                    fontweight="bold" if marker else "normal")
            last_label = x + L / 2 if lift == 0 else last_label
        x += L + GAP_KB
    ax.plot([kb_max - 2.5, kb_max], [-0.72, -0.72], color=INK, lw=1.2)   # barra de escala (abajo a la derecha)
    ax.text(kb_max - 2.62, -0.72, "2.5 kb", ha="right", va="center", fontsize=8.5, color=INK2)
    ax.text(-0.35, 0, "A", fontsize=13, fontweight="bold", color=INK, ha="right", va="center")
    ax.set_xlim(-0.75, kb_max + 0.3)
    ax.set_ylim(-0.95, 1.05)
    ax.axis("off")


def layout_panel_b(core, max_units=34):
    """Coloca las cajas de dominio de los genes centrales en filas (cada gen completo en una fila)."""
    items, row, x = [], 0, 0.0
    for (k, boxes), mods in zip(core, assign_modules([b for _, b in core])):
        widths = [max(1.0, (0.24 if ab in SHORT_LABELS else 0.19) * len(ab) + 0.5) for ab, *_ in boxes]
        w_boxes = sum(widths) + 0.14 * (len(widths) - 1)
        w_gene = max(w_boxes, 2.6)                               # la flecha debe alojar "Gen N"
        if x > 0 and x + w_gene > max_units:
            row, x = row + 1, 0.0
        xs, cx = [], x + (w_gene - w_boxes) / 2
        for w in widths:
            xs.append((cx, w))
            cx += w + 0.14
        items.append({"gene": k, "boxes": boxes, "xs": xs, "row": row, "x0": x, "x1": x + w_gene, "mods": mods})
        x += w_gene + 0.9
    return items, row + 1


def draw_panel_b(ax, items, n_rows, max_units=34):
    for it in items:
        yb, x0, x1 = -it["row"] * ROW_H, it["x0"], it["x1"]
        ay, ah = yb + 1.4, 0.5                                   # flecha del gen
        head = min(0.6, 0.3 * (x1 - x0))
        ax.add_patch(Polygon([(x0, ay), (x1 - head, ay), (x1 - head, ay - 0.1), (x1, ay + ah / 2),
                              (x1 - head, ay + ah + 0.1), (x1 - head, ay + ah), (x0, ay + ah)],
                             closed=True, facecolor=CATEGORY_COLORS[CAT_CORE], edgecolor="none"))
        ax.text((x0 + x1 - head) / 2, ay + ah / 2, f"Gen {it['gene']}", ha="center", va="center", color="white",
                fontsize=9.5, fontweight="bold")
        for s, e, lab in it["mods"]:                             # módulos M1, M2…
            mx0, mx1 = it["xs"][s][0], it["xs"][e][0] + it["xs"][e][1]
            ax.plot([mx0 + 0.05, mx1 - 0.05], [yb + 0.98] * 2, color=INK, lw=1)
            ax.text((mx0 + mx1) / 2, yb + 1.02, lab, ha="center", va="bottom", fontsize=8.5, fontweight="bold", color=INK)
        for (ab, name, mk), (bx, w) in zip(it["boxes"], it["xs"]):
            ax.add_patch(FancyBboxPatch((bx, yb), w, 0.8, boxstyle="round,pad=0,rounding_size=0.18", facecolor=BOX_FILL,
                                        edgecolor=INK if mk else "none", lw=2.0 if mk else 0))
            ax.text(bx + w / 2, yb + 0.4, ab, ha="center", va="center", color=INK, fontweight="bold",
                    fontsize=10 if ab in SHORT_LABELS else 8)
    ax.text(-0.35, 0.4, "B", fontsize=13, fontweight="bold", color=INK, ha="right", va="center")
    ax.set_xlim(-0.75, max_units + 0.3)
    ax.set_ylim(-(n_rows - 1) * ROW_H - 0.25, 2.0)
    ax.axis("off")


def panel_b_rows(genes, cls):
    return layout_panel_b(core_genes(genes, cls))[1]


def draw_architecture(fig, genes, title, kb_max, cls, marker_set=frozenset()):
    core = core_genes(genes, cls, marker_set)
    items, n_rows = layout_panel_b(core)
    numbered = {k for k, _ in core} | {k for k, g in enumerate(genes, start=1) if any(d[0] in marker_set for d in g["domains"])}
    gs = fig.add_gridspec(2, 1, height_ratios=[1.0, 0.25 + 0.95 * max(n_rows, 1)], hspace=0.05)
    axA, axB = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    draw_panel_a(axA, genes, kb_max, cls, numbered, marker_set)
    axA.set_title(title, loc="left", fontsize=12, fontweight="bold", color=INK, pad=4)
    if items:
        draw_panel_b(axB, items, n_rows)
    else:
        axB.text(0.0, 0.5, "B · Sin genes de biosíntesis central reconocidos", ha="left",
                 va="center", fontsize=10, color=MUTED, transform=axB.transAxes)
        axB.axis("off")


def block_height(genes, cls):
    return 1.0 + 0.25 + 0.95 * max(panel_b_rows(genes, cls), 1)


def architecture_legend(fig, y=0.0):
    handles = [plt.Rectangle((0, 0), 1, 1, color=CATEGORY_COLORS[c]) for c in CATEGORY_COLORS]
    handles.append(plt.Rectangle((0, 0), 1, 1, facecolor="white", edgecolor=INK, lw=2))
    fig.legend(handles, list(CATEGORY_COLORS) + ["borde negro = gen o dominio marcador del grupo (4.2)"],
               ncol=4, loc="upper center", bbox_to_anchor=(0.5, y), fontsize=9.5, handlelength=1.4, labelcolor=INK2)


def medoid(idx):
    """BGCs ordenados por su distancia de Jaccard media al resto (el primero es el medoide)."""
    D = np.nan_to_num(squareform(pdist(P[idx], metric="jaccard")), nan=1.0)
    return idx[np.argsort(D.mean(axis=1))]


def bgc_label(i):
    return f"{data.iloc[i][ACC]} región {data.iloc[i][REG] if REG in data.columns else i}"


def plot_architectures(blocks, kb, section, name, individual_name):
    """blocks = [(título, fila, clase, marcadores, sufijo)]: figura combinada + una figura por bloque."""
    heights = [block_height(all_arch[i], c) for _, i, c, _, _ in blocks]
    fig = plt.figure(figsize=(16, 1.1 * sum(heights) + 0.8))
    for sf, (t, i, c, mk, _) in zip(np.atleast_1d(fig.subfigures(len(blocks), 1, height_ratios=heights, hspace=0.04)), blocks):
        draw_architecture(sf, all_arch[i], t, kb, c, mk)
    architecture_legend(fig, y=0.0)
    show_and_save(fig, section, name)
    for (t, i, c, mk, suffix), h in zip(blocks, heights):
        fig = plt.figure(figsize=(16, 1.1 * h + 0.8))
        draw_architecture(fig, all_arch[i], t, kb, c, mk)
        architecture_legend(fig, y=0.0)
        show_and_save(fig, section, f"{individual_name}_{suffix}", individual=True)


map_col, seq_col = CONFIG["pfam_map_column"], CONFIG["sequence_column"]
ARCH_OK = map_col in data.columns
if not ARCH_OK:
    print(f"No existe la columna '{map_col}': se omiten 4.3 – 4.5.")
else:
    seq_src = data[seq_col] if seq_col in data.columns else pd.Series([np.nan] * len(data))
    all_arch = [parse_architecture(m, s) for m, s in zip(data[map_col], seq_src)]
    has_arch = np.array([len(a) > 0 for a in all_arch])
    print(f"Arquitecturas reconstruidas: {has_arch.sum()} de {len(all_arch)} BGCs · genes por BGC (mediana): "
          f"{int(np.median([len(a) for a in all_arch if a]))}")

    # ---- ¿Qué parte de los dominios de la tabla aparece en el mapa? --------------------
    col_of = {name: j for j, name in enumerate(domain_names)}
    in_map = np.zeros(P.shape, bool)                             # in_map[BGC, dominio] = el dominio aparece en el mapa del BGC
    for i, genes in enumerate(all_arch):
        for gn in genes:
            for d in gn["domains"]:
                if d[0] in col_of:
                    in_map[i, col_of[d[0]]] = True
    map_gaps = pd.DataFrame({"dominio": domain_names, "BGCs_en_tabla": P.sum(axis=0),
                             "BGCs_en_mapa": (P & in_map).sum(axis=0)})
    map_gaps["faltan_en_mapa"] = map_gaps["BGCs_en_tabla"] - map_gaps["BGCs_en_mapa"]
    map_gaps = save_table(map_gaps[map_gaps["faltan_en_mapa"] > 0]
                          .sort_values("faltan_en_mapa", ascending=False).reset_index(drop=True),
                          "4.3", "dominios_ausentes_del_mapa")
    map_coverage = float((P & in_map).sum() / P.sum())
    print(f"El mapa contiene {int((P & in_map).sum()):,} de las {int(P.sum()):,} presencias de dominio de la tabla "
          f"({100 * map_coverage:.1f} %). Los que más faltan: "
          + ", ".join(f"{r['dominio']} ({r['faltan_en_mapa']} BGCs)" for _, r in map_gaps.head(5).iterrows()))

    reps = [(g, i) for g in group_names for i in medoid(np.where((groups == g) & has_arch)[0])[:CONFIG["arch_representatives"]]]
    kb_max = max(sum(gn["length_aa"] * 3e-3 for gn in all_arch[i]) + GAP_KB * len(all_arch[i]) for _, i in reps)
    plot_architectures([(f"{g} · representante: {y[i]} · {bgc_label(i)} · {len(all_arch[i])} genes", i, y[i],
                         markers_by_group[g], g) for g, i in reps], kb_max, "4.3", "arquitectura_grupos",
                       "arquitectura")

    rep_genes = save_table(pd.DataFrame([{
        "grupo": g, "BGC": bgc_label(i), "clase": y[i], "gen": k, "longitud_aa": gn["length_aa"],
        "categoria": gene_category(gn, y[i]),
        "dominios_N_a_C": " · ".join(d[0] for d in sorted(gn["domains"], key=lambda d: d[2])),
        "dominios_marcadores": ", ".join(sorted({d[0] for d in gn["domains"]} & markers_by_group[g])),
    } for g, i in reps for k, gn in enumerate(all_arch[i], start=1)]), "4.3", "genes_representantes")
    for g, i in reps:
        print(f"{g}: representante = {y[i]} · {bgc_label(i)} · genes con dominio marcador: "
              + (", ".join(str(k) for k, gn in enumerate(all_arch[i], start=1) if any(d[0] in markers_by_group[g] for d in gn["domains"])) or "—"))

## 4.4 Módulos, tamaño y arquitecturas proteicas por grupo

En el artículo de referencia, la diferencia arquitectónica decisiva entre clusters relacionados fue la presencia o ausencia de un módulo NRPS completo. Aquí se compara entre grupos:

- **4.4a:** qué porcentaje de los BGCs de cada grupo tiene cada dominio de ensamblaje (C, A, T, TE, KS, AT; según la tabla) y qué porcentaje tiene al menos un módulo NRPS/PKS (según el mapa);
- **4.4b:** el número de genes (`num_cds_total`) y la longitud del BGC (`length_kb`), tal como vienen en la tabla;
- **4.4c:** las **arquitecturas proteicas** (secuencia ordenada de dominios de una proteína, p. ej. `Condensation · AMP-binding · AMP-binding_C · PP-binding`) cuya frecuencia más cambia entre grupos.

In [ ]:
from collections import Counter

if ARCH_OK:
    ASSEMBLY = {"C": "Condensation", "A": "AMP-binding", "T": "PP-binding", "TE": "Thioesterase", "KS": "ketoacyl-synt",
                "AT": "Acyl_transf_1"}
    per_bgc = pd.DataFrame({"grupo": groups, "clase": y})
    per_bgc["módulos"] = [n_modules(a, c) for a, c in zip(all_arch, y)]     # módulos NRPS/PKS (A + T o KS + T), según el mapa
    per_bgc["genes"] = data["num_cds_total"].to_numpy() if "num_cds_total" in data.columns else [len(a) for a in all_arch]
    per_bgc["kb"] = data["length_kb"].to_numpy() if "length_kb" in data.columns else [
        sum(gn["length_aa"] for gn in a) * 3e-3 for a in all_arch]
    save_table(per_bgc, "4.4", "arquitectura_por_bgc")

    # Presencia de cada dominio de ensamblaje según la TABLA (completa); módulos según el MAPA
    MODULE_COL = "≥ 1 módulo\nNRPS/PKS"
    has = pd.DataFrame({ab: (P[:, col_of[pf]] if pf in col_of else np.zeros(len(y), bool)) for ab, pf in ASSEMBLY.items()})
    has[MODULE_COL] = per_bgc["módulos"].to_numpy() > 0
    pct_with = 100 * has.groupby(groups).mean().reindex(group_names)
    summary = pd.concat([group_sizes.rename("n_BGC"),
                         pct_with.round(0).rename(columns=lambda c: "pct_con_" + c.replace("\n", " ")),
                         per_bgc.groupby("grupo")["módulos"].mean().round(2).rename("modulos_media"),
                         per_bgc.groupby("grupo")[["genes", "kb"]].median().add_prefix("mediana_")], axis=1)
    save_table(summary.rename_axis("grupo").reset_index(), "4.4", "resumen_por_grupo")

    # ---- 4.4a Dominios de ensamblaje y módulos por grupo -------------------------------
    fig, ax = plt.subplots(figsize=(1.15 * pct_with.shape[1] + 3.2, 0.8 * k_groups + 2.4))
    heatmap(ax, pct_with, [f"{g} (n = {group_sizes[g]})" for g in group_names],
            [f"{c}\n({ASSEMBLY[c]})" if c in ASSEMBLY else c for c in pct_with.columns], vmin=0, vmax=100,
            annot=np.vectorize(lambda v: f"{v:.0f}")(pct_with.to_numpy()), cbar_label="% de BGCs del grupo",
            annot_size=10)
    ax.tick_params(axis="y", rotation=0)
    ax.tick_params(axis="x", rotation=0, labelsize=9)
    titles(ax, "Dominios de ensamblaje y módulos por grupo",
           "Número = % de BGCs del grupo con el dominio (según la tabla) · última columna = % con al menos un módulo")
    show_and_save(fig, "4.4", "a_dominios_ensamblaje")

    # ---- 4.4b Genes y longitud por grupo ------------------------------------------------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 0.6 * k_groups + 2.4), sharey=True)
    labels_g = [f"{g} (n = {group_sizes[g]})" for g in group_names]
    hbox(ax1, [per_bgc.loc[groups == g, "genes"] for g in group_names], labels_g)
    hbox(ax2, [per_bgc.loc[groups == g, "kb"] for g in group_names], labels_g)
    ax1.set_xlabel("Genes por BGC")
    ax2.set_xlabel("Longitud del BGC (kb)")
    ax1.set_title("Genes por BGC", loc="left", fontsize=11)
    ax2.set_title("Longitud del BGC", loc="left", fontsize=11)
    fig.suptitle("Tamaño de los BGCs por grupo (caja = 50 % central, línea gruesa = mediana)",
                 x=0.01, ha="left", fontsize=13, fontweight="bold")
    fig.tight_layout()
    show_and_save(fig, "4.4", "b_tamano_grupos")

    # ---- 4.4c Arquitecturas proteicas que más cambian entre grupos --------
    freq = {g: Counter() for g in group_names}
    for a, g in zip(all_arch, groups):
        freq[g].update({" · ".join(d[0] for d in sorted(gn["domains"], key=lambda d: d[2])) for gn in a if gn["domains"]})
    arch_df = pd.DataFrame({g: {k: v / group_sizes[g] for k, v in freq[g].items()} for g in group_names}).fillna(0)
    arch_df["diferencia"] = arch_df.max(axis=1) - arch_df.min(axis=1)
    arch_df = arch_df.sort_values("diferencia", ascending=False)
    save_table(arch_df.round(4).rename_axis("arquitectura_proteica").reset_index(), "4.4",
               "arquitecturas_proteicas")
    top_arch = arch_df.head(15)
    short = [a if len(a) <= 64 else a[:63] + "…" for a in top_arch.index]
    fig, ax = plt.subplots(figsize=(1.3 * k_groups + 7.5, 0.4 * len(top_arch) + 2.0))
    heatmap(ax, 100 * top_arch[group_names].to_numpy(), short, group_names, vmin=0, vmax=100,
            annot=np.vectorize(lambda v: f"{v:.0f}")(100 * top_arch[group_names].to_numpy()),
            cbar_label="% de BGCs del grupo")
    ax.tick_params(axis="y", rotation=0, labelsize=9)
    ax.tick_params(axis="x", rotation=0)
    titles(ax, "Arquitecturas proteicas que más cambian entre grupos",
           "Dominios de una proteína en orden N→C · número = % de BGCs del grupo que tienen esa proteína")
    show_and_save(fig, "4.4", "c_arquitecturas_proteicas")
    display(summary)

## 4.5 Análisis de los BGCs de sideróforos

Esta sección aísla los BGCs de las clases de sideróforos (`CONFIG["siderophore_classes"]`: `nrp-metallophore`, sideróforos ensamblados por NRPS, y `ni-siderophore`, sideróforos independientes de NRPS sintetizados por enzimas de la familia IucA/IucC). Como comparación se incluyen las clases de `CONFIG["siderophore_compare_classes"]` (`nrps`), porque la sección 1.3 muestra que muchos BGCs `nrps` están dentro de las coordenadas de un BGC `nrp-metallophore` del mismo genoma.

1. **Tabla:** en qué grupo de 4.1 cae cada clase.
2. **4.5a:** porcentaje de BGCs de cada clase con cada dominio de una lista de dominios de biosíntesis y captación de sideróforos (`SID_DOMAINS`, editable; la descripción es el nombre de la familia en Pfam).
3. **4.5b:** número de módulos NRPS por BGC.
4. **4.5c:** arquitectura (paneles A y B, como en 4.3) del representante de cada clase de sideróforo en cada grupo con al menos `sid_min_bgcs` BGCs.

In [ ]:
# Dominios Pfam de biosíntesis y captación de sideróforos (nombre exacto → nombre de la familia en Pfam)
SID_DOMAINS = {
    "IucA_IucC": "familia IucA/IucC (sintetasa NIS)",
    "Lys_Orn_oxgnase": "L-lisina 6-monooxigenasa",
    "Acetyltransf_8": "acetiltransferasa (GNAT)",
    "FhuF": "reductasa férrica tipo FhuF",
    "Condensation": "dominio de condensación (C)",
    "AMP-binding": "enzima de unión a AMP (A)",
    "PP-binding": "sitio de unión de fosfopanteteína (T)",
    "Thioesterase": "dominio tioesterasa (TE)",
    "MbtH": "proteína tipo MbtH",
    "Chorismate_bind": "enzima de unión a corismato",
    "Isochorismatase": "familia isocorismatasa",
    "TauD": "dioxigenasa TauD/TfdA",
    "TonB_dep_Rec_b-barrel": "receptor dependiente de TonB (barril β)",
    "Plug": "dominio tapón del receptor TonB",
    "FecCD": "familia de transporte FecCD",
    "Peripla_BP_2": "proteína periplásmica de unión",
}
sid_classes = [c for c in class_order if c in CONFIG["siderophore_classes"]]
cmp_classes = [c for c in class_order if c in CONFIG["siderophore_compare_classes"]]
show_classes = sid_classes + cmp_classes
sid_mask = np.isin(y, sid_classes)

if not ARCH_OK or not sid_mask.any():
    print("No hay BGCs de las clases de sideróforos o falta la columna del mapa Pfam: se omite 4.5.")
else:
    print(f"BGCs de sideróforos: {sid_mask.sum()} de {len(y)} (" + ", ".join(f"{c} {int((y == c).sum())}" for c in sid_classes) + ")")
    for c in cmp_classes:                                        # solapamiento con la sección 1.3
        inside = {i for i in np.where(y == c)[0] if any(o in sid_classes for o in overlap_classes.get(i, ()))}
        print(f"BGCs {c} que se solapan con un BGC de sideróforo del mismo genoma: {len(inside)} de {int((y == c).sum())}")

    # ---- Tabla: clases × grupos -------------------------------------------------------------
    sid_groups = (pd.crosstab(pd.Series(y[np.isin(y, show_classes)], name="clase"),
                              pd.Series(groups[np.isin(y, show_classes)], name="grupo"))
                  .reindex(index=show_classes, columns=group_names, fill_value=0))
    save_table(sid_groups.reset_index(), "4.5", "sideroforos_por_grupo")
    display(sid_groups)

    # ---- 4.5a Dominios de sideróforos por clase ------------------------------------------------
    present = [d for d in SID_DOMAINS if d in domain_names]
    prev_sid = pd.DataFrame({c: [100 * P[y == c, domain_names.index(d)].mean() for d in present] for c in show_classes},
                            index=present)
    prev_sid = prev_sid.loc[prev_sid.max(axis=1) > 0]
    save_table(prev_sid.round(1).rename_axis("dominio").reset_index().assign(
        **{"familia_pfam": [SID_DOMAINS[d] for d in prev_sid.index]}), "4.5", "dominios_sideroforos")
    col_lab = [f"{c}\n(n = {int((y == c).sum())})" + ("\ncomparación" if c in cmp_classes else "") for c in show_classes]
    fig, ax = plt.subplots(figsize=(1.6 * len(show_classes) + 7.5, 0.42 * len(prev_sid) + 2.4))
    heatmap(ax, prev_sid.to_numpy(), [f"{d} — {SID_DOMAINS[d]}" for d in prev_sid.index], col_lab, vmin=0, vmax=100,
            annot=np.where(prev_sid.to_numpy() > 0, np.vectorize(lambda v: f"{v:.0f}")(prev_sid.to_numpy()), ""),
            cbar_label="% de BGCs de la clase")
    ax.tick_params(axis="y", rotation=0, labelsize=9.5)
    ax.tick_params(axis="x", rotation=0)
    titles(ax, "Dominios de biosíntesis y captación de sideróforos",
           "Número = % de BGCs de la clase con el dominio · vacío = 0")
    show_and_save(fig, "4.5", "a_dominios_sideroforos")

    # ---- 4.5b Módulos NRPS por BGC --------------------------------------------------------------
    rng = np.random.default_rng(RS)
    fig, ax = plt.subplots(figsize=(9, 0.75 * len(show_classes) + 1.9))
    for r, c in enumerate(show_classes):
        v = per_bgc.loc[y == c, "módulos"].to_numpy()
        st = CLASS_STYLE[c]
        ax.scatter(v + rng.uniform(-0.12, 0.12, len(v)), r + rng.uniform(-0.22, 0.22, len(v)), s=28 * (1.5 if st["hollow"] else 1),
                   marker=st["marker"], facecolors="none" if st["hollow"] else st["color"],
                   edgecolors=st["color"] if st["hollow"] else SURFACE, linewidths=1.2 if st["hollow"] else 0.5, alpha=0.85)
        ax.plot([np.median(v)] * 2, [r - 0.32, r + 0.32], color=INK, lw=2.2)
        ax.text(v.max() + 0.45, r, f"mediana = {np.median(v):g}", va="center", fontsize=9.5, color=INK2)
    ax.set_yticks(range(len(show_classes)))
    ax.set_yticklabels([c + (" (comparación)" if c in cmp_classes else "") for c in show_classes])
    ax.set_ylim(len(show_classes) - 0.5, -0.5)
    ax.set_xlim(-0.6, per_bgc.loc[np.isin(y, show_classes), "módulos"].max() + 2.2)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Módulos NRPS/PKS por BGC (A + T o KS + T)")
    titles(ax, "Módulos de ensamblaje en los BGCs de sideróforos",
           "Cada punto es un BGC (con un pequeño desplazamiento para que no se encimen) · línea negra = mediana")
    show_and_save(fig, "4.5", "b_modulos_sideroforos")

    sid_summary = (per_bgc[np.isin(y, show_classes)].groupby(["clase", "grupo"])
                   .agg(n_BGC=("kb", "size"), modulos_mediana=("módulos", "median"),
                        modulos_max=("módulos", "max"), genes_mediana=("genes", "median"),
                        kb_mediana=("kb", "median")).reset_index())
    save_table(sid_summary, "4.5", "resumen_sideroforos")

    # ---- 4.5c Arquitectura del representante de cada clase de sideróforo en cada grupo --
    reps_sid = []
    for c in sid_classes:
        for g in group_names:
            idx = np.where((y == c) & (groups == g) & has_arch)[0]
            if len(idx) >= CONFIG["sid_min_bgcs"]:
                reps_sid.append((c, g, medoid(idx)[0], len(idx)))
    if reps_sid:
        kb_sid = max(sum(gn["length_aa"] * 3e-3 for gn in all_arch[i]) + GAP_KB * len(all_arch[i]) for _, _, i, _ in reps_sid)
        plot_architectures([(f"{c} · {g} (n = {n}) · representante: {bgc_label(i)} · {len(all_arch[i])} genes",
                             i, c, markers_by_group[g], f"{slug(c)}_{g}") for c, g, i, n in reps_sid], kb_sid, "4.5",
                           "c_arquitectura_sideroforos", "c_arquitectura")
    sid_summary

## 4.6 Distancias de alineamiento entre secuencias

Hasta aquí los BGCs se comparan por sus dominios Pfam. Esta sección los compara por la **secuencia de sus proteínas**, para ver si las clases también se distinguen a nivel de secuencia.

1. Se toma una submuestra estratificada de `seq_per_class` BGCs por clase (el número de alineamientos crece con n²) y, de cada BGC, sus `seq_max_proteins_per_bgc` proteínas más largas.
2. Cada par de proteínas se alinea con **Smith–Waterman local** (BLOSUM62, apertura −11, extensión −1; Biopython). La puntuación se normaliza: s(a, b) = S(a, b) / √(S(a, a) · S(b, b)), entre 0 y 1.
3. Similitud entre dos BGCs = media de los mejores emparejamientos de proteínas en ambas direcciones; **distancia = 1 − similitud**.

- **4.6a:** matriz de distancias ordenada por clase. Bloques oscuros en la diagonal = BGCs de la misma clase con proteínas parecidas.
- **4.6b:** distancia media entre y dentro de clases.
- **4.6c:** relación entre la distancia por secuencia y la distancia por dominios del modelo (Jaccard sobre los *k* dominios). La **prueba de Mantel** evalúa si ambas matrices están correlacionadas.

Es la sección más lenta del notebook (unos minutos).

In [ ]:
from joblib import Parallel, delayed

BIO_OK = importlib.util.find_spec("Bio") is not None
SEQ_OK = BIO_OK and seq_col in data.columns
AA_VALID = set("ACDEFGHIKLMNPQRSTVWYBZX")


def parse_proteins(cell):
    """Proteínas del BGC separadas por "|" (también acepta FASTA o ; , espacios); U, O, J → X."""
    if pd.isna(cell):
        return []
    text = str(cell)
    chunks = (["".join(r.splitlines()[1:]) for r in text.split(">") if r.strip()] if ">" in text
              else re.split(r"[|;,\s]+", text))
    prots = []
    for ch in chunks:
        ch = "".join(c if c in AA_VALID else "X" for c in re.split(r"[:=]", ch)[-1].upper() if c.isalpha())
        if len(ch) >= CONFIG["seq_min_length"]:
            prots.append(ch)
    return prots


def make_aligner():
    """Smith–Waterman local con BLOSUM62 (implementación en C de Biopython)."""
    from Bio.Align import PairwiseAligner, substitution_matrices
    al = PairwiseAligner()
    al.mode = "local"
    al.substitution_matrix = substitution_matrices.load("BLOSUM62")
    al.open_gap_score, al.extend_gap_score = -11, -1
    return al


def row_similarities(i, prots, self_sc):
    """Similitud del BGC i con cada BGC j > i (se reparte por filas entre los núcleos)."""
    al = make_aligner()
    out = []
    for j in range(i + 1, len(prots)):
        S = np.array([[al.score(a, b) for b in prots[j]] for a in prots[i]])
        N = np.clip(S / np.sqrt(np.outer(self_sc[i], self_sc[j])), 0, 1)
        out.append((j, 0.5 * (N.max(axis=1).mean() + N.max(axis=0).mean())))
    return i, out


if not SEQ_OK:
    print("Falta Biopython o la columna de secuencias: se omite 4.6 (ejecuta la celda 0.1).")
else:
    all_prots = data[seq_col].map(parse_proteins)
    n_prot = all_prots.map(len).to_numpy()
    rng = np.random.default_rng(RS)
    seq_rows = np.concatenate([np.sort(rng.choice(np.where((y == c) & (n_prot > 0))[0],
                                                  min(CONFIG["seq_per_class"], int(((y == c) & (n_prot > 0)).sum())),
                                                  replace=False)) for c in class_order])
    seq_y = y[seq_rows]
    prots = [sorted(all_prots.iloc[r], key=len, reverse=True)[:CONFIG["seq_max_proteins_per_bgc"]] for r in seq_rows]
    al0 = make_aligner()
    self_sc = [np.array([al0.score(p, p) for p in pp]) for pp in prots]     # autopuntuaciones para normalizar
    n_seq = len(seq_rows)
    n_aln = sum(len(prots[i]) * len(prots[j]) for i in range(n_seq) for j in range(i + 1, n_seq))

    t0 = time.time()
    sim = np.eye(n_seq)
    for i, row in Parallel(n_jobs=CONFIG["n_jobs"])(delayed(row_similarities)(i, prots, self_sc) for i in range(n_seq)):
        for j, s in row:
            sim[i, j] = sim[j, i] = s
    D_seq = 1 - sim
    seq_ids = [f"{bgc_label(r)} ({c})" for r, c in zip(seq_rows, seq_y)]
    save_table(pd.DataFrame(D_seq.round(4), columns=seq_ids).assign(BGC=seq_ids).set_index("BGC").reset_index(),
               "4.6", "matriz_distancias_alineamiento")
    print(f"{n_seq} BGCs ({CONFIG['seq_per_class']} por clase) · {n_aln:,} alineamientos en {time.time() - t0:.0f} s")

**Figuras 4.6a – 4.6c.** La celda anterior calcula la matriz de distancias de alineamiento; esta dibuja las tres figuras descritas arriba y hace la prueba de Mantel.

In [ ]:
if SEQ_OK:
    # ---- Distancia por dominios del modelo (Jaccard sobre presencia de los k dominios) --
    D_pfam = np.nan_to_num(squareform(pdist(P[seq_rows][:, sel_idx], metric="jaccard")))
    present = [c for c in class_order if c in set(seq_y)]
    bounds = np.cumsum([0] + [int((seq_y == c).sum()) for c in present])

    # ---- 4.6a Matriz ordenada por clase -------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9.5, 8.3))
    im = ax.imshow(D_seq, cmap=SEQ_CMAP.reversed(), vmin=0, vmax=1, interpolation="nearest")
    for b in bounds[1:-1]:                                       # líneas entre clases
        ax.axhline(b - 0.5, color=SURFACE, lw=2)
        ax.axvline(b - 0.5, color=SURFACE, lw=2)
    centres = (bounds[:-1] + bounds[1:]) / 2 - 0.5
    ax.set_xticks(centres)
    ax.set_xticklabels(present)
    rotate_xlabels(ax)
    ax.set_yticks(centres)
    ax.set_yticklabels(present)
    ax.tick_params(length=0)
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.75, label="Distancia de alineamiento (1 − similitud)")
    titles(ax, "Distancias de alineamiento entre BGCs, ordenadas por clase",
           f"{n_seq} BGCs · oscuro = proteínas más parecidas · cada bloque es una clase")
    show_and_save(fig, "4.6", "a_matriz_alineamiento")

    # ---- 4.6b Distancia media entre y dentro de clases -----------------------------------
    class_mean = pd.DataFrame(index=present, columns=present, dtype=float)
    for a in present:
        for b in present:
            block = D_seq[np.ix_(seq_y == a, seq_y == b)]
            if a == b:
                block = block[~np.eye(len(block), dtype=bool)]       # sin la diagonal (distancia 0 consigo mismo)
            class_mean.loc[a, b] = block.mean() if block.size else np.nan
    save_table(class_mean.round(4).rename_axis("clase").reset_index(), "4.6", "distancia_media_clases")
    fig, ax = plt.subplots(figsize=(9, 7.6))
    heatmap(ax, class_mean.to_numpy(float), present, present, cmap=SEQ_CMAP.reversed(), vmin=0, vmax=1,
            cbar_label="Distancia media", annot_size=9)
    rotate_xlabels(ax)
    ax.tick_params(axis="y", rotation=0)
    intra = np.nanmean(np.diag(class_mean.to_numpy(float)))
    inter = np.nanmean(class_mean.to_numpy(float)[~np.eye(len(present), dtype=bool)])
    titles(ax, "Distancia de alineamiento media entre y dentro de clases",
           f"Diagonal = dentro de la clase · media dentro = {intra:.2f} · media entre clases = {inter:.2f}")
    show_and_save(fig, "4.6", "b_distancia_media_clases")

    # ---- 4.6c Secuencia frente a dominios: prueba de Mantel ------------------------------------
    iu = np.triu_indices(n_seq, 1)
    r_obs = spearmanr(D_seq[iu], D_pfam[iu])[0]
    perm_r = []
    for _ in range(CONFIG["mantel_permutations"]):               # permuta filas y columnas a la vez
        p = rng.permutation(n_seq)
        perm_r.append(spearmanr(D_seq[iu], D_pfam[np.ix_(p, p)][iu])[0])
    p_mantel = (1 + np.sum(np.array(perm_r) >= r_obs)) / (1 + len(perm_r))
    same = (seq_y[iu[0]] == seq_y[iu[1]])
    save_table(pd.DataFrame({"distancia_pfam": D_pfam[iu].round(4), "distancia_secuencia": D_seq[iu].round(4),
                             "misma_clase": same}), "4.6", "secuencia_vs_dominios")
    fig, ax = plt.subplots(figsize=(8.5, 6.4))
    ax.scatter(D_pfam[iu][~same], D_seq[iu][~same], s=10, c=CONTEXT, linewidths=0, label="pares de clases distintas")
    ax.scatter(D_pfam[iu][same], D_seq[iu][same], s=14, c=SERIES[0], linewidths=0, label="pares de la misma clase")
    ax.set_xlabel(f"Distancia por dominios del modelo (Jaccard sobre {k_optimal} dominios)")
    ax.set_ylabel("Distancia de alineamiento")
    ax.set_xlim(-0.03, 1.03)
    ax.set_ylim(-0.03, 1.03)
    ax.legend(loc="lower right")
    titles(ax, "¿Los BGCs con dominios parecidos también tienen proteínas parecidas?",
           f"Prueba de Mantel (Spearman): r = {r_obs:.2f} · p = {p_mantel:.3f} ({CONFIG['mantel_permutations']} permutaciones)")
    show_and_save(fig, "4.6", "c_secuencia_vs_dominios")
    print(f"Distancia media dentro de clase = {intra:.3f} · entre clases = {inter:.3f} · "
          f"Mantel r = {r_obs:.3f}, p = {p_mantel:.4f}")

---
# Parte 5 · Comparación de clústeres: dominios compartidos, repeticiones y sintenia

**Pregunta:** ¿qué dominios y genes se repiten entre los clústeres y dentro de ellos, y qué tan conservado está el orden de los genes (sintenia)?

| Sección | Qué hace | Figuras principales |
|---|---|---|
| 5.1 | Mapa de calor de todos los clústeres contra los dominios característicos de cada clase | `5.1_mapa_calor_clusters_dominios` |
| 5.2 | Dominios que se repiten en los clústeres: en cuántos BGCs y en cuántas clases aparece cada dominio | `5.2_dominios_repetidos_entre_clusters` |
| 5.3 | Genes que se repiten dentro de un mismo clúster (5.3a) y dominios que se repiten dentro de un mismo gen (5.3b) | `5.3_a_genes_repetidos`, `5.3_b_dominios_repetidos_en_un_gen` |
| 5.4 | Relación entre sintenias: qué tan parecido es el orden de los genes entre clústeres | `5.4_a_matriz_sintenia`, `5.4_b_sintenia_por_clase` |
| 5.5 | Todos los clústeres alineados en orden, locus por locus | `5.5_alineamiento_<clase>_locus<n>` |

**De dónde sale cada resultado.** Las secciones 5.1 y 5.2 usan las columnas `PF…` de la tabla, que registran todos los dominios de cada BGC. Las secciones 5.3 a 5.5 trabajan gen a gen: usan `protein_sequences` para decidir cuándo dos genes son el mismo gen y `protein_seq_pfam_map` para saber qué dominios tiene cada gen. Como muestra la sección 4.3, ese mapa no contiene todos los dominios de la tabla, por lo que los nombres de los genes solo incluyen los dominios que sí están en el mapa.

**Supuestos.** La tabla no trae las coordenadas ni la hebra de cada gen. Las secciones 5.4 y 5.5 asumen que los genes de cada BGC están listados en su orden en el genoma. Además, la lista de genes de un BGC puede sobresalir de las coordenadas de `location` (la sección 5.4 lo cuantifica).

## 5.1 Mapa de calor de clústeres contra dominios

Cada fila es un BGC y cada columna un dominio Pfam. Mostrar todos los dominios haría ilegible la figura, así que se muestran los dominios **más característicos de cada clase** (hasta `heatmap_domains_per_class` por clase): los que están en al menos la mitad de los BGCs de la clase y cuya frecuencia en la clase supera más a su frecuencia en el resto. Cada dominio se asigna a una sola clase, aquella en la que esa diferencia es mayor.

**Cómo leerlo:** las filas están agrupadas por clase y las columnas por la clase a la que caracterizan, así que una clase bien definida aparece como un bloque oscuro en la diagonal. Un bloque oscuro fuera de la diagonal indica que dos clases comparten esos dominios. El color es la puntuación del dominio en la tabla (0 = ausente); como las puntuaciones de los dominios presentes son todas altas, el mapa se lee en la práctica como presencia o ausencia.

In [ ]:
from collections import Counter
from itertools import combinations
from scipy.cluster.hierarchy import fcluster, leaves_list, linkage
from scipy.spatial.distance import pdist, squareform

N_CHAR = CONFIG["heatmap_domains_per_class"]
prev_class = np.array([P[y == c].mean(axis=0) for c in class_order])                    # clases × dominios: fracción de BGCs con el dominio
diff_class = prev_class - np.array([P[y != c].mean(axis=0) for c in class_order])       # frecuencia en la clase − en el resto
owner = diff_class.argmax(axis=0)                                # clase a la que más caracteriza cada dominio
blocks = []
for k in range(len(class_order)):
    mine = np.where((owner == k) & (prev_class[k] >= 0.5))[0]    # dominios propios de la clase y presentes en al menos la mitad de sus BGCs
    blocks.append(mine[np.argsort(-diff_class[k, mine], kind="stable")][:N_CHAR])
heat_cols = np.concatenate(blocks)

heat_rows = []                                                   # BGCs por clase; dentro de cada clase, perfiles parecidos juntos
for c in class_order:
    idx = np.where(y == c)[0]
    if len(idx) > 2:
        idx = idx[leaves_list(linkage(np.nan_to_num(pdist(P[idx][:, heat_cols], "jaccard"), nan=1.0), "average"))]
    heat_rows += list(idx)
heat_rows = np.array(heat_rows)

save_table(pd.DataFrame({
    "clase": np.repeat(class_order, [len(b) for b in blocks]),
    "dominio": [domain_names[j] for j in heat_cols],
    "pct_en_la_clase": [round(100 * prev_class[k, j], 1) for k, b in enumerate(blocks) for j in b],
    "pct_en_el_resto": [round(100 * (prev_class[k, j] - diff_class[k, j]), 1) for k, b in enumerate(blocks) for j in b]}),
    "5.1", "dominios_caracteristicos")

row_bounds = np.cumsum([0] + [int((y == c).sum()) for c in class_order])
col_bounds = np.cumsum([0] + [len(b) for b in blocks])
fig, ax = plt.subplots(figsize=(0.21 * len(heat_cols) + 4.4, 9.8))
im = ax.imshow(X[np.ix_(heat_rows, heat_cols)], cmap=SEQ_CMAP, vmin=0, vmax=1, aspect="auto", interpolation="nearest")
for b in row_bounds[1:-1]:
    ax.axhline(b - 0.5, color=INK, lw=0.8)
for b in col_bounds[1:-1]:
    ax.axvline(b - 0.5, color=INK, lw=0.8)
ax.set_yticks((row_bounds[:-1] + row_bounds[1:]) / 2 - 0.5)
ax.set_yticklabels([f"{c} ({counts[c]})" for c in class_order])
ax.set_xticks(range(len(heat_cols)))
ax.set_xticklabels([domain_names[j] for j in heat_cols], rotation=90, fontsize=8.5)
ax.tick_params(length=0)
ax.grid(False)
for k, c in enumerate(class_order):                              # clase a la que caracteriza cada bloque de columnas (en dos líneas si el bloque es estrecho)
    if len(blocks[k]):
        ax.text((col_bounds[k] + col_bounds[k + 1]) / 2 - 0.5, 1.006, c if len(blocks[k]) >= 6 else c.replace("-", "-\n"),
                transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=9, fontweight="bold", color=INK, linespacing=1.0)
fig.colorbar(im, ax=ax, shrink=0.5, pad=0.015, label="Puntuación del dominio en la tabla (0 = ausente)")
ax.set_title("Mapa de calor: clústeres × dominios característicos de cada clase", loc="left", pad=58)
ax.text(0, 1.058, f"Cada fila es un BGC ({len(heat_rows)}) · arriba, la clase a la que caracteriza cada bloque de dominios · "
        f"puntuación de los dominios presentes: {X[X > 0].min():.2f}–{X.max():.2f}",
        transform=ax.transAxes, ha="left", va="bottom", fontsize=10, color=INK2)
show_and_save(fig, "5.1", "mapa_calor_clusters_dominios")

## 5.2 Dominios que se repiten en los clústeres

Para cada dominio Pfam se cuenta en cuántos BGCs aparece y cómo se reparte entre las clases:

| Columna | Qué mide |
|---|---|
| `n_BGC` / `pct_BGC` | BGCs que tienen el dominio |
| `clases_con_el_dominio` | Clases en las que al menos un BGC tiene el dominio |
| `clases_donde_es_frecuente` | Clases en las que el dominio está en al menos la mitad de los BGCs |
| `pct_<clase>` | Porcentaje de los BGCs de esa clase que tienen el dominio |

**Cómo leer la figura:** cada fila es uno de los dominios más repetidos; la primera columna es el porcentaje sobre todos los BGCs y las demás, el porcentaje dentro de cada clase. Una fila oscura en varias columnas es un dominio compartido por varias clases; una fila oscura en una sola columna es un dominio muy repetido pero propio de esa clase. Varios dominios aparecen casi siempre en los mismos BGCs, por lo que sus filas son casi iguales.

In [ ]:
recurrence = pd.DataFrame({"dominio": domain_names, "n_BGC": P.sum(axis=0), "pct_BGC": (100 * P.mean(axis=0)).round(1),
                           "clases_con_el_dominio": (prev_class > 0).sum(axis=0),
                           "clases_donde_es_frecuente": (prev_class >= 0.5).sum(axis=0)})
for k, c in enumerate(class_order):
    recurrence[f"pct_{c}"] = (100 * prev_class[k]).round(1)
recurrence = save_table(recurrence.sort_values("n_BGC", ascending=False, kind="stable").reset_index(drop=True),
                        "5.2", "dominios_repetidos_entre_clusters")

top = recurrence.head(CONFIG["recurrent_top_domains"])
M = top[["pct_BGC"] + [f"pct_{c}" for c in class_order]].to_numpy()
fig, ax = plt.subplots(figsize=(0.95 * (len(class_order) + 1) + 3.4, 0.33 * len(top) + 2.6))
heatmap(ax, M, top["dominio"].tolist(), [f"Todos\n({len(y)})"] + [f"{c}\n({counts[c]})" for c in class_order],
        vmin=0, vmax=100, annot=np.where(M >= 0.5, np.vectorize(lambda v: f"{v:.0f}")(M), ""), cbar_label="% de BGCs con el dominio",
        annot_size=8.5)
ax.axvline(1, color=INK, lw=1.6)                                 # separa "Todos" de las clases
ax.tick_params(axis="y", rotation=0, labelsize=9.5)
ax.tick_params(axis="x", labelsize=8.5)
rotate_xlabels(ax)
titles(ax, f"Los {len(top)} dominios que más se repiten en los clústeres",
       "Número = % de BGCs con el dominio (primera columna: todos los BGCs; resto: por clase) · vacío = menos de 0.5 %")
show_and_save(fig, "5.2", "dominios_repetidos_entre_clusters")

FREQ = "clases_donde_es_frecuente"
in_all = recurrence[recurrence["clases_con_el_dominio"] == len(class_order)]
shared = recurrence[recurrence[FREQ] >= 2]
widest = recurrence[recurrence[FREQ] == recurrence[FREQ].max()]
print(f"Dominios frecuentes (en ≥ 50 % de los BGCs) en 2 o más clases: {len(shared)} de {len(recurrence)}")
print(f"Ningún dominio es frecuente en más de {recurrence[FREQ].max()} clases ({', '.join(widest['dominio'].head(6))})")
print(f"Dominios presentes en al menos un BGC de cada una de las {len(class_order)} clases: {len(in_all)}"
      + (" (" + ", ".join(in_all["dominio"].head(8)) + ("…" if len(in_all) > 8 else "") + ")" if len(in_all) else ""))
recurrence.iloc[:, :5].head(10)                                   # los % por clase están en la figura y en el CSV

## 5.3 Genes y dominios que se repiten dentro de un clúster

**¿Cuándo dos genes son el mismo gen?** La tabla no trae nombres de genes, así que los genes se agrupan en **familias** por el parecido de sus proteínas: de la más larga a la más corta, cada proteína se une a la familia con cuyo representante tiene una similitud de al menos `gene_family_min_similarity`; si no se parece a ningún representante, funda una familia nueva. La **similitud** entre dos proteínas es la puntuación de su alineamiento local (BLOSUM62) dividida entre la mayor de las puntuaciones que obtiene cada una alineada consigo misma (normalmente, la de la más larga): vale 1 si son idénticas y baja si las secuencias difieren o si una es mucho más corta que la otra. Cada familia se nombra con su **arquitectura de dominios** más frecuente (la lista ordenada de los dominios Pfam del gen según el mapa). Así, dos genes con los mismos dominios pero secuencia distinta quedan en familias diferentes.

Con esa definición se miden dos tipos de repetición dentro de un mismo BGC:

- **5.3a · Genes repetidos en el mismo clúster:** una familia que aparece dos o más veces en el BGC. El panel derecho de la figura muestra la similitud de secuencia entre las copias.
- **5.3b · Dominios repetidos en el mismo gen:** un dominio que aparece en dos o más regiones separadas de la misma proteína (por ejemplo, el dominio `Condensation` de una NRPS con varios módulos).

**Limitaciones**

- El umbral de similitud es un parámetro: con un valor más bajo se unen genes más lejanos y aumentan los genes repetidos; con uno más alto, disminuyen. Como referencia, la celda imprime también cuántos BGCs tienen dos o más genes con la misma arquitectura de dominios sin exigir parecido de secuencia.
- Para no alinear cada proteína con todos los representantes, solo se prueban los (hasta tres) que comparten con ella más fragmentos de cuatro aminoácidos. Por eso alguna pareja con similitud apenas por encima del umbral puede quedar en familias distintas.
- En 5.3b, dos etiquetas iguales seguidas (sin residuos entre ellas) cuentan una sola vez, porque la tabla no permite saber si son dos copias en tándem o un mismo dominio detectado en dos partes. El recuento es por tanto un mínimo; la tabla resumen da también el recuento contando cada etiqueta (`n_BGC_contando_etiquetas_seguidas`).

La primera celda agrupa las proteínas en familias (tarda alrededor de medio minuto).

In [ ]:
def gene_instances(gene):
    """Dominios del gen en orden N→C, una entrada por región (dos etiquetas iguales seguidas cuentan una vez)."""
    seen, out = set(), []
    for name, _, start, _ in sorted(gene["domains"], key=lambda d: d[2]):
        if (name, start) not in seen:
            seen.add((name, start))
            out.append(name)
    return out


def gene_architecture(gene):
    """Arquitectura de dominios: lista ordenada de los dominios Pfam del gen ("" si no tiene ninguno en el mapa)."""
    return " · ".join(gene_instances(gene))


def short_arch(text, n=44):
    """Arquitecturas largas: primer y último dominio y cuántos hay en total."""
    parts = text.split(" · ")
    return text if len(text) <= n or len(parts) < 3 else f"{parts[0]} · … · {parts[-1]} ({len(parts)} dominios)"


def gene_label(name):
    """Etiqueta corta: primer dominio del gen y cuántos dominios más tiene."""
    parts = name.split(" · ")
    label = parts[0] if len(parts) == 1 else f"{parts[0]} +{len(parts) - 1}"
    return label if len(label) <= 20 else label[:19] + "…"


COMPARE_OK = bool(globals().get("ARCH_OK")) and bool(globals().get("SEQ_OK"))
if not COMPARE_OK:
    print("Faltan el mapa Pfam, las secuencias o Biopython: se omiten 5.3 – 5.5 (ejecuta la celda 0.1).")
else:
    aligner = make_aligner()
    DIAGONAL = {a: float(aligner.substitution_matrix[a, a]) for a in aligner.substitution_matrix.alphabet}

    def self_score(seq):
        """Puntuación de una proteína alineada consigo misma: suma de la diagonal de BLOSUM62 (mejor tramo continuo)."""
        total = np.concatenate([[0.0], np.cumsum([DIAGONAL[ch] for ch in seq])])
        return float((total - np.minimum.accumulate(total)).max())

    def similarity(a, b):
        """Similitud entre dos proteínas: S(a, b) / max(S(a, a), S(b, b)). Vale 1 si son idénticas y es baja si las secuencias
        difieren o si una es mucho más corta que la otra."""
        return aligner.score(a, b) / max(self_score(a), self_score(b))

    def protein_families(proteins, min_sim, k=4, n_candidates=3, min_shared=0.10):
        """Agrupa proteínas en familias (agrupamiento voraz, de la más larga a la más corta). Cada proteína se compara con el
        representante de las familias con las que comparte más k-meros (hasta `n_candidates`, y al menos `min_shared` de los suyos)
        y se une a la primera cuya similitud sea ≥ min_sim; si no hay ninguna, funda una familia y es su representante."""
        family = np.full(len(proteins), -1)
        seen, reps, rep_self, index = {}, [], [], {}             # index: k-mero → familias cuyo representante lo contiene
        for i in sorted(range(len(proteins)), key=lambda j: (-len(proteins[j]), j)):
            seq = proteins[i]
            if seq and seq in seen:                              # proteína idéntica a una ya asignada
                family[i] = seen[seq]
                continue
            kmers = {seq[j:j + k] for j in range(len(seq) - k + 1)}
            shared = Counter(f for km in kmers for f in index.get(km, ()))
            own = self_score(seq) if seq else 0.0
            for f, n in sorted(shared.items(), key=lambda kv: (-kv[1], kv[0]))[:n_candidates]:       # orden fijo: mismo resultado en cada corrida
                if n < min_shared * len(kmers) or own <= 0:
                    break
                if aligner.score(seq, reps[f]) / max(own, rep_self[f]) >= min_sim:
                    family[i] = f
                    break
            if family[i] < 0:                                    # familia nueva
                family[i] = len(reps)
                reps.append(seq)
                rep_self.append(own)
                for km in kmers:
                    index.setdefault(km, []).append(family[i])
            seen[seq] = family[i]
        return family

    t0 = time.time()
    architectures = [[gene_architecture(g) for g in genes] for genes in all_arch]         # architectures[BGC][gen]
    seq_lists = [str(s).split("|") if not pd.isna(s) else [] for s in seq_src]
    proteins = [["".join(ch if ch in AA_OK else "X" for ch in seq_lists[i][g].upper()) if g < len(seq_lists[i]) else ""
                 for g in range(len(genes))] for i, genes in enumerate(all_arch)]         # proteins[BGC][gen]
    gene_index = [(i, g) for i, genes in enumerate(all_arch) for g in range(len(genes))]
    raw_family = protein_families([proteins[i][g] for i, g in gene_index], CONFIG["gene_family_min_similarity"])

    # Familias numeradas de mayor a menor (1 = la que tiene más genes) y nombradas por su arquitectura más frecuente
    by_size = sorted(Counter(raw_family.tolist()).items(), key=lambda kv: (-kv[1], kv[0]))
    number = {f: r for r, (f, _) in enumerate(by_size, start=1)}
    gene_fam = [[] for _ in all_arch]                            # gene_fam[BGC][gen] = número de familia
    for (i, g), f in zip(gene_index, raw_family.tolist()):
        gene_fam[i].append(number[f])
    members = {}
    for i, fams in enumerate(gene_fam):
        for g, f in enumerate(fams):
            members.setdefault(f, []).append((i, g))
    NO_DOMAIN = "sin dominios Pfam"
    fam_name = {f: Counter(architectures[i][g] for i, g in genes).most_common(1)[0][0] or NO_DOMAIN for f, genes in members.items()}

    gene_families = save_table(pd.DataFrame([{
        "familia": f, "nombre": fam_name[f], "genes": len(genes), "BGCs": len({i for i, _ in genes}),
        "clases": len({y[i] for i, _ in genes}), "longitud_mediana_aa": int(np.median([all_arch[i][g]["length_aa"] for i, g in genes])),
        "arquitecturas_distintas": len({architectures[i][g] for i, g in genes})} for f, genes in sorted(members.items())]),
        "5.3", "familias_de_genes")
    save_table(pd.DataFrame([{"fila_en_datos": i, "BGC": bgc_label(i), "clase": y[i], "gen": g + 1,
                              "longitud_aa": all_arch[i][g]["length_aa"], "familia": gene_fam[i][g],
                              "arquitectura": architectures[i][g]} for i, g in gene_index]), "5.3", "genes")
    print(f"{len(gene_index):,} genes ({len({proteins[i][g] for i, g in gene_index}):,} proteínas distintas) → {len(members):,} familias de genes "
          f"(similitud ≥ {CONFIG['gene_family_min_similarity']}) · {int((gene_families['BGCs'] >= 2).sum()):,} presentes en 2 o más BGCs · "
          f"{sum(a == '' for archs in architectures for a in archs):,} genes sin dominios en el mapa · {time.time() - t0:.0f} s")

**Tablas de 5.3a y 5.3b.** Con las familias ya definidas, esta celda busca las familias con dos o más genes en un mismo BGC (5.3a) y los dominios que aparecen en dos o más regiones separadas de un mismo gen (5.3b), guarda las tablas y resume los resultados por clase.

In [ ]:
if COMPARE_OK:
    # ---- 5.3a Genes repetidos en el mismo clúster: una familia con 2 o más genes en el BGC ----
    SIM = "similitud_entre_copias"
    rows = []
    for i, fams in enumerate(gene_fam):
        for f, k in Counter(fams).items():
            if k >= 2:
                where = [g for g, x in enumerate(fams) if x == f]
                sims = [similarity(proteins[i][a], proteins[i][b]) for a, b in combinations(where, 2)]
                rows.append({"fila_en_datos": i, "BGC": bgc_label(i), "clase": y[i], "familia": f, "gen": fam_name[f],
                             "copias": k, "posiciones": ", ".join(str(g + 1) for g in where), SIM: round(float(np.median(sims)), 3)})
    repeated_genes = save_table(pd.DataFrame(rows), "5.3", "genes_repetidos")
    gene_summary = (repeated_genes.groupby(["familia", "gen"])
                    .agg(n_BGC=("fila_en_datos", "nunique"), copias_max=("copias", "max"),
                         similitud_mediana=(SIM, "median"))
                    .sort_values("n_BGC", ascending=False, kind="stable").reset_index())
    save_table(gene_summary.round(3), "5.3", "genes_repetidos_resumen")

    # ---- 5.3b Dominios repetidos en el mismo gen ----------------------------------------
    rows, with_runs = [], {}                                     # with_runs[dominio] = BGCs con el dominio repetido si se cuenta cada etiqueta
    for i, genes in enumerate(all_arch):
        for g, gene in enumerate(genes, start=1):
            for name, k in Counter(gene_instances(gene)).items():
                if k >= 2:
                    rows.append({"fila_en_datos": i, "BGC": bgc_label(i), "clase": y[i], "gen": g, "dominio": name,
                                 "copias_en_el_gen": k, "longitud_aa": gene["length_aa"],
                                 "arquitectura_del_gen": architectures[i][g - 1]})
            for name, k in Counter(d[0] for d in gene["domains"]).items():
                if k >= 2:
                    with_runs.setdefault(name, set()).add(i)
    repeated_domains = save_table(pd.DataFrame(rows), "5.3", "dominios_repetidos_en_un_gen")
    domain_summary = (repeated_domains.groupby("dominio")
                      .agg(genes=("gen", "size"), n_BGC=("fila_en_datos", "nunique"),
                           copias_max=("copias_en_el_gen", "max"))
                      .sort_values("n_BGC", ascending=False, kind="stable").reset_index())
    domain_summary["n_BGC_contando_etiquetas_seguidas"] = [len(with_runs[d]) for d in domain_summary["dominio"]]
    save_table(domain_summary, "5.3", "dominios_repetidos_en_un_gen_resumen")

    # ---- Resumen ------------------------------------------------------------------------------------------------------
    n_bgc_rep = repeated_genes["fila_en_datos"].nunique()
    n_bgc_dom = repeated_domains["fila_en_datos"].nunique()
    n_bgc_dom_runs = len(set().union(*with_runs.values()))
    n_same_arch = sum(max(Counter(a for a in archs if a).values(), default=0) >= 2 for archs in architectures)     # misma arquitectura, sin mirar la secuencia
    near = repeated_genes[repeated_genes[SIM] >= 0.9]             # copias casi idénticas
    repeat_by_class = pd.DataFrame({
        "BGCs": [counts[c] for c in class_order],
        "pct_BGC_con_gen_repetido": [
            round(100 * repeated_genes[repeated_genes["clase"] == c]["fila_en_datos"].nunique() / counts[c], 1) for c in class_order],
        "pct_BGC_con_dominio_repetido_en_un_gen": [
            round(100 * repeated_domains[repeated_domains["clase"] == c]["fila_en_datos"].nunique() / counts[c], 1) for c in class_order],
    }, index=pd.Index(class_order, name="clase"))
    save_table(repeat_by_class.reset_index(), "5.3", "repeticiones_por_clase")

    print(f"5.3a · BGCs con algún gen repetido (misma familia): {n_bgc_rep} de {len(y)} · casos (BGC, familia): {len(repeated_genes):,} · "
          f"familias distintas: {repeated_genes['familia'].nunique()}")
    print(f"       similitud entre las copias: mediana {repeated_genes[SIM].median():.2f} · "
          f"casos con copias casi idénticas (≥ 0.9): {len(near)}"
          + (" (" + ", ".join(f"{g}: {k}" for g, k in near["gen"].value_counts().head(4).items()) + ")" if len(near) else ""))
    print(f"       como referencia, BGCs con 2 o más genes de la misma arquitectura de dominios, sin mirar la secuencia: {n_same_arch}")
    print(f"5.3b · BGCs con algún dominio repetido en un mismo gen: {n_bgc_dom} de {len(y)} "
          f"({n_bgc_dom_runs} si se cuenta cada etiqueta)")
    display(repeat_by_class)

**Figuras 5.3a y 5.3b.** Se muestran las `repeat_top_items` familias y dominios repetidos en más BGCs.

- **5.3a:** cada fila es una familia de genes repetida. El número es el porcentaje de los BGCs de la clase que la tienen dos o más veces; a la derecha, cada punto es la similitud entre las copias en un BGC.
- **5.3b:** cada fila es un dominio. El número es el porcentaje de los BGCs de la clase con un gen que lo tiene en dos o más regiones separadas.

In [ ]:
if COMPARE_OK:
    K = CONFIG["repeat_top_items"]

    def pct_by_class(table, key, items):
        """% de los BGCs de cada clase que aparecen en `table` para cada elemento."""
        return np.array([[100 * table[(table[key] == item) & (table["clase"] == c)]["fila_en_datos"].nunique() / counts[c]
                          for c in class_order] for item in items])

    # ---- 5.3a Figura: familias repetidas (izquierda) y similitud entre sus copias (derecha) ----
    show_g = gene_summary.head(K)
    names = [short_arch(n) for n in show_g["gen"]]
    labels = [f"{n}  (familia {f})" if names.count(n) > 1 else n for n, f in zip(names, show_g["familia"])]     # el número distingue familias con el mismo nombre
    M = pct_by_class(repeated_genes, "familia", show_g["familia"])
    fig, (ax, ax_sim) = plt.subplots(1, 2, figsize=(0.9 * len(class_order) + 9.2, 0.42 * len(show_g) + 2.9),
                                     gridspec_kw={"width_ratios": [0.9 * len(class_order), 3.4], "wspace": 0.05})
    heatmap(ax, M, labels, class_order, vmin=0, vmax=100, annot=np.where(M >= 0.5, np.vectorize(lambda v: f"{v:.0f}")(M), ""), annot_size=9, cbar=False)
    ax.tick_params(axis="y", rotation=0, labelsize=9.5)
    rotate_xlabels(ax)
    hbox(ax_sim, [repeated_genes[repeated_genes["familia"] == f][SIM].to_numpy() for f in show_g["familia"]], [""] * len(show_g))
    ax_sim.set_xlim(0, 1)
    ax_sim.set_xticks([0, 0.25, 0.5, 0.75, 1])
    ax_sim.set_xticklabels(["0", "0.25", "0.5", "0.75", "1"])
    ax_sim.tick_params(axis="y", length=0)
    ax_sim.set_xlabel("Similitud de secuencia entre las copias\n(1 = idénticas)")
    titles(ax, "Genes que se repiten dentro de un mismo clúster",
           "Cada fila es una familia de genes · número = % de BGCs de la clase con 2 o más copias · derecha: cada punto es un BGC")
    show_and_save(fig, "5.3", "a_genes_repetidos")

    # ---- 5.3b Figura: dominios repetidos en el mismo gen ----------------------------
    show_d = domain_summary.head(K)
    M = pct_by_class(repeated_domains, "dominio", show_d["dominio"])
    fig, ax = plt.subplots(figsize=(0.95 * len(class_order) + 6.2, 0.4 * len(show_d) + 2.8))
    heatmap(ax, M, [f"{d}  (hasta {k} por gen)" for d, k in zip(show_d["dominio"], show_d["copias_max"])],
            class_order, vmin=0, vmax=100, annot=np.where(M >= 0.5, np.vectorize(lambda v: f"{v:.0f}")(M), ""),
            cbar_label="% de BGCs de la clase", annot_size=9)
    ax.tick_params(axis="y", rotation=0, labelsize=9.5)
    rotate_xlabels(ax)
    titles(ax, "Dominios que se repiten dentro de un mismo gen",
           "Número = % de BGCs de la clase con un gen que tiene el dominio en 2 o más regiones separadas · vacío = menos de 0.5 %")
    show_and_save(fig, "5.3", "b_dominios_repetidos_en_un_gen")
    display(gene_summary.head(10).round(3))

## 5.4 Relación entre sintenias

La sintenia es la conservación del orden de los genes. Para medirla entre dos BGCs se usa un **índice de sintenia**: se recorre cada BGC en el orden de la tabla, se anotan todos los **pares de genes vecinos** (cada gen identificado por su familia, 5.3) y se calcula

> índice = pares de vecinos compartidos / pares de vecinos presentes en alguno de los dos BGCs

Vale 1 si los dos BGCs tienen exactamente los mismos genes vecinos y 0 si no comparten ningún par. Es la fórmula del índice de adyacencia de BiG-SCAPE (Navarro-Muñoz *et al.*, 2020), que allí se aplica a pares de dominios vecinos y aquí a pares de genes vecinos. Los pares no tienen dirección, así que el índice no cambia si un clúster está invertido.

Con el índice se definen dos niveles dentro de cada clase:

- **Locus:** grupo de BGCs conectados por pares de vecinos compartidos (dos BGCs están en el mismo locus si comparten algún par, directamente o a través de otros BGCs). Los BGCs de loci distintos no comparten ningún par de genes vecinos. El locus 1 es el que tiene más BGCs.
- **Tipo de sintenia:** variantes del orden de los genes dentro de un locus. Se obtienen con un agrupamiento jerárquico de enlace promedio: dos grupos de BGCs se unen mientras el índice medio entre ellos sea al menos `synteny_type_threshold`, así que el número de tipos depende de ese umbral. El tipo 1 es el más frecuente de la clase.

**Figuras**

- **5.4a · Matriz de sintenia** entre todos los BGCs, ordenados por clase. Un bloque oscuro en la diagonal es un grupo de BGCs con el mismo orden de genes; una clase con varios bloques separados tiene varios loci. Un bloque oscuro fuera de la diagonal son dos clases que comparten orden.
- **5.4b · Conservación dentro de cada clase:** para cada BGC, su índice medio con los demás BGCs de su clase. Una clase con varios loci tiene un índice medio bajo aunque cada locus esté bien conservado; la tabla de la celda da también el índice medio dentro del mismo locus.

La celda imprime además el índice de los pares de BGCs que se solapan en el genoma (sección 1.3) y en cuántos BGCs la suma de las longitudes de los genes supera la longitud de la región.

In [ ]:
from scipy import sparse
from scipy.sparse.csgraph import connected_components

if COMPARE_OK:
    # ---- Índice de sintenia entre todos los BGCs -----------------------------------------------------
    pair_sets = [{tuple(sorted(p)) for p in zip(f[:-1], f[1:])} for f in gene_fam]       # pares de familias vecinas, sin dirección
    vocab = {p: j for j, p in enumerate(sorted(set().union(*pair_sets)))}
    B = sparse.csr_matrix((np.ones(sum(len(ps) for ps in pair_sets)),
                           ([i for i, ps in enumerate(pair_sets) for _ in ps], [vocab[p] for ps in pair_sets for p in ps])),
                          shape=(len(pair_sets), len(vocab)))                 # BGCs × pares: 1 si el BGC tiene ese par
    shared_pairs = (B @ B.T).toarray()
    n_pairs = shared_pairs.diagonal()
    union_pairs = n_pairs[:, None] + n_pairs[None, :] - shared_pairs
    SYN = np.divide(shared_pairs, union_pairs, out=np.zeros_like(shared_pairs), where=union_pairs > 0)

    # ---- Loci y tipos de sintenia dentro de cada clase ---------------------------------------
    THR = CONFIG["synteny_type_threshold"]
    locus, syn_type, syn_pos, type_rows = np.zeros(len(y), int), np.zeros(len(y), int), np.zeros(len(y), int), []
    for c in class_order:
        idx = np.where(y == c)[0]
        sub = SYN[np.ix_(idx, idx)]
        comp = connected_components(sparse.csr_matrix(sub > 0), directed=False)[1]     # locus = BGCs conectados por pares compartidos
        order = {k: r for r, (k, _) in enumerate(sorted(Counter(comp.tolist()).items(), key=lambda kv: (-kv[1], kv[0])), start=1)}
        locus[idx] = [order[k] for k in comp.tolist()]
        D = 1 - sub
        np.fill_diagonal(D, 0)
        Z = linkage(squareform(D, checks=False), "average")
        lab = fcluster(Z, 1 - THR, "distance")                   # se unen grupos mientras su índice medio sea ≥ THR
        by_size = sorted(Counter(lab.tolist()).items(), key=lambda kv: (-kv[1], kv[0]))  # tipo 1 = el más frecuente
        rank = {t: r for r, (t, _) in enumerate(by_size, start=1)}
        syn_type[idx] = [rank[t] for t in lab.tolist()]
        syn_pos[idx[leaves_list(Z)]] = np.arange(len(idx))                             # orden del árbol: vecinos parecidos
        for t, n in by_size:
            in_type = idx[lab == t]
            rep = in_type[np.argmax(SYN[np.ix_(in_type, in_type)].mean(axis=1))]        # BGC más parecido al resto de su tipo
            type_rows.append({"clase": c, "locus": locus[rep], "tipo": rank[t], "n_BGC": n, "pct_de_la_clase": round(100 * n / len(idx), 1),
                              "representante": bgc_label(rep), "fila_representante": rep})
    synteny_types = save_table(pd.DataFrame(type_rows), "5.4", "tipos_de_sintenia")

    same_class = y[:, None] == y[None, :]
    same_locus = same_class & (locus[:, None] == locus[None, :])
    not_self = ~np.eye(len(y), dtype=bool)
    with np.errstate(invalid="ignore", divide="ignore"):         # un BGC solo en su locus no tiene media dentro del locus
        mean_in_class = (SYN * (same_class & not_self)).sum(axis=1) / (same_class & not_self).sum(axis=1)
        mean_in_locus = (SYN * (same_locus & not_self)).sum(axis=1) / (same_locus & not_self).sum(axis=1)
    save_table(pd.DataFrame({"fila_en_datos": np.arange(len(y)), "BGC": [bgc_label(i) for i in range(len(y))], "clase": y, "locus": locus,
                             "tipo_de_sintenia": syn_type, "genes": [len(f) for f in gene_fam],
                             "indice_medio_con_su_clase": mean_in_class.round(3),
                             "indice_medio_con_su_locus": mean_in_locus.round(3)}), "5.4", "b_locus_y_tipo_de_cada_bgc")

    class_mean_syn = pd.DataFrame(index=class_order, columns=class_order, dtype=float)
    for a in class_order:
        for b in class_order:
            block = SYN[np.ix_(y == a, y == b)]
            class_mean_syn.loc[a, b] = block[~np.eye(len(block), dtype=bool)].mean() if a == b else block.mean()
    save_table(class_mean_syn.round(3).rename_axis("clase").reset_index(), "5.4", "sintenia_media_entre_clases")

    def pair_mean(mask):
        """Índice medio de los pares de BGCs marcados (NaN si no hay ninguno)."""
        return round(float(SYN[mask].mean()), 3) if mask.any() else np.nan

    n_loci = pd.Series({c: int(locus[y == c].max()) for c in class_order})
    n_types = pd.Series({c: int(syn_type[y == c].max()) for c in class_order})
    class_summary = pd.DataFrame({
        "BGCs": [counts[c] for c in class_order],
        "loci": n_loci.to_numpy(),
        "BGCs_en_el_locus_1": [int(((y == c) & (locus == 1)).sum()) for c in class_order],
        "tipos_de_sintenia": n_types.to_numpy(),
        "BGCs_en_el_tipo_1": [int(((y == c) & (syn_type == 1)).sum()) for c in class_order],
        "indice_medio_en_la_clase": [pair_mean(same_class & not_self & (y == c)[:, None]) for c in class_order],
        "indice_medio_en_el_mismo_locus": [pair_mean(same_locus & not_self & (y == c)[:, None]) for c in class_order],
    }, index=pd.Index(class_order, name="clase"))
    save_table(class_summary.reset_index(), "5.4", "resumen_por_clase")

    # ---- 5.4a Matriz de sintenia, ordenada por clase → locus → tipo → árbol ---
    class_rank = np.array([class_order.index(c) for c in y])
    syn_rows = np.lexsort((syn_pos, syn_type, locus, class_rank))
    matrix_table = pd.DataFrame(SYN[np.ix_(syn_rows, syn_rows)].round(3), columns=[f"fila_{i}" for i in syn_rows])
    matrix_table.insert(0, "clase", y[syn_rows])
    matrix_table.insert(0, "fila_en_datos", syn_rows)
    save_table(matrix_table, "5.4", "a_matriz_sintenia")
    bounds = np.cumsum([0] + [int((y == c).sum()) for c in class_order])
    centres = (bounds[:-1] + bounds[1:]) / 2 - 0.5
    fig, ax = plt.subplots(figsize=(9.8, 9.2))
    im = ax.imshow(SYN[np.ix_(syn_rows, syn_rows)], cmap=SEQ_CMAP, vmin=0, vmax=1, interpolation="nearest")
    for b in bounds[1:-1]:
        ax.axhline(b - 0.5, color=INK, lw=0.7)
        ax.axvline(b - 0.5, color=INK, lw=0.7)
    ax.set_xticks(centres)
    ax.set_xticklabels(class_order, rotation=90)
    ax.set_yticks(centres)
    ax.set_yticklabels([f"{c} ({counts[c]})" for c in class_order])
    ax.tick_params(length=0)
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.7, label="Índice de sintenia (1 = mismos genes vecinos)")
    titles(ax, "Sintenia entre todos los BGCs",
           f"{len(y)} BGCs ordenados por clase · oscuro = orden de genes parecido · cada bloque de la diagonal es una clase")
    show_and_save(fig, "5.4", "a_matriz_sintenia")

    # ---- 5.4b Conservación de la sintenia dentro de cada clase -----------------------------------
    fig, ax = plt.subplots(figsize=(10.5, 0.5 * len(class_order) + 2.0))
    hbox(ax, [mean_in_class[y == c] for c in class_order], [f"{c} ({counts[c]})" for c in class_order])
    for r, c in enumerate(class_order):
        ax.text(1.03, r, f"{n_loci[c]} {'locus' if n_loci[c] == 1 else 'loci'} · {n_types[c]} tipos", va="center", ha="left", fontsize=9.5, color=INK2)
    ax.set_xlim(0, 1.0)
    ax.set_xlabel("Índice de sintenia medio con los demás BGCs de su clase")
    titles(ax, "Qué tan parecido es el orden de los genes dentro de cada clase",
           f"Cada punto es un BGC · a la derecha, loci y tipos de sintenia de la clase (tipos: índice medio ≥ {THR})")
    show_and_save(fig, "5.4", "b_sintenia_por_clase")

    syn_within, syn_between = pair_mean(same_class & not_self), pair_mean(~same_class)
    off = class_mean_syn.where(~np.eye(len(class_order), dtype=bool)).stack()
    print(f"Índice de sintenia medio entre BGCs de la misma clase: {syn_within:.3f} · de clases distintas: {syn_between:.3f}")
    print(f"Par de clases distintas con más sintenia: {off.idxmax()[0]} – {off.idxmax()[1]} ({off.max():.2f})")
    if len(overlaps):                                             # pares de filas que describen tramos solapados del mismo genoma (1.3)
        ov_syn = overlaps.assign(syn=[SYN[a, b] for a, b in zip(overlaps["fila_a"], overlaps["fila_b"])])
        print("Pares de BGCs solapados en el genoma (1.3), índice de sintenia mediano · "
              + " · ".join(f"{pair}: {g['syn'].median():.2f} (n = {len(g)})" for pair, g in ov_syn.groupby("par")))
    if "length_kb" in data.columns:                               # ¿la lista de genes cabe en la región?
        ratio = np.array([sum(g["length_aa"] for g in genes) * 3e-3 for genes in all_arch]) / data["length_kb"].to_numpy(float)
        print(f"La suma de las longitudes de los genes supera la longitud de la región (length_kb) en "
              f"{int((ratio > 1).sum())} de {len(ratio)} BGCs (mediana del cociente: {np.median(ratio):.2f})")
    display(class_summary)

## 5.5 Todos los clústeres alineados en orden

Los BGCs de loci distintos no tienen genes vecinos en común, así que no se pueden alinear entre sí. Por eso cada **locus** se alinea por separado, con un alineamiento progresivo del orden de los genes:

1. Cada BGC se escribe como la lista de las familias de sus genes (5.3), en el orden de la tabla.
2. Se parte del BGC más parecido a los demás de su locus (índice de 5.4) y en cada paso se añade el BGC más parecido a alguno de los ya alineados. Sus genes se colocan en las columnas que ya contienen un gen de la misma familia, respetando el orden. Donde al BGC le falta un gen queda un hueco; donde tiene genes de más, se abren columnas nuevas.
3. Cada BGC se prueba en sus dos orientaciones y se queda la que da más coincidencias con los BGCs ya alineados, porque la tabla no guarda la hebra.
4. Las filas se ordenan por tipo de sintenia (5.4) y, dentro de cada tipo, por parecido.

Se dibuja una figura por locus. Los loci con menos de `align_min_bgcs_shown` BGCs se guardan en la carpeta de resultados pero no se muestran aquí, y un locus con un solo BGC no tiene nada que alinear.

**Cómo leer cada figura**

- **A · Representantes.** El BGC representativo de cada tipo de sintenia principal del locus. Cada gen es una caja proporcional a la longitud de su proteína, en el mismo orden que en B; la separación entre genes es fija porque la tabla no trae coordenadas por gen. Cada representante se desplaza para que su primer gen en común con el primero quede en la misma posición.
- **B · Todos los BGCs del locus.** Cada fila es un BGC y cada casilla un gen. Los genes de la misma familia quedan en la misma columna y una casilla vacía es un hueco. Un gen sin pareja se coloca en la columna siguiente a la de su vecino, así que una columna puede contener también genes de otra familia (se ven en tono claro). Las etiquetas de arriba nombran la familia más frecuente de las columnas que comparten al menos `align_label_min_fraction` de los BGCs (primer dominio del gen y, si tiene más, cuántos); cuando no caben todas, se nombran las más conservadas.
- **Color (A y B):** fracción de los BGCs del locus que tienen un gen de esa misma familia en esa misma columna (más oscuro = posición más conservada).

**Limitaciones:** el resultado depende del orden en que se añaden los BGCs (dos BGCs con el mismo orden de genes pueden quedar en columnas ligeramente distintas) y del umbral de similitud que define las familias.

Las tablas `5.5_alineamiento_genes` (columna de cada gen), `5.5_alineamiento_bgcs` (orientación de cada BGC) y `5.5_columnas` (familia más frecuente de cada columna) guardan el alineamiento completo. Al final se muestra un resumen por locus:

| Columna del resumen | Qué mide |
|---|---|
| `columnas_50pct`, `columnas_90pct` | Columnas en las que al menos el 50 % (o el 90 %) de los BGCs del locus tiene un gen de la misma familia |
| `pct_genes_en_su_columna` | Porcentaje de los genes que quedan en una columna cuya familia más frecuente es la suya; cuanto más alto, más limpio es el alineamiento |

In [ ]:
ALIGN_GAP_KB = 0.12                                              # separación dibujada entre genes (kb)
# Misma rampa azul sin su extremo blanco: un gen poco conservado sigue viéndose como una casilla
CONS_CMAP = LinearSegmentedColormap.from_list("cons_blue", [SEQ_CMAP(v) for v in np.linspace(0.14, 1, 64)])


def match_to_profile(seq, profile):
    """Empareja los genes de un BGC con las columnas que ya contienen un gen de su familia, respetando el orden (subsecuencia
    común de mayor peso; el peso de una columna es el número de BGCs que ya tienen ahí esa familia). Devuelve (puntuación,
    [(gen, columna), …])."""
    m, w = len(seq), len(profile)
    weight = np.zeros((m, w))
    positions = {}
    for g, f in enumerate(seq):
        positions.setdefault(f, []).append(g)
    for col, genes in enumerate(profile):
        for f, k in genes.items():
            for g in positions.get(f, ()):
                weight[g, col] = k
    best = np.zeros((m + 1, w + 1))                               # best[g, col]: mejor puntuación con los primeros g genes y col columnas
    for g in range(m):
        best[g + 1, 1:] = np.maximum.accumulate(np.maximum(best[g, 1:], best[g, :-1] + weight[g]))
    pairs, g, col = [], m, w
    while g > 0 and col > 0:                                      # se recorre hacia atrás para recuperar los emparejamientos
        if weight[g - 1, col - 1] > 0 and best[g, col] == best[g - 1, col - 1] + weight[g - 1, col - 1]:
            pairs.append((g - 1, col - 1))
            g, col = g - 1, col - 1
        elif best[g, col] == best[g - 1, col]:
            g -= 1
        else:
            col -= 1
    return best[m, w], pairs[::-1]


def align_locus(idx):
    """Alineamiento progresivo de los BGCs de un locus. Devuelve {BGC: columna de cada gen, en el orden en que se dibuja},
    {BGC: invertido} y el perfil [Counter(familias) de cada columna]."""
    sub = SYN[np.ix_(idx, idx)]
    first = int(np.argmax(sub.mean(axis=1)))                     # se empieza por el más parecido a los demás
    order, nearest, left = [first], sub[first].copy(), set(range(len(idx))) - {first}
    while left:                                                  # después, siempre el más parecido a alguno de los ya alineados
        nxt = max(left, key=lambda j: (nearest[j], -j))
        order.append(nxt)
        left.discard(nxt)
        nearest = np.maximum(nearest, sub[nxt])
    profile, columns, flipped = [], {}, {}

    def open_columns(at, count):
        """Abre `count` columnas vacías en la posición `at`: los BGCs ya colocados quedan con un hueco ahí."""
        if count > 0:
            profile[at:at] = [Counter() for _ in range(count)]
            for k, cols in columns.items():
                columns[k] = [col + count if col >= at else col for col in cols]

    for i in idx[order]:
        options = [(*match_to_profile(gene_fam[i][::o], profile), o) for o in (1, -1)]      # las dos orientaciones
        score, pairs, o = max(options, key=lambda t: (t[0], t[2]))                       # en empate, sin invertir
        seq = gene_fam[i][::o]
        if not pairs:                                                    # primer BGC (o nada en común con los anteriores): columnas nuevas al final
            cols = list(range(len(profile), len(profile) + len(seq)))
            open_columns(len(profile), len(seq))
        else:
            cols = [0] * len(seq)
            g0, c0 = pairs[0]
            shift = max(0, g0 - c0)                                      # columnas abiertas hasta ahora para este BGC
            open_columns(0, shift)
            cols[:g0 + 1] = range(c0 + shift - g0, c0 + shift + 1)       # genes anteriores al primer emparejamiento, pegados a él
            for (ga, ca), (gb, cb) in zip(pairs[:-1], pairs[1:]):
                extra = (gb - ga) - (cb - ca)                            # genes de más entre dos emparejamientos
                open_columns(cb + shift, extra)
                shift += max(0, extra)
                cols[ga + 1:gb] = range(cols[ga] + 1, cols[ga] + gb - ga)
                cols[gb] = cb + shift
            g_last = pairs[-1][0]
            cols[g_last + 1:] = range(cols[g_last] + 1, cols[g_last] + len(seq) - g_last)
            open_columns(len(profile), cols[-1] + 1 - len(profile))
        columns[i], flipped[i] = cols, o == -1
        for f, col in zip(seq, cols):
            profile[col][f] += 1
    return columns, flipped, profile


def aligned_figure(c, k_locus, idx):
    """Alinea un locus, dibuja su figura y devuelve sus tablas."""
    n = len(idx)
    columns, flipped, profile = align_locus(idx)
    width = len(profile)
    rows = idx[np.lexsort((syn_pos[idx], syn_type[idx]))]                                # tipo → orden del árbol
    drawn = {i: (all_arch[i][::-1] if flipped[i] else all_arch[i]) for i in idx}         # genes en el orden en que se dibujan
    fams = {i: (gene_fam[i][::-1] if flipped[i] else gene_fam[i]) for i in idx}
    share = {i: [profile[col][f] / n for col, f in zip(columns[i], fams[i])] for i in idx}     # fracción del locus con esa familia en esa columna
    cons = np.full((n, width), np.nan)
    key = {"clase": c, "locus": k_locus}
    gene_rows = []
    for r, i in enumerate(rows):
        numbers = range(len(drawn[i]), 0, -1) if flipped[i] else range(1, len(drawn[i]) + 1)
        for col, g, f, v in zip(columns[i], numbers, fams[i], share[i]):
            cons[r, col] = v
            gene_rows.append({**key, "fila_en_datos": i, "BGC": bgc_label(i), "tipo_de_sintenia": syn_type[i], "gen": g,
                              "columna": col + 1, "familia": f, "nombre": fam_name[f],
                              "pct_BGCs_con_esa_familia_en_esa_columna": round(100 * v, 1)})
    bgc_rows = [{**key, "fila_en_datos": i, "BGC": bgc_label(i), "tipo_de_sintenia": syn_type[i],
                 "invertido": flipped[i], "genes": len(drawn[i]), "primera_columna": columns[i][0] + 1,
                 "ultima_columna": columns[i][-1] + 1} for i in rows]
    occupied = Counter(col for i in idx for col in columns[i])
    top = [genes.most_common(1)[0] for genes in profile]                                 # familia más frecuente de cada columna y cuántos BGCs la tienen ahí
    col_rows = [{**key, "columna": col + 1, "familia": f, "nombre": fam_name[f], "n_BGC_con_esa_familia": k,
                 "pct_del_locus": round(100 * k / n, 1), "n_BGC_con_algun_gen": occupied[col]} for col, (f, k) in enumerate(top)]
    matched = [top[col][0] == f for i in idx for col, f in zip(columns[i], fams[i])]
    stats = {**key, "BGCs": n, "tipos_de_sintenia": len(set(syn_type[idx].tolist())), "BGCs_invertidos": sum(flipped.values()),
             "columnas_50pct": sum(k >= 0.5 * n for _, k in top), "columnas_90pct": sum(k >= 0.9 * n for _, k in top),
             "pct_genes_en_su_columna": round(100 * float(np.mean(matched)), 1)}
    if n < 2:                                                                            # un solo BGC: nada que alinear ni que dibujar
        return bgc_rows, gene_rows, col_rows, stats

    # tipos de sintenia del locus, en el orden de las filas: los que tienen pocos BGCs no llevan etiqueta
    sizes = Counter(syn_type[idx].tolist())
    big = [t for t in sorted(sizes) if sizes[t] >= max(3, 0.04 * n)] or [min(sizes, key=lambda t: (-sizes[t], t))]
    blocks, start = [], 0                                                                # [etiqueta o None, primera fila, última fila + 1]
    for t in sorted(sizes):
        if t in big:
            blocks.append([f"tipo {t} · n = {sizes[t]}", start, start + sizes[t]])
        elif blocks and blocks[-1][0] is None:                                           # tipos pequeños seguidos: un solo bloque
            blocks[-1][2] = start + sizes[t]
        else:
            blocks.append([None, start, start + sizes[t]])
        start += sizes[t]
    blocks = [(label or f"otros tipos · n = {b - a}", a, b) for label, a, b in blocks]
    rep_of = synteny_types[synteny_types["clase"] == c].set_index("tipo")["fila_representante"]
    reps = [int(rep_of[t]) for t in big[:CONFIG["align_representatives"]]]

    # ---- Tamaños en pulgadas: la figura crece con el número de columnas y de BGCs ----
    mat_w = float(np.clip(0.2 * width, 7.0, 16.0))
    col_w = mat_w / width
    # etiquetas de arriba: inclinadas si las columnas son anchas, verticales si son estrechas; si no caben todas, primero las más conservadas
    rotation = 55 if col_w >= 0.195 else 90
    step = int(np.ceil((1.15 * 8.5 / 72) / (col_w * np.sin(np.radians(rotation)))))      # columnas entre dos etiquetas para que no se encimen
    shown = []
    for k, col, f in sorted(((k, col, f) for col, (f, k) in enumerate(top)), key=lambda t: (-t[0], t[1])):
        if k >= CONFIG["align_label_min_fraction"] * n and all(abs(col - other) >= step for other, _ in shown):
            shown.append((col, gene_label(fam_name[f])))
    shown.sort()
    label_h = 0.62 * 8.5 / 72 * max([len(t) for _, t in shown] or [0]) * np.sin(np.radians(rotation)) + 0.2      # alto de las etiquetas
    left, right, top_m, bottom = 2.5, 1.9, 1.05, 0.25
    hA, hB = 0.52 * len(reps) + 0.45, max(0.05 * n, min(2.8, 0.25 * n), 1.8)       # alto mínimo: cabe la barra de color
    bar_h, gap = min(2.0, hB), label_h + 0.75
    fig_w, fig_h = left + mat_w + right, top_m + hA + gap + hB + bottom
    fig = plt.figure(figsize=(fig_w, fig_h))
    axA = fig.add_axes([left / fig_w, (bottom + hB + gap) / fig_h, mat_w / fig_w, hA / fig_h])
    axB = fig.add_axes([left / fig_w, bottom / fig_h, mat_w / fig_w, hB / fig_h])
    cax = fig.add_axes([(left + mat_w + 0.3) / fig_w, (bottom + hB - bar_h) / fig_h, 0.16 / fig_w, bar_h / fig_h])

    # ---- A representantes: cajas proporcionales a cada proteína; se desplazan para que coincida su primer gen en común con el primero ----
    def gene_starts(i):
        """{columna: (inicio en kb, familia)} de cada gen del BGC."""
        x, out = 0.0, {}
        for col, gene, f in zip(columns[i], drawn[i], fams[i]):
            out[col] = (x, f)
            x += gene["length_aa"] * 3e-3 + ALIGN_GAP_KB
        return out

    ref_x = gene_starts(reps[0])
    xmin, xmax = 0.0, 1.0
    for r, i in enumerate(reps):
        xs = gene_starts(i)
        same = [col for col in sorted(xs) if col in ref_x and xs[col][1] == ref_x[col][1]]
        shift = ref_x[same[0]][0] - xs[same[0]][0] if same else 0.0
        for col, gene, v in zip(columns[i], drawn[i], share[i]):
            x, length = xs[col][0] + shift, gene["length_aa"] * 3e-3
            axA.add_patch(FancyBboxPatch((x, -r - 0.3), length, 0.6, boxstyle="round,pad=0,rounding_size=0.06", mutation_aspect=1,
                                         facecolor=CONS_CMAP(v), edgecolor="none"))
            xmin, xmax = min(xmin, x), max(xmax, x + length)
        x0, x1 = min(v[0] for v in xs.values()) + shift, max(v[0] for v in xs.values()) + shift
        axA.plot([x0, x1], [-r, -r], color=AXIS, lw=1, zorder=0)                         # línea del clúster
        axA.text(-0.012, -r, f"tipo {syn_type[i]} · n = {sizes[syn_type[i]]}" + (" · invertido" if flipped[i] else "") + f"\n{bgc_label(i)}",
                 transform=axA.get_yaxis_transform(), ha="right", va="center", fontsize=8.5, color=INK, linespacing=1.2)
    bar = 5 if xmax - xmin > 40 else 2 if xmax - xmin > 12 else 1                         # barra de escala
    axA.plot([xmax - bar, xmax], [-len(reps) + 0.3] * 2, color=INK, lw=1.4)
    axA.text(xmax - bar / 2, -len(reps) + 0.2, f"{bar} kb", ha="center", va="top", fontsize=8.5, color=INK2)
    axA.set_xlim(xmin - 0.01 * (xmax - xmin), xmax + 0.01 * (xmax - xmin))
    axA.set_ylim(-len(reps) - 0.05, 0.6)
    axA.axis("off")

    # ---- B todos los BGCs del locus: una fila por BGC, una casilla por gen ------------
    im = axB.imshow(cons, cmap=CONS_CMAP, vmin=0, vmax=1, aspect="auto", interpolation="nearest", extent=(-0.5, width - 0.5, n - 0.5, -0.5))
    for x in np.arange(0.5, width - 0.5, 1):
        axB.axvline(x, color=SURFACE, lw=0.7)
    for label, a, b in blocks:
        if a > 0:
            axB.axhline(a - 0.5, color=INK, lw=1)
        axB.text(-0.012, (a + b) / 2 - 0.5, label, transform=axB.get_yaxis_transform(), ha="right", va="center", fontsize=9, color=INK)
    axB.set_xticks([col for col, _ in shown])
    axB.set_xticklabels([t for _, t in shown], rotation=rotation, fontsize=8.5, **({"ha": "left", "rotation_mode": "anchor"} if rotation < 90 else {"ha": "center"}))
    axB.xaxis.tick_top()
    axB.tick_params(axis="x", length=0, pad=2)
    axB.set_yticks([])
    axB.set_xlim(-0.5, width - 0.5)
    axB.set_ylim(n - 0.5, -0.5)
    axB.grid(False)
    for spine in axB.spines.values():
        spine.set_visible(False)
    cb = fig.colorbar(im, cax=cax)
    cb.set_label("Fracción de BGCs con esa\nfamilia en esa columna", fontsize=9)
    cb.outline.set_visible(False)

    several = int(n_loci[c]) > 1
    fig.text(0.1 / fig_w, 1 - 0.12 / fig_h, f"{c}" + (f" · locus {k_locus}" if several else "") + f": {n} BGCs alineados en orden"
             + (f" (de los {counts[c]} de la clase)" if several else ""), ha="left", va="top", fontsize=13, fontweight="bold", color=INK)
    fig.text(0.1 / fig_w, 1 - 0.43 / fig_h, f"Cada casilla es un gen · los genes de la misma familia quedan en la misma columna · más oscuro = posición más conservada · vacío = hueco · "
             + ("1 BGC invertido" if sum(flipped.values()) == 1 else f"{sum(flipped.values())} BGCs invertidos"),
             ha="left", va="top", fontsize=10, color=INK2)
    fig.text(0.1 / fig_w, (bottom + hB + gap + hA + 0.08) / fig_h, "A · Representante de cada tipo de sintenia principal (tamaño de los genes a escala)",
             ha="left", va="bottom", fontsize=11, fontweight="bold", color=INK)
    fig.text(0.1 / fig_w, (bottom + hB + label_h + 0.22) / fig_h, f"B · Los {n} BGCs (arriba: familia de las columnas que comparte al menos el "
             f"{100 * CONFIG['align_label_min_fraction']:.0f} % de los BGCs)", ha="left", va="bottom", fontsize=11, fontweight="bold", color=INK)
    show_and_save(fig, "5.5", f"alineamiento_{slug(c)}_locus{k_locus}", individual=n < CONFIG["align_min_bgcs_shown"])
    return bgc_rows, gene_rows, col_rows, stats


if COMPARE_OK:
    bgc_rows, gene_rows, col_rows, locus_rows = [], [], [], []
    for c in class_order:
        for k_locus in range(1, int(n_loci[c]) + 1):
            b, g, k, s = aligned_figure(c, k_locus, np.where((y == c) & (locus == k_locus))[0])
            bgc_rows += b
            gene_rows += g
            col_rows += k
            locus_rows.append(s)
    save_table(pd.DataFrame(bgc_rows), "5.5", "alineamiento_bgcs")
    save_table(pd.DataFrame(gene_rows), "5.5", "alineamiento_genes")
    save_table(pd.DataFrame(col_rows), "5.5", "columnas")
    locus_summary = save_table(pd.DataFrame(locus_rows), "5.5", "resumen_por_locus")
    display(locus_summary[locus_summary["BGCs"] >= CONFIG["align_min_bgcs_shown"]].set_index(["clase", "locus"]))

---
# Anexo · Modelo exportado, resumen y archivos generados

| Sección | Qué hace | Salidas |
|---|---|---|
| A.1 | Guarda el modelo final y muestra cómo usarlo con BGCs nuevos | `A.1_modelo_svm.joblib` |
| A.2 | Reúne en una tabla las cifras clave de todo el notebook | `A.2_resumen_resultados.csv` |
| A.3 | Lista todos los archivos generados, por carpeta | `A.3_archivos_generados.csv` |

## A.1 Modelo exportado y ejemplo de uso

El archivo `joblib` contiene el pipeline ajustado (escalado + SVM), la lista ordenada de columnas Pfam que espera, las clases, la configuración y las versiones de las librerías. **No hay que escalar los datos nuevos**: el escalado va dentro del pipeline. Las columnas Pfam que falten en una tabla nueva se rellenan con 0 (dominio ausente).

> `joblib` ejecuta código al cargar un archivo: abre solo modelos de fuentes confiables.

In [ ]:
import platform
import joblib
import sklearn

bundle = {
    "model": final_model,
    "feature_names": selected_features,          # columnas del CSV que espera, en este orden
    "classes": list(final_model.classes_),
    "target_column": TARGET,
    "k_domains": k_optimal,
    "C": C_final,
    "metrics": {"cv_accuracy_train": round(cv_final, 4), "test_accuracy": float(metrics_test.at[0, "accuracy"]),
                "permutation_p": float(p_perm)},
    "config": {k: (list(v) if isinstance(v, (tuple, np.ndarray)) else v) for k, v in CONFIG.items()},
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                 "scikit-learn": sklearn.__version__},
}
model_path = out_path("A.1", "modelo_svm", "joblib")
joblib.dump(bundle, model_path, compress=3)
print(f"Modelo guardado: {model_path} ({model_path.stat().st_size / 1024:.0f} kB)")

# ---- Ejemplo de uso: aquí se usan las 5 primeras filas del CSV como si fueran BGCs nuevos --
# (es solo una demostración del formato: esas filas pueden haber estado en el entrenamiento)
loaded = joblib.load(model_path)
new_bgcs = pd.read_csv(CONFIG["input_csv"], encoding="utf-8-sig", low_memory=False).head(5)
X_new = new_bgcs.reindex(columns=loaded["feature_names"]).fillna(0.0).to_numpy(float)     # mismo orden de columnas; faltantes = 0
pd.DataFrame({"clase_en_el_csv": new_bgcs[TARGET], "clase_predicha": loaded["model"].predict(X_new),
              "probabilidad": loaded["model"].predict_proba(X_new).max(axis=1).round(3)})

## A.2 Resumen de resultados

Las cifras clave de cada parte, con la sección donde se calculan. Esta tabla es la fuente de los números que se citen en un resumen o póster.

In [ ]:
G = globals()
rows = [
    ("1.2", "BGCs analizados (tras unir GenBank/RefSeq y excluir clases raras)", len(y)),
    ("1.2", "Genomas", data["genome"].nunique()),
    ("1.3", "Clases retenidas", f"{len(class_order)}: " + ", ".join(f"{c} ({int((y == c).sum())})" for c in class_order)),
    ("1.3", "Pares de BGCs del mismo genoma que se solapan", len(overlaps)),
    ("1.4", "Dominios Pfam en la matriz", X.shape[1]),
    ("1.4", "Perfiles de presencia/ausencia distintos", n_profiles),
    ("2.1", "BGCs de entrenamiento / prueba", f"{len(idx_train)} / {len(idx_test)}"),
    ("2.2", "Dominios usados por el modelo (k)", k_optimal),
    ("2.3", "C elegido", C_final),
    ("2.5", "Prueba de permutación: p", round(float(p_perm), 4)),
    ("2.6", "Exactitud por validación cruzada (entrenamiento)", f"{cv_final:.3f} ± {cv_final_se:.3f}"),
    ("3.1", "Exactitud en la prueba apartada (IC 95 %)",
     f"{metrics_test.at[0, 'accuracy']:.3f} ({ci_low:.3f}–{ci_high:.3f}) · {n_correct}/{len(y_test)}"),
    ("3.1", "Exactitud balanceada / F1 macro / MCC (prueba)",
     f"{metrics_test.at[0, 'balanced_accuracy']:.3f} / {metrics_test.at[0, 'f1_macro']:.3f} / {metrics_test.at[0, 'mcc']:.3f}"),
    ("3.1", "Brecha entrenamiento − prueba", round(gap, 3)),
    ("3.1", "Exactitud en BGCs de prueba con / sin perfil idéntico en el entrenamiento",
     f"{acc_twin:.3f} (n = {int(has_twin.sum())}) / {acc_new:.3f} (n = {int((~has_twin).sum())})"),
    ("3.4", "Errores en la prueba (de ellos, en BGCs solapados)", f"{int(wrong.sum())} ({n_err_overlap})"),
    ("3.4", "Errores (real → predicha)", "; ".join(f"{t} → {p}" for t, p in zip(y_test[wrong], y_test_pred[wrong])) or "—"),
    ("4.1", "Grupos (tamaño)", ", ".join(f"{g} ({group_sizes[g]})" for g in group_names)),
    ("4.1", "Silueta con el k usado / k con la silueta más alta",
     f"{sil.set_index('k').at[k_groups, 'silhouette']:.3f} / {best_k}"),
]
for g in group_names:
    comp = composition.loc[g]
    rows.append(("4.1", f"Composición de {g}", ", ".join(f"{c} {n}" for c, n in comp[comp > 0].sort_values(ascending=False).items())))
for g in group_names:
    top3 = markers[markers["grupo"] == g].head(3)
    rows.append(("4.2", f"Marcadores de {g}", "; ".join(
        f"{r['dominio']} {r['direccion']} ({100 * r['prevalencia_grupo']:.0f} % vs {100 * r['prevalencia_resto']:.0f} %)"
        for _, r in top3.iterrows())))
rows.append(("4.2", "Árbol de decisión: exactitud balanceada (validación cruzada)", f"{tree_cv.mean():.3f}"))
if G.get("ARCH_OK"):
    for g, i in reps:
        rows.append(("4.3", f"Representante de {g}", f"{y[i]} · {bgc_label(i)}"))
    for c in G.get("show_classes", []):
        v = per_bgc.loc[y == c, "módulos"]
        rows.append(("4.5", f"Módulos NRPS por BGC (mediana, máx.) · {c}", f"{v.median():g}, {v.max():g}"))
if G.get("SEQ_OK"):
    rows.append(("4.6", "Distancia de alineamiento media dentro / entre clases", f"{intra:.3f} / {inter:.3f}"))
    rows.append(("4.6", "Mantel: secuencia frente a dominios del modelo", f"r = {r_obs:.3f}, p = {p_mantel:.4f}"))

# ---- Parte 5 · Comparación de clústeres ----
rows.append(("5.2", "Dominios frecuentes (≥ 50 % de los BGCs) en 2 o más clases", len(shared)))
rows.append(("5.2", "Dominios frecuentes en más clases (número de clases)",
             f"{', '.join(widest['dominio'].head(6))} ({recurrence[FREQ].max()})"))
rows.append(("5.2", "Dominios en más BGCs",
             "; ".join(f"{r['dominio']} ({r['n_BGC']} BGCs, {r['pct_BGC']:g} %)" for _, r in recurrence.head(3).iterrows())))
if G.get("ARCH_OK"):
    rows.append(("4.3", "Presencias de dominio de la tabla que aparecen en el mapa",
                 f"{int((P & in_map).sum()):,} / {int(P.sum()):,} ({100 * map_coverage:.1f} %)"))
if G.get("COMPARE_OK"):
    rows.append(("5.3", f"Familias de genes (similitud ≥ {CONFIG['gene_family_min_similarity']})", f"{len(members):,} ({len(gene_index):,} genes)"))
    rows.append(("5.3", "BGCs con algún gen repetido en el mismo clúster (misma familia)", f"{n_bgc_rep} / {len(y)}"))
    rows.append(("5.3", "BGCs con 2 o más genes de la misma arquitectura de dominios, sin mirar la secuencia",
                 f"{n_same_arch} / {len(y)}"))
    rows.append(("5.3", "Genes repetidos en más BGCs (BGCs; similitud mediana entre copias)",
                 "; ".join(f"{r['gen']} ({r['n_BGC']}; {r['similitud_mediana']:.2f})" for _, r in gene_summary.head(3).iterrows())))
    rows.append(("5.3", "Casos (BGC, familia) con copias casi idénticas (similitud ≥ 0.9)",
                 f"{len(near)} / {len(repeated_genes)}" + (" · " + ", ".join(f"{g}: {k}" for g, k in near["gen"].value_counts().items()) if len(near) else "")))
    rows.append(("5.3", "BGCs con algún dominio repetido en un mismo gen (si se cuenta cada etiqueta)",
                 f"{n_bgc_dom} ({n_bgc_dom_runs}) / {len(y)}"))
    rows.append(("5.3", "Dominios repetidos en un mismo gen en más BGCs (BGCs; copias máx. por gen)",
                 "; ".join(f"{r['dominio']} ({r['n_BGC']}; {r['copias_max']})" for _, r in domain_summary.head(3).iterrows())))
    rows.append(("5.4", "Índice de sintenia medio dentro / entre clases", f"{syn_within:.3f} / {syn_between:.3f}"))
    rows.append(("5.4", "Par de clases distintas con más sintenia", f"{off.idxmax()[0]} – {off.idxmax()[1]} ({off.max():.2f})"))
    rows.append(("5.4", "Loci por clase (BGCs en el locus 1)",
                 "; ".join(f"{c} {n_loci[c]} ({class_summary.loc[c, 'BGCs_en_el_locus_1']})" for c in class_order)))
    rows.append(("5.4", "Tipos de sintenia por clase (BGCs en el tipo 1)",
                 "; ".join(f"{c} {n_types[c]} ({class_summary.loc[c, 'BGCs_en_el_tipo_1']})" for c in class_order)))
    first = locus_summary[locus_summary["locus"] == 1].set_index("clase")
    rows.append(("5.5", "Columnas con la misma familia en ≥ 90 % de los BGCs, en el locus 1 de cada clase",
                 "; ".join(f"{c} {first.loc[c, 'columnas_90pct']}" for c in class_order)))
rows.sort(key=lambda r: [int(x) if x.isdigit() else 99 for x in r[0].split(".")])      # por sección

summary_table = save_table(pd.DataFrame(rows, columns=["seccion", "indicador", "valor"]),
                           "A.2", "resumen_resultados")
pd.set_option("display.max_colwidth", 140)
summary_table

## A.3 Archivos generados

Todos los archivos que produjo el notebook, por carpeta y tipo. El nombre de cada archivo empieza con la sección que lo genera.

In [ ]:
files = sorted(p for p in OUT.rglob("*") if p.is_file())
listing = save_table(pd.DataFrame({
    "carpeta": [p.parent.name for p in files],
    "archivo": [p.name for p in files],
    "seccion": [p.name.split("_", 1)[0] for p in files],
    "tipo": [p.suffix.lstrip(".") for p in files],
    "kB": [round(p.stat().st_size / 1024, 1) for p in files]}), "A.3", "archivos_generados")
print(f"{len(files)} archivos en la carpeta {OUT}/")
listing.pivot_table(index="carpeta", columns="tipo", values="kB", aggfunc="count", fill_value=0)